# RSNA 2026: End-to-End Self-Contained SOTA Inference Pipeline

**Architecture:** trained DINOv2 and optional CoAtNet checkpoints; d4-derived target-weighted rank blend when both families are attached (not the full d4 model graph)
**Resolution & Slices:** $518 \times 518$ Native DINOv2 Patches ($37 \times 37$ tokens), Depth $D=32$
**Design Philosophy:**
1. **Self-Contained Code:** All Python scripts, modules, preprocessing, model definitions, and inference logic are built **directly inside this notebook**.
2. **Weights-Only Zip:** The only external artifact uploaded is `rsna.zip` (containing trained fold checkpoints (`*_best.pt`, with `*_ema.pt`/`*_swa.pt` fallback); include both DINOv2 and CoAtNet folders to activate the family blend).
3. **Inverted-Cache Inference:** Test studies are cached once; horizontal flips remain disabled to protect medial/lateral anatomical validity.
4. **Kaggle Dual-T4 GPU Saturation:** Automated multi-GPU `DataParallel` and mixed-precision `autocast`.

---
### 📋 Kaggle Submission Checklist
- [x] **Internet:** Disabled in Notebook settings (mandatory for submission).
- [x] **Accelerator:** GPU T4 x2 (Dual T4).
- [x] **Input Data:** `rsna-knee-abnormality-detection` competition dataset attached.
- [x] **Model Weights:** Upload your `rsna.zip` (containing the `.pt` checkpoints) as a private dataset.

In [ ]:
# ==============================================================================
# STEP 1: OFFLINE WHEEL INSTALLATION
# ==============================================================================
import os
import sys
import glob

# Auto-detect and install any offline wheels (.whl) in /kaggle/input (e.g. dicomsdl, timm)
wheels = glob.glob('/kaggle/input/**/*.whl', recursive=True)
if wheels:
    print(f'[SETUP] Found {len(wheels)} offline wheel packages. Installing...')
    for whl in wheels:
        os.system(f'pip install "{whl}" --no-index --find-links /kaggle/input/ --quiet')
    print('[SETUP] Wheel installation complete.')
else:
    print('[SETUP] No .whl packages found. Proceeding with pre-installed environment packages.')


In [ ]:
# ==============================================================================
# STEP 2: AUTO-EXTRACT MODEL WEIGHTS (RSNA.ZIP)
# ==============================================================================
import zipfile
import shutil

WEIGHTS_DIR = '/kaggle/working/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Search for rsna.zip or any zip containing model weights
zip_candidates = glob.glob('/kaggle/input/**/rsna*.zip', recursive=True) + glob.glob('/kaggle/input/**/*.zip', recursive=True)
extracted_from_zip = False

if zip_candidates:
    target_zip = zip_candidates[0]
    print(f'[AUTO-DETECT] Found weights archive: {target_zip}')
    print(f'[AUTO-DETECT] Extracting weights to {WEIGHTS_DIR}...')
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        zip_ref.extractall(WEIGHTS_DIR)
    extracted_from_zip = True
    print('[SUCCESS] Weights extracted successfully.')
else:
    print('[INFO] No .zip archive found. Checking for unzipped dataset folders in /kaggle/input...')

# Discover checkpoint files (.pt)
found_checkpoints = glob.glob(f'{WEIGHTS_DIR}/**/*.pt', recursive=True) + glob.glob('/kaggle/input/**/*.pt', recursive=True)
print(f'[SUCCESS] Discovered {len(found_checkpoints)} checkpoint file(s):')
for ckpt in found_checkpoints[:10]:
    print(f'   - {ckpt}')


In [ ]:
# ==============================================================================
# CELL 3: EXTRACT EXACT CURRENT PIPELINE MODULES (NO CODE IN WEIGHTS ZIP)
# ==============================================================================
import os
import sys
import base64

REPO_ROOT = '/kaggle/working'
PIPELINE_MODULES = {
    'src/main.py': '...',
    'src/__init__.py': 'IiIiUlNOQSBLbmVlIEFibm9ybWFsaXR5IERldGVjdGlvbiBzb3VyY2UgcGFja2FnZSAodjIgcHJlcHJvY2Vzc2luZyBsYXllcikuIiIiCg==',
    'src/core/config.py': 'IiIiQ2VudHJhbCBjb25maWd1cmF0aW9uIOKAlCB2MiB1cGdyYWRlIGxheWVyLg0KDQpVcGdyYWRlIGhpc3RvcnkNCi0tLS0tLS0tLS0tLS0tLQ0KSiAgRXBvY2hzIDEw4oaSMTUsIHdlaWdodF9kZWNheSAwLjAy4oaSMC4wNSwgTFJfQkFDS0JPTkUgOGUtNuKGkjVlLTYuDQogICBCYXNpczogVmlUIGZpbmUtdHVuaW5nIGJlc3QtcHJhY3RpY2VzIGNvbnNlbnN1cyAyMDI1Og0KICAgICAtIHdlaWdodF9kZWNheT0wLjA1IGlzIHN0YW5kYXJkIGZvciBWaVRzIChEb3Nvdml0c2tpeSBldCBhbC4gMjAyMSwNCiAgICAgICBEZWlUIElJSSBUb3V2cm9uIGV0IGFsLiAyMDIyLCBESU5PdjIgT3F1YWIgZXQgYWwuIDIwMjMpLg0KICAgICAtIE1vcmUgY29uc2VydmF0aXZlIGJhY2tib25lIExSICg1ZS02KSBhcHByb3ByaWF0ZSBmb3IgbGFyZ2VyIEJhc2UgbW9kZWwuDQogICAgIC0gMjAgZXBvY2hzIG9uIERHWCBTcGFyayAobm8gS2FnZ2xlIDktaHIgY2FwKTsgY29zaW5lIHNjaGVkdWxlIGZ1bGx5IGRlY2F5cy4NCg0KICAgVElNRV9CVURHRVRfSE9VUlMgc2V0IHRvIDk5OTkgKGRpc2FibGVkIG9uIERHWCBTcGFyazsgbm8gS2FnZ2xlIGNhcCkuDQoNCkFTTCBkZWZhdWx0cyByZWdpc3RlcmVkIGhlcmUgc28gdGhleSBjYW4gYmUgb3ZlcnJpZGRlbiBmcm9tIGEgc2luZ2xlIHBsYWNlLg0KDQpMZWdhY3kgY29uc3RhbnRzIChTRUVELCBUQVJHRVRTLCBTTE9UUywgQ0FOT05fT1JJRU5UIOKApikgYXJlIHVuY2hhbmdlZCBzbw0KZXhpc3RpbmcgY29kZSBhbmQgbm90ZWJvb2tzIGtlZXAgd29ya2luZyB3aXRob3V0IG1vZGlmaWNhdGlvbi4NCiIiIg0KaW1wb3J0IG9zDQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIHJlcGxhY2UNCg0KIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgSGFyZHdhcmUgU2FmZXR5ICYgQ2lyY3VpdCBCcmVha2VyIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgA0KIyBIYXJkIFVuaWZpZWQgTWVtb3J5IExpbWl0IChER1ggMTI4R0Igbm9kZSk6DQojIElmIHRvdGFsIG1lbW9yeSB1c2FnZSBleGNlZWRzIDExOCBHQiBpbiBhbnkgc3RlcCwgY2xlYW5seSBhYm9ydCwgZmx1c2ggbWVtb3J5LA0KIyBzYXZlIGVtZXJnZW5jeSBjaGVja3BvaW50LCBhbmQgcHJvbXB0IHRvIHJlc3RhcnQuDQpDSVJDVUlUX0JSRUFLRVJfTUFYX1JBTV9HQiA9IGZsb2F0KG9zLmVudmlyb24uZ2V0KCJSU05BX01BWF9SQU1fR0IiLCAiMTE4LjAiKSkNCg0KIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgbGVnYWN5IHRyYWluaW5nIGNvbnN0YW50cyDilIDilIDilIDilIDilIDilIDilIDilIANClNFRUQgICAgICAgICAgICAgPSAyMDI2DQpFUE9DSFMgICAgICAgICAgID0gMTAgICAgICAgICAgIyBPcHRpbWl6ZWQ6IHBlYWsgdmFsaWRhdGlvbiBBVUMgY29udmVyZ2VzIGJ5IGVwb2NoIDYtODsgMTAgYXZvaWRzIGxhYmVsLW5vaXNlIG92ZXJmaXR0aW5nDQpCQVRDSF9TSVpFICAgICAgID0gMjQgICAgICAgICAgIyBVcGdyYWRlZDogSGFyZHdhcmUtb3B0aW1pemVkIGZvciAyOEdCIFZSQU0gKyBHcmFkaWVudCBDaGVja3BvaW50aW5nDQpOVU1fV09SS0VSUyAgICAgID0gMTIgICAgICAgICAgIyAxMiB3b3JrZXJzIG9wdGltYWxseSBiYWxhbmNlcyBOVk1lIHRocm91Z2hwdXQgd2hpbGUga2VlcGluZyBSQU0gc3RyaWN0bHkgPCA5MEdCDQpHUkFEX0FDQ1VNICAgICAgID0gMSAgICAgICAgICAgIyBFZmZlY3RpdmUgYmF0Y2ggPSAxNiBzdHVkaWVzIHBlciBvcHRpbWl6ZXIgc3RlcA0KTl9XSU5ET1dTX1RSQUlOICA9IDUgICAgICAgICAgICMgNSBzdHJhdGlmaWVkIHdpbmRvd3M6IDc1JSBhcnRpY3VsYXIgY292ZXJhZ2UgKHByZXZlbnRzIE1JTCBmb2NhbCB0ZWFyIGRpbHV0aW9uKQ0KTFJfSEVBRCAgICAgICAgICA9IDJlLTMNCkxSX0JBQ0tCT05FICAgICAgPSAxZS01ICAgICAgICAjIFNjYWxlZCBmb3IgZWZmZWN0aXZlIGJhdGNoPTE2DQpXRUlHSFRfREVDQVkgICAgID0gMC4wNSAgICAgICAgIyBTdGFuZGFyZCBWaVQgcmVjaXBlDQpDT0FUTkVUX0xSX0hFQUQgICAgICA9IDFlLTMNCkNPQVRORVRfTFJfQkFDS0JPTkUgID0gM2UtNQ0KQ09BVE5FVF9XRUlHSFRfREVDQVkgPSAwLjAyDQpDT0FUTkVUX0lOUFVUX1NJWkUgICA9IDM4NA0KQ09BVE5FVF9FTkNPREVfQ0hVTksgPSA4DQpVTkZSRUVaRV9MQVNUICAgID0gOCAgICAgICAgICAgIyBUb3AgOCBibG9ja3MgZmluZS10dW5lZCB3aXRoIExMUkQ7IGJsb2NrcyAwLTMgZnJvemVuIChwcmVzZXJ2ZXMgMi41RCBNUkkgYWRhcHRhdGlvbikNCkxPUkFfUkFOSyAgICAgICAgPSAwICAgICAgICAgICMgVXBncmFkZSBCIOKAlCByYW5rIGZvciBRViBMb1JBIGFkYXB0ZXJzDQpMT1JBX0FMUEhBICAgICAgID0gMzIgICAgICAgICAgIyBMb1JBIHNjYWxpbmc6IHNjYWxlID0gTE9SQV9BTFBIQSAvIExPUkFfUkFOSyA9IDINClRJTUVfQlVER0VUX0hPVVJTID0gOTk5OS4wICAgICAjIERHWCBTcGFyazogbm8gS2FnZ2xlIHRpbWUgY2FwIOKAlCBkaXNhYmxlZA0KRUFSTFlfU1RPUF9QQVRJRU5DRSA9IDMgICAgICAgICMgRWFybHkgc3RvcHBpbmcgcGF0aWVuY2Ugb24gdmFsaWRhdGlvbiBNYWNyby1BVUMNCg0KIyBTV0Egc2V0dGluZ3MgKEl6bWFpbG92IGV0IGFsLiwgVUFJIDIwMTgpDQpTV0FfRVBPQ0hTICAgICAgPSAzICAgICAgICAgICAgIyBhcHBseSBTV0EgZm9yIHRoZSBsYXN0IFNXQV9FUE9DSFMgb2YgdHJhaW5pbmcNClNXQV9MUiAgICAgICAgICA9IDFlLTYgICAgICAgICAjIGNvbnN0YW50IExSIGR1cmluZyBTV0EgcGhhc2UNCg0KIyBBc3ltbWV0cmljIExvc3MgZGVmYXVsdHMgKFJpZG5payBldCBhbC4sIElDQ1YgMjAyMSkNCkFTTF9HQU1NQV9ORUcgICA9IDQuMCAgICAgICAgICAjIHBhcGVyIGRlZmF1bHQ9NDsgb3VyIDk4LjclIHVubGFiZWxlZCAoZGUtZmFjdG8gbmVnYXRpdmUpIGRhdGFzZXQgbmVlZHMgZnVsbCBuZWdhdGl2ZSBzdXBwcmVzc2lvbg0KQVNMX0dBTU1BX1BPUyAgID0gMC4wICAgICAgICAgICMgcG9zaXRpdmUgZm9jdXNpbmcgKGtlZXAgMCDigJQgbmV2ZXIgcGVuYWxpc2UgdHJ1ZSBwb3NpdGl2ZXMpDQpBU0xfQ0xJUCAgICAgICAgPSAwLjA1ICAgICAgICAgIyBwcm9iYWJpbGl0eSBzaGlmdCDigJQgZGlzY2FyZHMgbWlzbGFiZWxsZWQgZWFzeSBuZWdhdGl2ZXMNCg0KIyBQZXItdGFyZ2V0IEFTTCBnYW1tYV9uZWcgKEZJWDogcmVwbGFjZXMgZ2xvYmFsIHNjYWxhcikuDQojIEdyb3VuZGVkIGluIGdvbGQtbGFiZWwgcHJldmFsZW5jZSBmcm9tIHRoZSA1OC1zdHVkeSBnb2xkIHNldDoNCiMgICBFZmZ1c2lvbj02MC4zJSwgU3lub3ZpdGlzPTQ2LjYlLCBBQ0w9NDEuNCUsIE1lZGlhbCBNZW5pc2N1cz00NC44JSAtPiBsb3dlciBnYW1tYQ0KIyAgIE1DTD0xNS41JSwgTGF0ZXJhbCBPQT0xOSUsIEJha2Vycz0yMC43JSwgTWVkaWFsIE9BPTI1LjklIC0+IGhpZ2hlciBnYW1tYQ0KIyBIaWdoZXIgZ2FtbWFfbmVnID0gc3VwcHJlc3MgZWFzeSBuZWdhdGl2ZXMgaGFyZGVyID0gYmV0dGVyIGZvciByYXJlIHBvc2l0aXZlcy4NCiMgTG93ZXIgZ2FtbWFfbmVnID0gZG9uJ3Qgc3VwcHJlc3MgPSBwcmVzZXJ2ZSBsZWFybmluZyBzaWduYWwgZm9yIGNvbW1vbiBwb3NpdGl2ZXMuDQpBU0xfR0FNTUFfTkVHX1BFUl9UQVJHRVQgPSBbDQogICAgMi4wLCAgICMgQUNMICAgICAgICAgICAgICg0MS40JSBwb3MpDQogICAgNC4wLCAgICMgTUNMICAgICAgICAgICAgICgxNS41JSBwb3MpIOKAlCByYXJlLCBzdXBwcmVzcyBuZWdhdGl2ZXMgaGFyZA0KICAgIDIuMCwgICAjIE1lZGlhbCBNZW5pc2N1cyAoNDQuOCUgcG9zKQ0KICAgIDIuNSwgICAjIExhdGVyYWwgTWVuaXNjdXMoMzkuNyUgcG9zKQ0KICAgIDMuMCwgICAjIE1lZGlhbCBPQSAgICAgICAoMjUuOSUgcG9zKQ0KICAgIDMuNSwgICAjIExhdGVyYWwgT0EgICAgICAoMTkuMCUgcG9zKSDigJQgcmFyZQ0KICAgIDIuMCwgICAjIFBGIE9BICAgICAgICAgICAoMzYuMiUgcG9zKQ0KICAgIDEuMCwgICAjIEVmZnVzaW9uICAgICAgICAoNjAuMyUgcG9zKSDigJQgdmVyeSBjb21tb247IERPTidUIHN1cHByZXNzIG5lZ2F0aXZlcyBoYXJkDQogICAgMi4wLCAgICMgU3lub3ZpdGlzICAgICAgICg0Ni42JSBwb3MpDQogICAgMy4wLCAgICMgQmFrZXIncyAgICAgICAgICgyMC43JSBwb3MpDQogICAgMi41LCAgICMgQ29udHVzaW9uICAgICAgICgzMi44JSBwb3MpDQogICAgMi41LCAgICMgRnJhY3R1cmUgICAgICAgICgzMS4wJSBwb3MpDQpdDQoNCiMgTGFiZWwgc21vb3RoaW5nIChTemVnZWR5IGV0IGFsLiAyMDE2OyBzdGFuZGFyZCBWaVQgcmVjaXBlKQ0KTEFCRUxfU01PT1RISU5HID0gMC4wICAgICAgICAgICMgZGlzYWJsZWQ6IEFTTCBjbGlwPTAuMDUgYWxyZWFkeSBoYW5kbGVzIGxhYmVsIG5vaXNlDQoNCiMgQXVnbWVudGF0aW9uICh1bmNoYW5nZWQgZnJvbSBiYXNlbGluZSkNCkFVR19ST1RfREVHICA9IDguMA0KQVVHX1NDQUxFICAgID0gMC4wOA0KQVVHX1NISUZUICAgID0gMC4wNQ0KQVVHX0lOVEVOU0lUWSA9IDAuMQ0KDQojIE1peHVwIChaaGFuZyBldCBhbC4sIElDTFIgMjAxOCDigJQgYXBwbGllZCBpbiBmZWF0dXJlIHNwYWNlLCBVcGdyYWRlIEkpDQojIFJhcmUgdGFyZ2V0IGxvc3MgbXVsdGlwbGllcnMgKEZJWCA0KS4NCiMgRnJhY3R1cmUvQmFrZXJzL1N5bm92aXRpcy9Db250dXNpb24gaGF2ZSA8NSUgcHJldmFsZW5jZSBhbmQgZG9taW5hdGUgbWFjcm8tQVVDIHZhcmlhbmNlLg0KUkFSRV9UQVJHRVRfV0VJR0hUUyA9IHsNCiAgICAiRnJhY3R1cmUiOiAgIDIuNSwNCiAgICAiQmFrZXIncyI6ICAgIDIuMCwNCiAgICAiU3lub3ZpdGlzIjogIDEuNSwNCiAgICAiQ29udHVzaW9uIjogIDEuNSwNCn0NCg0KTUlYVVBfQUxQSEEgID0gMC4wICAgICAgICAgICAgICMgQmV0YShhbHBoYSwgYWxwaGEpIG1peGluZyBjb2VmZmljaWVudDsgMCA9IGRpc2FibGVkDQoNCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSAIHRhcmdldCAvIHNsb3Qgc2NoZW1hIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgA0KVEFSR0VUUyA9IFsNCiAgICAiQUNMIiwgIk1DTCIsICJNZWRpYWwgTWVuaXNjdXMiLCAiTGF0ZXJhbCBNZW5pc2N1cyIsICJNZWRpYWwgT0EiLA0KICAgICJMYXRlcmFsIE9BIiwgIlBGIE9BIiwgIkVmZnVzaW9uIiwgIlN5bm92aXRpcyIsICJCYWtlcidzIiwNCiAgICAiQ29udHVzaW9uIiwgIkZyYWN0dXJlIiwNCl0NCg0KIyAobmFtZSwgcGxhbmUsIGZsdWlkLXdlaWdodGVkLCBmYXQtc3VwcHJlc3NlZCkNClNMT1RTID0gWw0KICAgICgiU0FHX0ZMVUlEX0ZTIiwgICAiU2FnaXR0YWwiLCAgVHJ1ZSwgIFRydWUpLA0KICAgICgiQ09SX0ZMVUlEX0ZTIiwgICAiQ29yb25hbCIsICAgVHJ1ZSwgIFRydWUpLA0KICAgICgiQVhfRkxVSURfRlMiLCAgICAiQXhpYWwiLCAgICAgVHJ1ZSwgIFRydWUpLA0KICAgICgiU0FHX0ZMVUlEX05PRlMiLCAiU2FnaXR0YWwiLCAgVHJ1ZSwgIEZhbHNlKSwNCiAgICAoIkNPUl9UMSIsICAgICAgICAgIkNvcm9uYWwiLCAgIEZhbHNlLCBGYWxzZSksDQogICAgKCJTQUdfVDEiLCAgICAgICAgICJTYWdpdHRhbCIsICBGYWxzZSwgRmFsc2UpLA0KXQ0KTl9TTE9UUyA9IGxlbihTTE9UUykNCg0KIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgZGF0YXNldCBmYWN0cyAodmVyaWZpZWQgYnkgdGhlIDIwMjYgaW50ZWdyaXR5IGF1ZGl0KSDilIANCkNBTk9OX09SSUVOVCAgICAgICA9IHsiU2FnaXR0YWwiOiAiUEkiLCAiQ29yb25hbCI6ICJMSSIsICJBeGlhbCI6ICJMUCJ9DQpMQVRfTUlOX09GRlNFVF9NTSAgPSAyMC4wDQpDT01QRVRJVElPTiAgICAgICAgPSAicnNuYS1rbmVlLWFibm9ybWFsaXR5LWRldGVjdGlvbiINClJPT1RfQ0FORElEQVRFUyAgICA9IFsNCiAgICBmIi9rYWdnbGUvaW5wdXQvY29tcGV0aXRpb25zL3tDT01QRVRJVElPTn0iLA0KICAgIGYiL2thZ2dsZS9pbnB1dC97Q09NUEVUSVRJT059IiwNCiAgICAiZDovS25lZV9SU05BX0RhdGEiLA0KICAgICJkOi9LbmVlX1JTTkEvZGF0YSIsDQogICAgImQ6L0tuZWVfUlNOQSIsDQpdDQoNCg0KIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgUHJlQ2ZnIChwcmVwcm9jZXNzaW5nKSDilIDilIDilIDilIDilIDilIDilIANCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpDQpjbGFzcyBQcmVDZmc6DQogICAgbmFtZTogc3RyID0gInYyIg0KICAgICMgaW4tcGxhbmUNCiAgICBjcm9wX21tOiBmbG9hdCAgICAgID0gMTMwLjANCiAgICBpbWdfc2l6ZTogaW50ICAgICAgID0gMzM2DQogICAgcGFkX21vZGU6IHN0ciAgICAgICA9ICJib3JkZXJfbWVkaWFuIg0KICAgIHJlc2l6ZV9tb2RlOiBzdHIgICAgPSAiYXJlYSINCiAgICB3aWRlX2NlbnRlcjogYm9vbCAgID0gVHJ1ZQ0KICAgIHdpZGVfcmF0aW86IGZsb2F0ICAgPSAxLjMNCiAgICAjIGludGVuc2l0eQ0KICAgIG5vcm1fbG86IGZsb2F0ICAgICAgPSAxLjANCiAgICBub3JtX2hpOiBmbG9hdCAgICAgID0gOTkuNQ0KICAgIGNsaXBfbmVnYXRpdmU6IGJvb2wgPSBUcnVlDQogICAgIyB0aHJvdWdoLXBsYW5lDQogICAgb3JkZXJfbW9kZTogc3RyICAgICA9ICJwb3NpdGlvbiINCiAgICB6X21vZGU6IHN0ciAgICAgICAgID0gIm1tIg0KICAgIHN0YWNrX2RlcHRoOiBpbnQgICAgPSAyNA0KICAgIHpfc3RlcF9tbTogZmxvYXQgICAgPSA0LjANCiAgICBiYW5kOiB0dXBsZSAgICAgICAgID0gKDAuMiwgMC44KQ0KICAgICMgMi41RCB3aW5kb3dzDQogICAgZ3JvdXA6IGludCAgICAgICAgICA9IDMNCiAgICB3aW5fc3RyaWRlOiBpbnQgICAgID0gMg0KICAgICMgY2Fub25pY2FsaXNhdGlvbg0KICAgIHJlb3JpZW50OiBib29sICAgICAgPSBUcnVlDQogICAgbGF0X2Nhbm9uOiBib29sICAgICA9IFRydWUNCiAgICAjIHNsb3QgbG9naWMNCiAgICBzbG90X3ByZWZlcl8yZDogYm9vbCAgICA9IFRydWUNCiAgICBzbG90X2ZzX3ByaW9yaXR5OiBzdHIgICA9ICJjc3YiDQogICAgc2xvdF9jc3ZfZmFsbGJhY2s6IGJvb2wgPSBUcnVlDQoNCiAgICBAcHJvcGVydHkNCiAgICBkZWYgbl93aW5kb3dzKHNlbGYpIC0+IGludDoNCiAgICAgICAgcmV0dXJuIG1heCgxLCAoc2VsZi5zdGFja19kZXB0aCAtIHNlbGYuZ3JvdXApIC8vIHNlbGYud2luX3N0cmlkZSArIDEpDQoNCg0KUFJFU0VUUyA9IHsNCiAgICAicHVibGljIjogUHJlQ2ZnKA0KICAgICAgICBuYW1lPSJwdWJsaWMiLCBwYWRfbW9kZT0icHVibGljIiwgcmVzaXplX21vZGU9ImJpbGluZWFyIiwNCiAgICAgICAgd2lkZV9jZW50ZXI9RmFsc2UsIG5vcm1fbG89MS4wLCBub3JtX2hpPTk5LjAsIGNsaXBfbmVnYXRpdmU9RmFsc2UsDQogICAgICAgIG9yZGVyX21vZGU9Im5vcm1hbCIsIHpfbW9kZT0iaW5kZXgiLCBzdGFja19kZXB0aD0xMiwgYmFuZD0oMC4yLCAwLjgpLA0KICAgICAgICB3aW5fc3RyaWRlPTEsIHJlb3JpZW50PUZhbHNlLCBzbG90X3ByZWZlcl8yZD1GYWxzZSwgc2xvdF9jc3ZfZmFsbGJhY2s9RmFsc2UsDQogICAgKSwNCiAgICAidjIiOiBQcmVDZmcobmFtZT0idjIiLCBpbWdfc2l6ZT01MTgsIHN0YWNrX2RlcHRoPTMyKSwNCn0NCg0KDQpkZWYgZ2V0X2NmZyhuYW1lOiBzdHIgPSAidjIiLCAqKm92ZXJyaWRlcykgLT4gUHJlQ2ZnOg0KICAgIGJhc2UgPSBQUkVTRVRTW25hbWVdDQogICAgcmV0dXJuIHJlcGxhY2UoYmFzZSwgKipvdmVycmlkZXMpIGlmIG92ZXJyaWRlcyBlbHNlIGJhc2UNCg0KDQpkZWYgY2ZnX2Zyb21fZGljdChkOiBkaWN0KSAtPiBQcmVDZmc6DQogICAgIiIiUmVidWlsZCBhIFByZUNmZyBmcm9tIGEgSlNPTi1sb2FkZWQgZGljdCAobGlzdHMg4oaSIHR1cGxlcykuIiIiDQogICAgZCA9IGRpY3QoZCkNCiAgICBpZiAiYmFuZCIgaW4gZDoNCiAgICAgICAgZFsiYmFuZCJdID0gdHVwbGUoZFsiYmFuZCJdKQ0KICAgIHJldHVybiBQcmVDZmcoKipkKQ0KDQoNCiMgTGVnYWN5IGFsaWFzZXMNCkNST1BfTU0gICAgPSBQUkVTRVRTWyJ2MiJdLmNyb3BfbW0NCklNR19TSVpFICAgPSBQUkVTRVRTWyJ2MiJdLmltZ19zaXplDQpHUk9VUF9TSVpFID0gUFJFU0VUU1sidjIiXS5ncm91cA0KDQoNCg0K',
    'src/core/__init__.py': '',
    'src/data/dataset.py': 'IiIidG9yY2ggRGF0YXNldCBvdmVyIHRoZSBtZW1tYXAgY2FjaGUuIEFsbCByZWFsIHdvcmsgaGFwcGVucyBpbiBwcmVwcm9jZXNzLmxvYWRlciAobnVtcHkpLiIiIg0KaW1wb3J0IGpzb24NCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHRvcmNoDQpmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQNCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCmZyb20gc3JjLmRhdGEucHJlcHJvY2VzcyBpbXBvcnQgY2FjaGUgYXMgY2FjaGVfbW9kDQpmcm9tIHNyYy5kYXRhLnByZXByb2Nlc3MgaW1wb3J0IGxvYWRlcg0KDQoNCmNsYXNzIFJTTkFEYXRhc2V0KERhdGFzZXQpOg0KICAgICIiImRmOiBTdHVkeUluc3RhbmNlVUlEICsgdGFyZ2V0IGNvbHVtbnMgKE5hTiA9IHVubGFiZWxlZCkgWysgJzx0YXJnZXQ+X3dlaWdodCcgY29sdW1uc10uDQoNCiAgICBSZXR1cm5zIChpbWdzIHVpbnQ4IFtTLFcsMyxILFddLCBzbG90X21hc2sgW1NdLCB3aW5fbWFzayBbUyxXXSwgdGFyZ2V0cyBbMTJdLCB3ZWlnaHRzIFsxMl0pLg0KICAgIFRocm91Z2hwdXQgbm90ZXM6IHRoZSBtZW1tYXAgaXMgb3BlbmVkIGxhemlseSBwZXIgd29ya2VyIChmb3JrLXNhZmUpOyBsYWJlbHMgYXJlIGRlbnNlIGFycmF5czsgdGhlIGF1Z21lbnRhdGlvbg0KICAgIFJORyBpcyBzZWVkZWQgZnJvbSAoc2VlZCwgZXBvY2gsIGluZGV4KSBzbyBydW5zIGFyZSByZXByb2R1Y2libGUgcmVnYXJkbGVzcyBvZiB3b3JrZXIgc2NoZWR1bGluZy4iIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkZiwgY2FjaGVfcHJlZml4LCBjZmc9Tm9uZSwgaXNfdHJhaW49RmFsc2UsIG5fd2luZG93c191c2U9Tm9uZSwgc2VlZD1jb25maWcuU0VFRCwgYXVnPVRydWUpOg0KICAgICAgICBzZWxmLnByZWZpeCwgc2VsZi5jZmcgPSBjYWNoZV9wcmVmaXgsIGNmZyBvciBjYWNoZV9tb2QuY2ZnX29mKGNhY2hlX3ByZWZpeCkgICAjIGRlZmF1bHQ6IHRoZSBjYWNoZSdzIG93biBjb25maWcNCiAgICAgICAgc2VsZi5pc190cmFpbiwgc2VsZi5uX3VzZSwgc2VsZi5zZWVkLCBzZWxmLmF1Zywgc2VsZi5lcG9jaCA9IGlzX3RyYWluLCBuX3dpbmRvd3NfdXNlLCBzZWVkLCAwLCAwDQogICAgICAgIHdpdGggb3BlbihmJ3tjYWNoZV9wcmVmaXh9Lm1ldGEuanNvbicpIGFzIGZoOg0KICAgICAgICAgICAgc3R1ZGllcyA9IGpzb24ubG9hZChmaClbJ3N0dWRpZXMnXQ0KICAgICAgICBhdmFpbGFibGUgPSBzZXQoc3R1ZGllcykNCiAgICAgICAgYWxpZ25lZCA9IGRmW2RmWydTdHVkeUluc3RhbmNlVUlEJ10uYXN0eXBlKHN0cikuaXNpbihhdmFpbGFibGUpXS5yZXNldF9pbmRleChkcm9wPVRydWUpDQogICAgICAgIHNlbGYuaWRzID0gYWxpZ25lZFsnU3R1ZHlJbnN0YW5jZVVJRCddLmFzdHlwZShzdHIpLnRvbGlzdCgpDQogICAgICAgIHNlbGYucm93cywgc2VsZi5ZLCBzZWxmLld0ID0gbG9hZGVyLmxhYmVsX2FycmF5cyhhbGlnbmVkLCB7czogaSBmb3IgaSwgcyBpbiBlbnVtZXJhdGUoc3R1ZGllcyl9KQ0KICAgICAgICBzZWxmLl9jYWNoZSA9IE5vbmUNCg0KICAgIGRlZiBzZXRfZXBvY2goc2VsZiwgZSk6DQogICAgICAgIHNlbGYuZXBvY2ggPSBpbnQoZSkNCg0KICAgIGRlZiBfX2xlbl9fKHNlbGYpOg0KICAgICAgICByZXR1cm4gbGVuKHNlbGYucm93cykNCg0KICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBrKToNCiAgICAgICAgaWYgc2VsZi5fY2FjaGUgaXMgTm9uZToNCiAgICAgICAgICAgIHNlbGYuX2NhY2hlID0gY2FjaGVfbW9kLlN0dWR5Q2FjaGUoc2VsZi5wcmVmaXgsICdyJykNCiAgICAgICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKFtzZWxmLnNlZWQsIHNlbGYuZXBvY2gsIGludChrKV0pDQogICAgICAgIGltZ3MsIHNsb3QsIHdtID0gbG9hZGVyLm1ha2Vfc2FtcGxlKHNlbGYuX2NhY2hlLCBpbnQoc2VsZi5yb3dzW2tdKSwgc2VsZi5jZmcsIHNlbGYuaXNfdHJhaW4sIHNlbGYubl91c2UsIHJuZywgc2VsZi5hdWcpDQogICAgICAgIHJldHVybiAodG9yY2guZnJvbV9udW1weShpbWdzKSwgdG9yY2guZnJvbV9udW1weShzbG90KS5mbG9hdCgpLCB0b3JjaC5mcm9tX251bXB5KHdtKS5mbG9hdCgpLA0KICAgICAgICAgICAgICAgIHRvcmNoLmZyb21fbnVtcHkoc2VsZi5ZW2tdKSwgdG9yY2guZnJvbV9udW1weShzZWxmLld0W2tdKSkNCg==',
    'src/data/labels.py': 'IiIiTGFiZWwgdGFibGUgZm9yIHRyYWluaW5nOiBnb2xkIGxhYmVscyArIG9wdGlvbmFsIGV4dHJhIChyZXBvcnQtZGVyaXZlZCAvIHBzZXVkbykgbGFiZWxzIHdpdGggY29uZmlkZW5jZSB3ZWlnaHRzLg0KDQpDb250cmFjdCBjb25zdW1lZCBieSBSU05BRGF0YXNldDogU3R1ZHlJbnN0YW5jZVVJRCwgdGhlIDEyIFRBUkdFVFMgKE5hTiA9IHVubGFiZWxlZCAtPiBtYXNrZWQpLCBvcHRpb25hbA0KJzx0YXJnZXQ+X3dlaWdodCcgY29sdW1ucy4gOTguNyUgb2YgdHJhaW4gc3R1ZGllcyBoYXZlIG5vIGdvbGQgbGFiZWwsIHNvIHRoZSBleHRyYSB0YWJsZSBpcyB3aGVyZSBtb3N0IHN1cGVydmlzaW9uDQpjb21lcyBmcm9tOyBnb2xkIGFsd2F5cyB3aW5zIHdoZXJlIGl0IGV4aXN0cy4NCiIiIg0KaW1wb3J0IG9zDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCg0KDQpkZWYgYnVpbGRfbGFiZWxzKHJvb3QsIGV4dHJhX2Nzdj1Ob25lLCBleHRyYV93ZWlnaHQ9MC41LCBvdXRfY3N2PU5vbmUpOg0KICAgICIiInJvb3Q6IGNvbXBldGl0aW9uIGZvbGRlci4gZXh0cmFfY3N2OiBzYW1lIHNjaGVtYSBhcyB0cmFpbi5jc3YgKHRhcmdldHMgbWF5IGJlIHNvZnQgcHJvYmFiaWxpdGllcyBpbiBbMCwxXSksDQogICAgb3B0aW9uYWxseSB3aXRoICc8dGFyZ2V0Pl93ZWlnaHQnIGNvbHVtbnMgKGV4dHJhY3RvciBjb25maWRlbmNlKS4gLT4gRGF0YUZyYW1lIChhbmQgQ1NWIGlmIG91dF9jc3YpLiIiIg0KICAgIGlmIG5vdCBucC5pc2Zpbml0ZShleHRyYV93ZWlnaHQpIG9yIGV4dHJhX3dlaWdodCA8IDA6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHRyYV93ZWlnaHQgbXVzdCBiZSBmaW5pdGUgYW5kIG5vbi1uZWdhdGl2ZSwgZ290IHtleHRyYV93ZWlnaHQhcn0iKQ0KICAgIHRyID0gcGQucmVhZF9jc3Yob3MucGF0aC5qb2luKHJvb3QsICd0cmFpbi5jc3YnKSkNCiAgICBpZiB0clsnU3R1ZHlJbnN0YW5jZVVJRCddLmlzbmEoKS5hbnkoKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidHJhaW4uY3N2IGNvbnRhaW5zIGEgbWlzc2luZyBTdHVkeUluc3RhbmNlVUlEIikNCiAgICBvdXQgPSBwZC5EYXRhRnJhbWUoeydTdHVkeUluc3RhbmNlVUlEJzogdHJbJ1N0dWR5SW5zdGFuY2VVSUQnXS5hc3R5cGUoc3RyKS5zdHIuc3RyaXAoKX0pDQogICAgaWYgb3V0WydTdHVkeUluc3RhbmNlVUlEJ10uZXEoJycpLmFueSgpIG9yIG91dFsnU3R1ZHlJbnN0YW5jZVVJRCddLmR1cGxpY2F0ZWQoKS5hbnkoKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidHJhaW4uY3N2IG11c3QgaGF2ZSBub24tZW1wdHksIHVuaXF1ZSBTdHVkeUluc3RhbmNlVUlEIHZhbHVlcyIpDQogICAgZ29sZCA9IHt0OiAodHJbdF0uYXN0eXBlKGZsb2F0KSBpZiB0IGluIHRyLmNvbHVtbnMgZWxzZSBwZC5TZXJpZXMobnAubmFuLCBpbmRleD10ci5pbmRleCkpIGZvciB0IGluIGNvbmZpZy5UQVJHRVRTfQ0KICAgIGZvciB0YXJnZXQsIHZhbHVlcyBpbiBnb2xkLml0ZW1zKCk6DQogICAgICAgIGZpbml0ZSA9IHZhbHVlc1tucC5pc2Zpbml0ZSh2YWx1ZXMpXQ0KICAgICAgICBpZiBub3QgZmluaXRlLmJldHdlZW4oMC4wLCAxLjApLmFsbCgpOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImdvbGQgbGFiZWxzIGZvciB7dGFyZ2V0IXJ9IG11c3QgYmUgaW4gWzAsIDFdIikNCiAgICBleCA9IE5vbmUNCiAgICBpZiBleHRyYV9jc3YgYW5kIG9zLnBhdGguZXhpc3RzKGV4dHJhX2Nzdik6DQogICAgICAgIGV4X3JhdyA9IHBkLnJlYWRfY3N2KGV4dHJhX2NzdikNCiAgICAgICAgaWYgJ1N0dWR5SW5zdGFuY2VVSUQnIG5vdCBpbiBleF9yYXcuY29sdW1ucyBvciBleF9yYXdbJ1N0dWR5SW5zdGFuY2VVSUQnXS5pc25hKCkuYW55KCk6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXh0cmEgbGFiZWxzIG11c3QgY29udGFpbiBub24tbnVsbCBTdHVkeUluc3RhbmNlVUlEIHZhbHVlczoge2V4dHJhX2Nzdn0iKQ0KICAgICAgICBleF9yYXdbJ1N0dWR5SW5zdGFuY2VVSUQnXSA9IGV4X3Jhd1snU3R1ZHlJbnN0YW5jZVVJRCddLmFzdHlwZShzdHIpLnN0ci5zdHJpcCgpDQogICAgICAgIGlmIGV4X3Jhd1snU3R1ZHlJbnN0YW5jZVVJRCddLmVxKCcnKS5hbnkoKSBvciBleF9yYXdbJ1N0dWR5SW5zdGFuY2VVSUQnXS5kdXBsaWNhdGVkKCkuYW55KCk6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXh0cmEgbGFiZWxzIG11c3QgaGF2ZSBub24tZW1wdHksIHVuaXF1ZSBTdHVkeUluc3RhbmNlVUlEIHZhbHVlczoge2V4dHJhX2Nzdn0iKQ0KICAgICAgICBleCA9IGV4X3Jhdy5zZXRfaW5kZXgoJ1N0dWR5SW5zdGFuY2VVSUQnKS5yZWluZGV4KG91dFsnU3R1ZHlJbnN0YW5jZVVJRCddKQ0KICAgIG91dFsnc291cmNlJ10gPSAnbm9uZScNCiAgICBhbnlfZ29sZCA9IHBkLmNvbmNhdChnb2xkLCBheGlzPTEpLm5vdG5hKCkuYW55KGF4aXM9MSkudmFsdWVzDQogICAgb3V0LmxvY1thbnlfZ29sZCwgJ3NvdXJjZSddID0gJ2dvbGQnDQogICAgZm9yIHQgaW4gY29uZmlnLlRBUkdFVFM6DQogICAgICAgIGcgPSBnb2xkW3RdLnZhbHVlcw0KICAgICAgICB5ID0gZy5jb3B5KCkNCiAgICAgICAgdyA9IG5wLndoZXJlKG5wLmlzZmluaXRlKGcpLCAxLjAsIDAuMCkNCiAgICAgICAgaWYgZXggaXMgbm90IE5vbmUgYW5kIHQgaW4gZXguY29sdW1uczoNCiAgICAgICAgICAgIGUgPSBleFt0XS5hc3R5cGUoZmxvYXQpLnZhbHVlcw0KICAgICAgICAgICAgZmluaXRlID0gZVtucC5pc2Zpbml0ZShlKV0NCiAgICAgICAgICAgIGlmIG5vdCBucC5sb2dpY2FsX2FuZChmaW5pdGUgPj0gMC4wLCBmaW5pdGUgPD0gMS4wKS5hbGwoKToNCiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXh0cmEgbGFiZWxzIGZvciB7dCFyfSBtdXN0IGJlIGluIFswLCAxXSIpDQogICAgICAgICAgICBldyA9IGV4W2Yne3R9X3dlaWdodCddLmFzdHlwZShmbG9hdCkuZmlsbG5hKDEuMCkudmFsdWVzIGlmIGYne3R9X3dlaWdodCcgaW4gZXguY29sdW1ucyBlbHNlIDEuMA0KICAgICAgICAgICAgaWYgbnAuaXNzY2FsYXIoZXcpOg0KICAgICAgICAgICAgICAgIHBhc3MNCiAgICAgICAgICAgIGVsaWYgbm90IG5wLmlzZmluaXRlKGV3KS5hbGwoKSBvciAoZXcgPCAwKS5hbnkoKToNCiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXh0cmEgd2VpZ2h0cyBmb3Ige3Qhcn0gbXVzdCBiZSBmaW5pdGUgYW5kIG5vbi1uZWdhdGl2ZSIpDQogICAgICAgICAgICB1c2UgPSB+bnAuaXNmaW5pdGUoZykgJiBucC5pc2Zpbml0ZShlKQ0KICAgICAgICAgICAgeSA9IG5wLndoZXJlKHVzZSwgZSwgZykNCiAgICAgICAgICAgIHcgPSBucC53aGVyZSh1c2UsIGV4dHJhX3dlaWdodCAqIGV3LCB3KQ0KICAgICAgICAgICAgb3V0LmxvY1t1c2UgJiAob3V0Wydzb3VyY2UnXSA9PSAnbm9uZScpLCAnc291cmNlJ10gPSAnZXh0cmEnDQogICAgICAgIG91dFt0XSA9IHkNCiAgICAgICAgb3V0W2Yne3R9X3dlaWdodCddID0gdw0KICAgIGlmIG91dF9jc3Y6DQogICAgICAgIG91dC50b19jc3Yob3V0X2NzdiwgaW5kZXg9RmFsc2UpDQogICAgcmV0dXJuIG91dA0K',
    'src/data/__init__.py': '',
    'src/data/preprocess/bench.py': 'IiIiQ2FsaWJyYXRpb246IG1lYXN1cmUgcmVhbCB0aHJvdWdocHV0IG9uIGEgU01BTEwgc2FtcGxlIG9mIHRoZSBhY3R1YWwgZGF0YXNldCwgdGhlbiBwcm9qZWN0IHRoZSBmdWxsIGNvc3QuDQoNClJ1biBpdCBmaXJzdCBpbiBhIGZyZXNoIEthZ2dsZSBzZXNzaW9uICh0YWtlcyB+MS0zIG1pbnV0ZXMpOiBpdCB0ZWxscyB5b3UgaG93IGxvbmcgdGhlIGluZGV4IHNjYW4gYW5kIHRoZSBjYWNoZSBidWlsZA0Kd2lsbCB0YWtlIG9uIFRISVMgbWFjaGluZS9maWxlc3lzdGVtLCBob3cgbXVjaCBzY3JhdGNoIGRpc2sgdGhlIGNhY2hlIG5lZWRzLCBhbmQgd2hldGhlciB0aGUgdHJhaW5pbmcgbG9vcCdzIGRhdGENCmxvYWRpbmcgY2FuIGtlZXAgYSBHUFUgZmVkLg0KIiIiDQppbXBvcnQgb3MNCmltcG9ydCB0aW1lDQoNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQppbXBvcnQgc3JjLmNvcmUuY29uZmlnIGFzIGNvbmZpZw0KZnJvbSAuIGltcG9ydCBpbmRleCBhcyBwaXgNCmZyb20gLiBpbXBvcnQgc2xvdHMgYXMgcHNsb3RzDQpmcm9tIC4gaW1wb3J0IHBpcGVsaW5lDQpmcm9tIC4gaW1wb3J0IGNhY2hlIGFzIHBjYWNoZQ0KZnJvbSAuIGltcG9ydCBsb2FkZXINCmZyb20gLiBpbXBvcnQgcnVubmVyDQoNCg0KZGVmIF9pdGVtc19mb3Iocm9vdCwgc3BsaXQsIHN0dWRpZXMpOg0KICAgIGJhc2UgPSBvcy5wYXRoLmpvaW4ocm9vdCwgcGl4LlNQTElUX0RJUlNbc3BsaXRdKQ0KICAgIGl0ZW1zID0gW10NCiAgICBmb3Igc3QgaW4gc3R1ZGllczoNCiAgICAgICAgZm9yIGUgaW4gc29ydGVkKG9zLnNjYW5kaXIob3MucGF0aC5qb2luKGJhc2UsIHN0KSksIGtleT1sYW1iZGEgeDogeC5uYW1lKToNCiAgICAgICAgICAgIGlmIGUuaXNfZGlyKCk6DQogICAgICAgICAgICAgICAgaXRlbXMuYXBwZW5kKChzcGxpdCwgc3QsIGUubmFtZSwgZS5wYXRoKSkNCiAgICByZXR1cm4gaXRlbXMNCg0KDQpkZWYgY2FsaWJyYXRlKHJvb3QsIHNwbGl0PSd0cmFpbicsIG5fc3R1ZGllcz0yNCwgd29ya2Vycz1Ob25lLCBwcmVzZXQ9J3YyJywgc2NyYXRjaD1Ob25lLCBzdGVwX3NlY29uZHM9MS4wLA0KICAgICAgICAgICAgICBiYXRjaF9zaXplPWNvbmZpZy5CQVRDSF9TSVpFLCBuX3VzZT00KToNCiAgICBmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgUHJvY2Vzc1Bvb2xFeGVjdXRvcg0KICAgIGltcG9ydCBtdWx0aXByb2Nlc3NpbmcgYXMgbXANCiAgICBjZmcgPSBjb25maWcuZ2V0X2NmZyhwcmVzZXQpDQogICAgd29ya2VycyA9IHdvcmtlcnMgb3IgbWF4KDIsIChvcy5jcHVfY291bnQoKSBvciA0KSkNCiAgICBzY3JhdGNoID0gc2NyYXRjaCBvciBydW5uZXIucGlja19zY3JhdGNoKHZlcmJvc2U9RmFsc2UpDQogICAgYmFzZSA9IG9zLnBhdGguam9pbihyb290LCBwaXguU1BMSVRfRElSU1tzcGxpdF0pDQogICAgc3R1ZGllcyA9IHNvcnRlZChlLm5hbWUgZm9yIGUgaW4gb3Muc2NhbmRpcihiYXNlKSBpZiBlLmlzX2RpcigpKVs6bl9zdHVkaWVzXQ0KICAgIGl0ZW1zID0gX2l0ZW1zX2Zvcihyb290LCBzcGxpdCwgc3R1ZGllcykNCiAgICBuX3RvdGFsX3NlcmllcyA9IG5fdG90YWxfc3R1ZGllcyA9IE5vbmUNCiAgICBmb3IgbmFtZSwgYXR0ciBpbiAoKCd0cmFpbl9zZXJpZXMuY3N2JyBpZiBzcGxpdCA9PSAndHJhaW4nIGVsc2UgJ3Rlc3Rfc2VyaWVzLmNzdicsICdzZXJpZXMnKSwgKCd0cmFpbi5jc3YnIGlmIHNwbGl0ID09ICd0cmFpbicgZWxzZSAndGVzdC5jc3YnLCAnc3R1ZGllcycpKToNCiAgICAgICAgcCA9IG9zLnBhdGguam9pbihyb290LCBuYW1lKQ0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhwKToNCiAgICAgICAgICAgIG4gPSBsZW4ocGQucmVhZF9jc3YocCwgdXNlY29scz1bMF0pKQ0KICAgICAgICAgICAgaWYgYXR0ciA9PSAnc2VyaWVzJzoNCiAgICAgICAgICAgICAgICBuX3RvdGFsX3NlcmllcyA9IG4NCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgbl90b3RhbF9zdHVkaWVzID0gbg0KICAgIG5fdG90YWxfc2VyaWVzID0gbl90b3RhbF9zZXJpZXMgb3IgbGVuKGl0ZW1zKSAqIDQ0MDcgLy8gbWF4KGxlbihzdHVkaWVzKSwgMSkNCiAgICBuX3RvdGFsX3N0dWRpZXMgPSBuX3RvdGFsX3N0dWRpZXMgb3IgNDQwNw0KICAgIHJlcyA9IHt9DQogICAgIyAtLS0tIDEuIGluZGV4IHNjYW4NCiAgICB0ID0gdGltZS50aW1lKCkNCiAgICB0cnk6DQogICAgICAgIGV4ID0gUHJvY2Vzc1Bvb2xFeGVjdXRvcihtYXhfd29ya2Vycz13b3JrZXJzLCBtcF9jb250ZXh0PW1wLmdldF9jb250ZXh0KCdmb3JrJykpDQogICAgICAgIHJvd3MgPSBsaXN0KGV4Lm1hcChwaXguc2Nhbl9zZXJpZXMsIGl0ZW1zLCBjaHVua3NpemU9NCkpDQogICAgICAgIGV4LnNodXRkb3duKCkNCiAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICByb3dzID0gW3BpeC5zY2FuX3NlcmllcyhpKSBmb3IgaSBpbiBpdGVtc10NCiAgICBkdCA9IHRpbWUudGltZSgpIC0gdA0KICAgIGRmID0gcGl4LmF0dGFjaF9jc3ZfZmxhZ3MocGQuRGF0YUZyYW1lKHJvd3MpLCByb290KQ0KICAgIGZpbGVzID0gaW50KGRmWyduX2ZpbGVzJ10uc3VtKCkpDQogICAgcmVzWydpbmRleF9zX3Blcl9zZXJpZXMnXSA9IGR0IC8gbGVuKGl0ZW1zKQ0KICAgIHJlc1snaW5kZXhfZmlsZXNfcGVyX3MnXSA9IGZpbGVzIC8gZHQNCiAgICByZXNbJ2luZGV4X21pbnV0ZXNfZnVsbCddID0gcmVzWydpbmRleF9zX3Blcl9zZXJpZXMnXSAqIG5fdG90YWxfc2VyaWVzIC8gNjANCiAgICAjIC0tLS0gMi4gY2FjaGUgYnVpbGQNCiAgICBhbm4sIHRhYiA9IHBzbG90cy5hc3NpZ25fYWxsKGRmLCBjZmcuc2xvdF9wcmVmZXJfMmQsIGNmZy5zbG90X2ZzX3ByaW9yaXR5LCBjZmcuc2xvdF9jc3ZfZmFsbGJhY2spDQogICAgc2lkZXMgPSBwaXBlbGluZS5zdHVkeV9zaWRlcyhkZikNCiAgICByZWNvcmRzID0gcGlwZWxpbmUuaW5kZXhfdG9fcmVjb3Jkcyhhbm4pDQogICAgc2xvdF9yb3dzID0ge3M6IHRhYi5sb2Nbc10udG9fZGljdCgpIGZvciBzIGluIHN0dWRpZXMgaWYgcyBpbiB0YWIuaW5kZXh9DQogICAgcHJlZml4ID0gb3MucGF0aC5qb2luKHNjcmF0Y2gsICdjYWxpYl9jYWNoZScpDQogICAgY2FjaGUsIHN0ID0gcGNhY2hlLmJ1aWxkX2NhY2hlKHByZWZpeCwgc3R1ZGllcywgc2xvdF9yb3dzLCByZWNvcmRzLCBzaWRlcywgY2ZnLCB3b3JrZXJzLCByZXN1bWU9RmFsc2UsIHByb2dyZXNzPUZhbHNlKQ0KICAgIHJlc1snY2FjaGVfc19wZXJfc3R1ZHlfd2FsbCddID0gc3RbJ3NlY29uZHMnXSAvIGxlbihzdHVkaWVzKQ0KICAgIHJlc1snY2FjaGVfY3B1X3NfcGVyX3N0dWR5J10gPSBzdFsnc2Vjb25kcyddICogd29ya2VycyAvIGxlbihzdHVkaWVzKQ0KICAgIHJlc1snY2FjaGVfbWludXRlc19mdWxsJ10gPSByZXNbJ2NhY2hlX3NfcGVyX3N0dWR5X3dhbGwnXSAqIG5fdG90YWxfc3R1ZGllcyAvIDYwDQogICAgcmVzWydjYWNoZV9nYl9mdWxsJ10gPSBydW5uZXIuZXN0aW1hdGVfY2FjaGVfZ2Iobl90b3RhbF9zdHVkaWVzLCBjZmcpDQogICAgIyAtLS0tIDMuIHRyYWluaW5nIGxvb3AgZGF0YSByYXRlIChzaW5nbGUgcHJvY2Vzcywgd2l0aCBhdWdtZW50YXRpb24pDQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApDQogICAgaWR4ID0gW2kgZm9yIGkgaW4gcmFuZ2UobGVuKHN0dWRpZXMpKSBpZiBjYWNoZS5zbG90W2ldLmFueSgpXQ0KICAgIHQgPSB0aW1lLnRpbWUoKQ0KICAgIHJlcHMgPSAzDQogICAgZm9yIF8gaW4gcmFuZ2UocmVwcyk6DQogICAgICAgIGZvciBpIGluIGlkeDoNCiAgICAgICAgICAgIGxvYWRlci5tYWtlX3NhbXBsZShjYWNoZSwgaSwgY2ZnLCBUcnVlLCBuX3VzZSwgcm5nLCBUcnVlKQ0KICAgIHBlciA9ICh0aW1lLnRpbWUoKSAtIHQpIC8gKHJlcHMgKiBsZW4oaWR4KSkNCiAgICByZXNbJ2xvYWRlcl9tc19wZXJfc2FtcGxlXzFwcm9jJ10gPSBwZXIgKiAxMDAwDQogICAgcmVzWydsb2FkZXJfc2FtcGxlc19wZXJfc18xcHJvYyddID0gMS4wIC8gcGVyDQogICAgbmVlZCA9IGJhdGNoX3NpemUgLyBzdGVwX3NlY29uZHMNCiAgICByZXNbJ2xvYWRlcl9uZWVkZWRfc2FtcGxlc19wZXJfcyddID0gbmVlZA0KICAgIHJlc1snbG9hZGVyX2hlYWRyb29tX3gnXSA9ICgxLjAgLyBwZXIpICogbWluKDQsIHdvcmtlcnMpIC8gbmVlZA0KICAgIGZyZWUgPSBwY2FjaGUuc2h1dGlsLmRpc2tfdXNhZ2Uoc2NyYXRjaCkuZnJlZSAvIDFlOQ0KICAgIHJlc1snc2NyYXRjaF9mcmVlX2diJ10gPSBmcmVlDQogICAgcmVzWydzdHVkaWVzX3NhbXBsZWQnXSA9IGxlbihzdHVkaWVzKQ0KICAgIHJlc1snc2VyaWVzX3NhbXBsZWQnXSA9IGxlbihpdGVtcykNCiAgICBmb3IgayBpbiAoJ2NhbGliX2NhY2hlLmltYWdlcy51OCcsICdjYWxpYl9jYWNoZS52YWxpZC51MScsICdjYWxpYl9jYWNoZS5zbG90LnUxJywgJ2NhbGliX2NhY2hlLmRvbmUudTEnLCAnY2FsaWJfY2FjaGUubWV0YS5qc29uJywgJ2NhbGliX2NhY2hlLm1ldGEuY3N2Jyk6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIG9zLnJlbW92ZShvcy5wYXRoLmpvaW4oc2NyYXRjaCwgaykpDQogICAgICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICAgICAgcGFzcw0KICAgIHByaW50KGYic2FtcGxlOiB7bGVuKHN0dWRpZXMpfSBzdHVkaWVzIC8ge2xlbihpdGVtcyl9IHNlcmllcyAvIHtmaWxlc30gc2xpY2VzIG9uIHt3b3JrZXJzfSB3b3JrZXJzIChwcmVzZXQge2NmZy5uYW1lfSkiKQ0KICAgIHByaW50KGYiICBpbmRleCBzY2FuICAgOiB7cmVzWydpbmRleF9maWxlc19wZXJfcyddOi4wZn0gZmlsZXMvcyAgLT4gZnVsbCB0cmFpbiBzY2FuIH4ge3Jlc1snaW5kZXhfbWludXRlc19mdWxsJ106LjFmfSBtaW4iKQ0KICAgIHByaW50KGYiICBjYWNoZSBidWlsZCAgOiB7cmVzWydjYWNoZV9zX3Blcl9zdHVkeV93YWxsJ106LjJmfSBzL3N0dWR5IHdhbGwgKHtyZXNbJ2NhY2hlX2NwdV9zX3Blcl9zdHVkeSddOi4xZn0gY3B1LXMpIg0KICAgICAgICAgIGYiICAtPiBmdWxsIHRyYWluIGNhY2hlIH4ge3Jlc1snY2FjaGVfbWludXRlc19mdWxsJ106LjBmfSBtaW4sIHtyZXNbJ2NhY2hlX2diX2Z1bGwnXTouMWZ9IEdCIChzY3JhdGNoIGZyZWUge2ZyZWU6LjBmfSBHQikiKQ0KICAgIHByaW50KGYiICB0cmFpbiBsb2FkZXIgOiB7cmVzWydsb2FkZXJfbXNfcGVyX3NhbXBsZV8xcHJvYyddOi4wZn0gbXMvc2FtcGxlL3Byb2MgLT4ge3Jlc1snbG9hZGVyX2hlYWRyb29tX3gnXTouMWZ9eCBoZWFkcm9vbSAiDQogICAgICAgICAgZiJvdmVyIHRoZSB7bmVlZDouMGZ9IHNhbXBsZXMvcyBhIHtzdGVwX3NlY29uZHM6LjFmfXMgc3RlcCB3aXRoIGJhdGNoIHtiYXRjaF9zaXplfSBuZWVkcyAoNCB3b3JrZXJzKSIpDQogICAgaWYgcmVzWydjYWNoZV9nYl9mdWxsJ10gPiBmcmVlICogMC45MjoNCiAgICAgICAgcHJpbnQoJyAgISEgY2FjaGUgZG9lcyBub3QgZml0IHRoZSBzY3JhdGNoIGRpc2s6IHVzZSBydW5uZXIuZml0X2NhY2hlX2NmZyguLi4pIHRvIHBpY2sgYSBzbWFsbGVyIGltZ19zaXplL3N0YWNrX2RlcHRoJykNCiAgICByZXR1cm4gcmVzDQo=',
    'src/data/preprocess/cache.py': 'IiIiTWVtb3J5LW1hcHBlZCB1aW50OCBzdHVkeSBjYWNoZSArIGEgcGFyYWxsZWwgYnVpbGRlciB3aXRoIG5vIGludGVyLXByb2Nlc3MgYXJyYXkgdHJhZmZpYy4NCg0KTGF5b3V0IChvbmUgY29udGlndW91cyAxNiBNQiBibG9jayBwZXIgc3R1ZHkgYXQgRD0yNCwgUz02LCAzMzZeMiAtPiBvbmUgc2VxdWVudGlhbCByZWFkIHBlciBzYW1wbGUpOg0KICAgIDxwcmVmaXg+LmltYWdlcy51OCAgIFtOLCBTLCBELCBILCBXXSB1aW50OCAgICAgICAgPHByZWZpeD4udmFsaWQudTEgIFtOLCBTLCBEXQ0KICAgIDxwcmVmaXg+LnNsb3QudTEgICAgIFtOLCBTXSAgICAgICAgICAgICAgICAgICAgICAgPHByZWZpeD4uZG9uZS51MSAgIFtOXQ0KICAgIDxwcmVmaXg+Lm1ldGEuanNvbiAgIHN0dWR5IGlkcywgY2ZnLCBzaGFwZXMgICAgICAgPHByZWZpeD4ubWV0YS5jc3YgIHBlci1zdHVkeSBzaWRlL3NvdXJjZS9xYw0KDQpUaHJvdWdocHV0IGRlc2lnbjoNCiAgKiB3b3JrZXJzIHdyaXRlIHN0cmFpZ2h0IGludG8gdGhlIHNoYXJlZCBtZW1tYXAgKE1BUF9TSEFSRUQpIC0+IG5vdGhpbmcgYnV0IGEgdGlueSBkaWN0IGlzIHBpY2tsZWQgYmFjaw0KICAqIGxvbmdlc3QtcHJvY2Vzc2luZy10aW1lLWZpcnN0IHNjaGVkdWxpbmcgKDMtRCBzZXJpZXMgYXJlIH41eCBhIDItRCBvbmUpIHJlbW92ZXMgdGhlIHN0cmFnZ2xlciB0YWlsDQogICogcmVzdW1hYmxlOiBgZG9uZWAgZmxhZ3MgbGl2ZSBpbiBhIG1lbW1hcCwgc28gYSBraWxsZWQgcnVuIHJlc3RhcnRzIHdoZXJlIGl0IHN0b3BwZWQNCiAgKiBvcHRpb25hbCBzdHJlYW1pbmcgaG9vayAoYG9uX3JlYWR5YCkgbGV0cyBpbmZlcmVuY2UgcnVuIHRoZSBHUFUgb24gY2h1bmsgayB3aGlsZSBDUFVzIGJ1aWxkIGNodW5rIGsrMQ0KIiIiDQppbXBvcnQgb3MNCmltcG9ydCBqc29uDQppbXBvcnQgdGltZQ0KaW1wb3J0IHNodXRpbA0KaW1wb3J0IG11bHRpcHJvY2Vzc2luZyBhcyBtcA0KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgYXNkaWN0DQpmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgUHJvY2Vzc1Bvb2xFeGVjdXRvciwgVGhyZWFkUG9vbEV4ZWN1dG9yLCBhc19jb21wbGV0ZWQNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmltcG9ydCBzcmMuY29yZS5jb25maWcgYXMgY29uZmlnDQpmcm9tIC4gaW1wb3J0IHBpcGVsaW5lDQoNCl9HID0ge30NCg0KDQpkZWYgX3BhdGhzKHByZWZpeCk6DQogICAgcmV0dXJuIHtrOiBmJ3twcmVmaXh9Lnt2fScgZm9yIGssIHYgaW4gZGljdChpbWFnZXM9J2ltYWdlcy51OCcsIHZhbGlkPSd2YWxpZC51MScsIHNsb3Q9J3Nsb3QudTEnLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRvbmU9J2RvbmUudTEnLCBtZXRhPSdtZXRhLmpzb24nLCBjc3Y9J21ldGEuY3N2JykuaXRlbXMoKX0NCg0KDQpjbGFzcyBTdHVkeUNhY2hlOg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwcmVmaXgsIG1vZGU9J3InKToNCiAgICAgICAgc2VsZi5wcmVmaXgsIHNlbGYubW9kZSA9IHByZWZpeCwgbW9kZQ0KICAgICAgICBwID0gX3BhdGhzKHByZWZpeCkNCiAgICAgICAgd2l0aCBvcGVuKHBbJ21ldGEnXSkgYXMgZmg6DQogICAgICAgICAgICBzZWxmLm1ldGEgPSBqc29uLmxvYWQoZmgpDQogICAgICAgIHNlbGYuc3R1ZGllcyA9IHNlbGYubWV0YVsnc3R1ZGllcyddDQogICAgICAgIE4sIFMsIEQsIEggPSBzZWxmLm1ldGFbJ04nXSwgc2VsZi5tZXRhWydTJ10sIHNlbGYubWV0YVsnRCddLCBzZWxmLm1ldGFbJ0gnXQ0KICAgICAgICBtID0gJ3IrJyBpZiBtb2RlID09ICdyKycgZWxzZSAncicNCiAgICAgICAgc2VsZi5pbWFnZXMgPSBucC5tZW1tYXAocFsnaW1hZ2VzJ10sIG5wLnVpbnQ4LCBtLCBzaGFwZT0oTiwgUywgRCwgSCwgSCkpDQogICAgICAgIHNlbGYudmFsaWQgPSBucC5tZW1tYXAocFsndmFsaWQnXSwgbnAudWludDgsIG0sIHNoYXBlPShOLCBTLCBEKSkNCiAgICAgICAgc2VsZi5zbG90ID0gbnAubWVtbWFwKHBbJ3Nsb3QnXSwgbnAudWludDgsIG0sIHNoYXBlPShOLCBTKSkNCiAgICAgICAgc2VsZi5kb25lID0gbnAubWVtbWFwKHBbJ2RvbmUnXSwgbnAudWludDgsIG0sIHNoYXBlPShOLCkpDQogICAgICAgIHNlbGYuaW5kZXggPSB7czogaSBmb3IgaSwgcyBpbiBlbnVtZXJhdGUoc2VsZi5zdHVkaWVzKX0NCiAgICAgICAgIyBMSU5VWCAvIEdCMTAgU1BFRURVUDogSGludCBrZXJuZWwgZm9yIHN0YW5kYXJkIGNhY2hlZCBhY2Nlc3Mgc28gUkFNIGJ1ZmZlcnMgYWN0aXZlIHN0dWRpZXMNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgaW1wb3J0IG1tYXAgYXMgX3B5X21tYXANCiAgICAgICAgICAgICMgU09UQSBNZW1vcnkgT3B0aW1pemF0aW9uOiBGb3IgcmVhZCBtb2RlICh0cmFpbmluZy93b3JrZXJzKSwgdXNlIE1BRFZfUkFORE9NLg0KICAgICAgICAgICAgIyBUaGlzIHByZXZlbnRzIHRoZSBMaW51eCBrZXJuZWwgcGFnZS1jYWNoZSBmcm9tIGxvY2tpbmcgNjVHQiBvZiBkaXNrIHBhZ2VzIGluIFJBTS4NCiAgICAgICAgICAgIGFkdl9mbGFnID0gZ2V0YXR0cihfcHlfbW1hcCwgJ01BRFZfUkFORE9NJywgZ2V0YXR0cihfcHlfbW1hcCwgJ01BRFZfTk9STUFMJywgTm9uZSkpIGlmIG1vZGUgPT0gJ3InIGVsc2UgZ2V0YXR0cihfcHlfbW1hcCwgJ01BRFZfTk9STUFMJywgTm9uZSkNCiAgICAgICAgICAgIGlmIGFkdl9mbGFnIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgICAgIGZvciBhIGluIChzZWxmLmltYWdlcywgc2VsZi52YWxpZCwgc2VsZi5zbG90LCBzZWxmLmRvbmUpOg0KICAgICAgICAgICAgICAgICAgICBtbSA9IGdldGF0dHIoYSwgJ19tbWFwJywgTm9uZSkgb3IgZ2V0YXR0cihnZXRhdHRyKGEsICdiYXNlJywgTm9uZSksICdfbW1hcCcsIE5vbmUpDQogICAgICAgICAgICAgICAgICAgIGlmIG1tIGFuZCBoYXNhdHRyKG1tLCAnbWFkdmlzZScpOg0KICAgICAgICAgICAgICAgICAgICAgICAgbW0ubWFkdmlzZShhZHZfZmxhZykNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgIHBhc3MNCg0KICAgIGRlZiBfX2xlbl9fKHNlbGYpOg0KICAgICAgICByZXR1cm4gbGVuKHNlbGYuc3R1ZGllcykNCg0KICAgIEBzdGF0aWNtZXRob2QNCiAgICBkZWYgY3JlYXRlKHByZWZpeCwgc3R1ZGllcywgY2ZnKToNCiAgICAgICAgTiwgUywgRCwgSCA9IGxlbihzdHVkaWVzKSwgY29uZmlnLk5fU0xPVFMsIGNmZy5zdGFja19kZXB0aCwgY2ZnLmltZ19zaXplDQogICAgICAgIG5lZWQgPSBOICogUyAqIEQgKiBIICogSA0KICAgICAgICBkID0gb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChwcmVmaXgpKQ0KICAgICAgICBvcy5tYWtlZGlycyhkLCBleGlzdF9vaz1UcnVlKQ0KICAgICAgICBmcmVlID0gc2h1dGlsLmRpc2tfdXNhZ2UoZCkuZnJlZQ0KICAgICAgICBpZiBuZWVkICogMS4wMiA+IGZyZWU6DQogICAgICAgICAgICByYWlzZSBPU0Vycm9yKGYnY2FjaGUgbmVlZHMge25lZWQgLyAxZTk6LjFmfSBHQiBidXQgb25seSB7ZnJlZSAvIDFlOTouMWZ9IEdCIGZyZWUgaW4ge2R9JykNCiAgICAgICAgcCA9IF9wYXRocyhwcmVmaXgpDQogICAgICAgIG5wLm1lbW1hcChwWydpbWFnZXMnXSwgbnAudWludDgsICd3KycsIHNoYXBlPShOLCBTLCBELCBILCBIKSkuZmx1c2goKQ0KICAgICAgICBucC5tZW1tYXAocFsndmFsaWQnXSwgbnAudWludDgsICd3KycsIHNoYXBlPShOLCBTLCBEKSkuZmx1c2goKQ0KICAgICAgICBucC5tZW1tYXAocFsnc2xvdCddLCBucC51aW50OCwgJ3crJywgc2hhcGU9KE4sIFMpKS5mbHVzaCgpDQogICAgICAgIG5wLm1lbW1hcChwWydkb25lJ10sIG5wLnVpbnQ4LCAndysnLCBzaGFwZT0oTiwpKS5mbHVzaCgpDQogICAgICAgIHdpdGggb3BlbihwWydtZXRhJ10sICd3JykgYXMgZmg6DQogICAgICAgICAgICBqc29uLmR1bXAoZGljdChzdHVkaWVzPWxpc3Qoc3R1ZGllcyksIE49TiwgUz1TLCBEPUQsIEg9SCwgY2ZnPWFzZGljdChjZmcpKSwgZmgpDQogICAgICAgIHJldHVybiBTdHVkeUNhY2hlKHByZWZpeCwgJ3IrJykNCg0KICAgIGRlZiBzbG90X3N0YWNrKHNlbGYsIGksIHMpOg0KICAgICAgICByZXR1cm4gc2VsZi5pbWFnZXNbaSwgc10NCg0KICAgIGRlZiBmbHVzaChzZWxmKToNCiAgICAgICAgZm9yIGEgaW4gKHNlbGYuaW1hZ2VzLCBzZWxmLnZhbGlkLCBzZWxmLnNsb3QsIHNlbGYuZG9uZSk6DQogICAgICAgICAgICBhLmZsdXNoKCkNCg0KDQpkZWYgY2ZnX29mKHByZWZpeCk6DQogICAgIiIiVGhlIFByZUNmZyBhIGNhY2hlIHdhcyBidWlsdCB3aXRoIChzaW5nbGUgc291cmNlIG9mIHRydXRoIGZvciB0cmFpbmluZy9pbmZlcmVuY2Ugb24gdGhhdCBjYWNoZSkuIiIiDQogICAgd2l0aCBvcGVuKF9wYXRocyhwcmVmaXgpWydtZXRhJ10pIGFzIGZoOg0KICAgICAgICByZXR1cm4gY29uZmlnLmNmZ19mcm9tX2RpY3QoanNvbi5sb2FkKGZoKVsnY2ZnJ10pDQoNCg0KZGVmIF93b3JrKGkpOg0KICAgIGcgPSBfRw0KICAgIHN0ID0gZ1snc3R1ZGllcyddW2ldDQogICAgcmVzID0gcGlwZWxpbmUucHJlcGFyZV9zdHVkeShzdCwgZ1snc2xvdF9yb3dzJ10uZ2V0KHN0LCB7fSksIGdbJ3JlY29yZHMnXSwgZ1snc2lkZXMnXS5nZXQoc3QsIHt9KS5nZXQoJ3NpZGUnLCAnVScpLCBnWydjZmcnXSkNCiAgICBjID0gZ1snY2FjaGUnXQ0KICAgIGMuaW1hZ2VzW2ldID0gcmVzWydzdGFjayddDQogICAgYy52YWxpZFtpXSA9IHJlc1sndmFsaWQnXQ0KICAgIGMuc2xvdFtpXSA9IHJlc1snc2xvdF9tYXNrJ10NCiAgICBjLmRvbmVbaV0gPSAxDQogICAgZXJycyA9IHN1bShsZW4odi5nZXQoJ2Vycm9ycycsIFtdKSkgZm9yIHYgaW4gcmVzWydpbmZvJ10udmFsdWVzKCkpDQogICAgcGFkID0gc3VtKGJvb2wodi5nZXQoJ3BhZGRlZCcpKSBmb3IgdiBpbiByZXNbJ2luZm8nXS52YWx1ZXMoKSkNCiAgICBtaXNzID0gc3VtKGJvb2wodi5nZXQoJ3BzX21pc3NpbmcnKSkgZm9yIHYgaW4gcmVzWydpbmZvJ10udmFsdWVzKCkpDQogICAgcmV0dXJuIGksIGludChyZXNbJ3Nsb3RfbWFzayddLnN1bSgpKSwgZXJycywgcGFkLCBtaXNzDQoNCg0KZGVmIF9jb3N0KHN0dWR5LCBzbG90X3Jvd3MsIHJlY29yZHMpOg0KICAgIGMgPSAwDQogICAgZm9yIHVpZCBpbiBzbG90X3Jvd3MuZ2V0KHN0dWR5LCB7fSkudmFsdWVzKCk6DQogICAgICAgIHIgPSByZWNvcmRzLmdldCh1aWQpIGlmIHVpZCBlbHNlIE5vbmUNCiAgICAgICAgaWYgcjoNCiAgICAgICAgICAgIGMgKz0gKHIuZ2V0KCduX3NsaWNlcycpIG9yIDApICogKDUgaWYgc3RyKHIuZ2V0KCdNUkFjcXVpc2l0aW9uVHlwZScpKSA9PSAnM0QnIGVsc2UgMSkNCiAgICByZXR1cm4gYw0KDQoNCiMgRklYIEE6IEluaXRpYWxpemUgZ2xvYmFscyBmb3IgUHJvY2Vzc1Bvb2xFeGVjdXRvciB1bmRlciAic3Bhd24iIGNvbnRleHQNCmRlZiBfaW5pdF93b3JrZXIoc3R1ZGllcywgc2xvdF9yb3dzLCByZWNvcmRzLCBzaWRlcywgY2ZnLCBwcmVmaXgpOg0KICAgIGdsb2JhbCBfRw0KICAgIF9HLnVwZGF0ZShzdHVkaWVzPXN0dWRpZXMsIHNsb3Rfcm93cz1zbG90X3Jvd3MsIHJlY29yZHM9cmVjb3Jkcywgc2lkZXM9c2lkZXMsIGNmZz1jZmcsIGNhY2hlPVN0dWR5Q2FjaGUocHJlZml4LCAncisnKSkNCg0KDQpkZWYgYnVpbGRfY2FjaGUocHJlZml4LCBzdHVkaWVzLCBzbG90X3Jvd3MsIHJlY29yZHMsIHNpZGVzLCBjZmcsIHdvcmtlcnM9Tm9uZSwgcmVzdW1lPVRydWUsIG9yZGVyPSdscHQnLA0KICAgICAgICAgICAgICAgIGNodW5rPTY0LCBvbl9yZWFkeT1Ob25lLCBwcm9ncmVzcz1UcnVlKToNCiAgICAiIiJCdWlsZCAob3IgcmVzdW1lKSB0aGUgY2FjaGUuIFJldHVybnMgdGhlIG9wZW4gU3R1ZHlDYWNoZSAobW9kZSByKykgYW5kIGEgc3RhdHMgZGljdC4NCg0KICAgIHN0dWRpZXM6IG9yZGVyZWQgbGlzdCBvZiBTdHVkeUluc3RhbmNlVUlELiBzbG90X3Jvd3M6IHtzdHVkeToge3Nsb3Q6IHNlcmllcyB1aWR8Tm9uZX19Lg0KICAgIG9uX3JlYWR5KGNhY2hlLCBhLCBiKTogY2FsbGVkIGluIHRoZSBtYWluIHRocmVhZCB3aGVuZXZlciBzdHVkaWVzIFthLGIpIGFyZSBhbGwgZG9uZSAodXNlIG9yZGVyPSdzZXEnKS4iIiINCiAgICB3b3JrZXJzID0gd29ya2VycyBvciBtYXgoMiwgb3MuY3B1X2NvdW50KCkgb3IgMikNCiAgICBwID0gX3BhdGhzKHByZWZpeCkNCiAgICBpZiByZXN1bWUgYW5kIG9zLnBhdGguZXhpc3RzKHBbJ21ldGEnXSk6DQogICAgICAgIGNhY2hlID0gU3R1ZHlDYWNoZShwcmVmaXgsICdyKycpDQogICAgICAgIGlmIGNhY2hlLnN0dWRpZXMgIT0gbGlzdChzdHVkaWVzKSBvciBjYWNoZS5tZXRhWydjZmcnXSAhPSBqc29uLmxvYWRzKGpzb24uZHVtcHMoYXNkaWN0KGNmZykpKToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2V4aXN0aW5nIGNhY2hlIHdhcyBidWlsdCBmb3IgZGlmZmVyZW50IHN0dWRpZXMvY2ZnOyB1c2UgcmVzdW1lPUZhbHNlIG9yIGFub3RoZXIgcHJlZml4JykNCiAgICBlbHNlOg0KICAgICAgICBjYWNoZSA9IFN0dWR5Q2FjaGUuY3JlYXRlKHByZWZpeCwgc3R1ZGllcywgY2ZnKQ0KICAgIHRvZG8gPSBbaSBmb3IgaSBpbiByYW5nZShsZW4oc3R1ZGllcykpIGlmIG5vdCBjYWNoZS5kb25lW2ldXQ0KICAgIGlmIG9yZGVyID09ICdscHQnOg0KICAgICAgICB0b2RvLnNvcnQoa2V5PWxhbWJkYSBpOiAtX2Nvc3Qoc3R1ZGllc1tpXSwgc2xvdF9yb3dzLCByZWNvcmRzKSkNCiAgICBfRy51cGRhdGUoc3R1ZGllcz1saXN0KHN0dWRpZXMpLCBzbG90X3Jvd3M9c2xvdF9yb3dzLCByZWNvcmRzPXJlY29yZHMsIHNpZGVzPXNpZGVzLCBjZmc9Y2ZnLCBjYWNoZT1jYWNoZSkNCiAgICB0cnk6DQogICAgICAgIGV4ID0gUHJvY2Vzc1Bvb2xFeGVjdXRvcihtYXhfd29ya2Vycz13b3JrZXJzLCBtcF9jb250ZXh0PW1wLmdldF9jb250ZXh0KCdmb3JrJykpDQogICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgIyBGSVggQTogVXNlIHNwYXduIGluc3RlYWQgb2YgR0lMLWJsb2NrZWQgVGhyZWFkUG9vbEV4ZWN1dG9yDQogICAgICAgIGV4ID0gUHJvY2Vzc1Bvb2xFeGVjdXRvcigNCiAgICAgICAgICAgIG1heF93b3JrZXJzPXdvcmtlcnMsDQogICAgICAgICAgICBtcF9jb250ZXh0PW1wLmdldF9jb250ZXh0KCdzcGF3bicpLA0KICAgICAgICAgICAgaW5pdGlhbGl6ZXI9X2luaXRfd29ya2VyLA0KICAgICAgICAgICAgaW5pdGFyZ3M9KGxpc3Qoc3R1ZGllcyksIHNsb3Rfcm93cywgcmVjb3Jkcywgc2lkZXMsIGNmZywgcHJlZml4KQ0KICAgICAgICApDQogICAgdDAgPSB0aW1lLnRpbWUoKQ0KICAgIHN0YXRzID0gZGljdChzdHVkaWVzPWxlbihzdHVkaWVzKSwgYnVpbHQ9MCwgcmVzdW1lZD1sZW4oc3R1ZGllcykgLSBsZW4odG9kbyksIGRlY29kZV9lcnJvcnM9MCwgcGFkZGVkPTAsIHBzX21pc3Npbmc9MCwNCiAgICAgICAgICAgICAgICAgZW1wdHlfc3R1ZGllcz0wKQ0KICAgIGZ1dHMgPSB7ZXguc3VibWl0KF93b3JrLCBpKTogaSBmb3IgaSBpbiB0b2RvfQ0KICAgIHBlbmRpbmcgPSB7fQ0KICAgIGZvciBhIGluIHJhbmdlKDAsIGxlbihzdHVkaWVzKSwgY2h1bmspOg0KICAgICAgICBwZW5kaW5nW2FdID0gc3VtKDEgZm9yIGkgaW4gcmFuZ2UoYSwgbWluKGEgKyBjaHVuaywgbGVuKHN0dWRpZXMpKSkgaWYgbm90IGNhY2hlLmRvbmVbaV0pDQogICAgZm9yIGEsIGxlZnQgaW4gbGlzdChwZW5kaW5nLml0ZW1zKCkpOg0KICAgICAgICBpZiBsZWZ0ID09IDAgYW5kIG9uX3JlYWR5Og0KICAgICAgICAgICAgb25fcmVhZHkoY2FjaGUsIGEsIG1pbihhICsgY2h1bmssIGxlbihzdHVkaWVzKSkpDQogICAgICAgIGlmIGxlZnQgPT0gMDoNCiAgICAgICAgICAgIHBlbmRpbmcucG9wKGEpDQogICAgZm9yIGssIGYgaW4gZW51bWVyYXRlKGFzX2NvbXBsZXRlZChmdXRzKSk6DQogICAgICAgIGksIG5zbG90LCBlcnJzLCBwYWQsIG1pc3MgPSBmLnJlc3VsdCgpDQogICAgICAgIHN0YXRzWydidWlsdCddICs9IDENCiAgICAgICAgc3RhdHNbJ2RlY29kZV9lcnJvcnMnXSArPSBlcnJzDQogICAgICAgIHN0YXRzWydwYWRkZWQnXSArPSBwYWQNCiAgICAgICAgc3RhdHNbJ3BzX21pc3NpbmcnXSArPSBtaXNzDQogICAgICAgIHN0YXRzWydlbXB0eV9zdHVkaWVzJ10gKz0gaW50KG5zbG90ID09IDApDQogICAgICAgIGEgPSAoaSAvLyBjaHVuaykgKiBjaHVuaw0KICAgICAgICBpZiBhIGluIHBlbmRpbmc6DQogICAgICAgICAgICBwZW5kaW5nW2FdIC09IDENCiAgICAgICAgICAgIGlmIHBlbmRpbmdbYV0gPT0gMDoNCiAgICAgICAgICAgICAgICBpZiBvbl9yZWFkeToNCiAgICAgICAgICAgICAgICAgICAgb25fcmVhZHkoY2FjaGUsIGEsIG1pbihhICsgY2h1bmssIGxlbihzdHVkaWVzKSkpDQogICAgICAgICAgICAgICAgcGVuZGluZy5wb3AoYSkNCiAgICAgICAgaWYgcHJvZ3Jlc3MgYW5kIChrICsgMSkgJSAyMDAgPT0gMDoNCiAgICAgICAgICAgIGVsYXBzZWQgPSB0aW1lLnRpbWUoKSAtIHQwDQogICAgICAgICAgICB0aHJvdWdocHV0ID0gKGsgKyAxKSAvIGVsYXBzZWQNCiAgICAgICAgICAgIGxlZnQgPSBsZW4odG9kbykgLSAoayArIDEpDQogICAgICAgICAgICBldGEgPSBsZWZ0IC8gdGhyb3VnaHB1dCBpZiB0aHJvdWdocHV0ID4gMCBlbHNlIDANCiAgICAgICAgICAgIGV0YV9tLCBldGFfcyA9IGRpdm1vZChpbnQoZXRhKSwgNjApDQogICAgICAgICAgICBldGFfaCwgZXRhX20gPSBkaXZtb2QoZXRhX20sIDYwKQ0KICAgICAgICAgICAgcHJpbnQoZicgIGNhY2hlIHtrICsgMX0ve2xlbih0b2RvKX0gfCB7dGhyb3VnaHB1dDouMWZ9IHN0dWRpZXMvcyB8IEVUQToge2V0YV9oOjAyZH06e2V0YV9tOjAyZH06e2V0YV9zOjAyZH0nLCBmbHVzaD1UcnVlKQ0KICAgIGV4LnNodXRkb3duKCkNCiAgICBjYWNoZS5mbHVzaCgpDQogICAgc3RhdHNbJ3NlY29uZHMnXSA9IHRpbWUudGltZSgpIC0gdDANCiAgICBwZC5EYXRhRnJhbWUoZGljdChTdHVkeUluc3RhbmNlVUlEPXN0dWRpZXMsIHNpZGU9W3NpZGVzLmdldChzLCB7fSkuZ2V0KCdzaWRlJywgJ1UnKSBmb3IgcyBpbiBzdHVkaWVzXSwNCiAgICAgICAgICAgICAgICAgICAgICBzaWRlX3NvdXJjZT1bc2lkZXMuZ2V0KHMsIHt9KS5nZXQoJ3NvdXJjZScsICdub25lJykgZm9yIHMgaW4gc3R1ZGllc10sDQogICAgICAgICAgICAgICAgICAgICAgbl9zbG90cz1ucC5hc2FycmF5KGNhY2hlLnNsb3QpLnN1bSgxKSkpLnRvX2NzdihwWydjc3YnXSwgaW5kZXg9RmFsc2UpDQogICAgIyBNRU1PUlkgQVVESVQgRklYOiBQdXJnZSBtb2R1bGUtbGV2ZWwgZ2xvYmFsIGRpY3QgdG8gcmVsZWFzZSBhbGwgcmVjb3JkcyBhbmQgc2VyaWVzIG1ldGFkYXRhDQogICAgX0cuY2xlYXIoKQ0KICAgIGltcG9ydCBnYw0KICAgIGdjLmNvbGxlY3QoKQ0KICAgIHJldHVybiBjYWNoZSwgc3RhdHMNCg==',
    'src/data/preprocess/dicomio.py': 'IiIiUm9idXN0IERJQ09NIGhlYWRlciAvIHBpeGVsIGFjY2Vzcy4NCg0KRGF0YXNldCBmYWN0cyB0aGlzIG1vZHVsZSBpcyBidWlsdCBhcm91bmQgKHZlcmlmaWVkIG9uIGFsbCAyNCwzNzEgdHJhaW4gc2VyaWVzKToNCiAgKiBldmVyeSBmaWxlIGlzIHVuY29tcHJlc3NlZCBFeHBsaWNpdCBWUiBMRSwgYnV0IHRoZSBoaWRkZW4gdGVzdCBzZXQgaXMgbm90IGd1YXJhbnRlZWQgdG8gYmUgLT4gZGVjb2RlIGVycm9ycw0KICAgIGFyZSBjYXVnaHQgcGVyIHNsaWNlLCBuZXZlciBmYXRhbA0KICAqIDI2JSBvZiBzZXJpZXMgYXJlIHNpZ25lZCBpbnQxNiwgQ2Fub24gc2VyaWVzIGNvbnRhaW4gbmVnYXRpdmUgcGl4ZWxzIChkb3duIHRvIC0xODcxKQ0KICAqIFJlc2NhbGVTbG9wZS9JbnRlcmNlcHQgYXJlIGNvbnN0YW50IGluc2lkZSBhIHNlcmllcyAoUGhpbGlwcyBzbG9wZSByZWFjaGVzIDI2MikNCiAgKiBESUNPTSBXaW5kb3dDZW50ZXIvV2lkdGggdmFyeSBwZXIgc2xpY2UgaW4gNTIlIG9mIHNlcmllcyAtPiBuZXZlciB1c2VkDQoiIiINCmltcG9ydCBvcw0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcHlkaWNvbQ0KDQpPUkRFUl9UQUdTID0gWydJbWFnZVBvc2l0aW9uUGF0aWVudCcsICdJbWFnZU9yaWVudGF0aW9uUGF0aWVudCcsICdJbnN0YW5jZU51bWJlcicsDQogICAgICAgICAgICAgICdSb3dzJywgJ0NvbHVtbnMnLCAnUGl4ZWxTcGFjaW5nJ10NCg0KUkVQX1RBR1MgPSBbJ1Nlcmllc0Rlc2NyaXB0aW9uJywgJ1NlcXVlbmNlTmFtZScsICdTY2FuT3B0aW9ucycsICdTY2FubmluZ1NlcXVlbmNlJywgJ1JlcGV0aXRpb25UaW1lJywgJ0VjaG9UaW1lJywNCiAgICAgICAgICAgICdJbnZlcnNpb25UaW1lJywgJ01SQWNxdWlzaXRpb25UeXBlJywgJ0xhdGVyYWxpdHknLCAnSW1hZ2VMYXRlcmFsaXR5JywgJ1NsaWNlVGhpY2tuZXNzJywNCiAgICAgICAgICAgICdTcGFjaW5nQmV0d2VlblNsaWNlcycsICdNYW51ZmFjdHVyZXInLCAnTWFudWZhY3R1cmVyTW9kZWxOYW1lJywgJ01hZ25ldGljRmllbGRTdHJlbmd0aCcsDQogICAgICAgICAgICAnQml0c1N0b3JlZCcsICdQaXhlbFJlcHJlc2VudGF0aW9uJywgJ1Bob3RvbWV0cmljSW50ZXJwcmV0YXRpb24nLCAnQm9keVBhcnRFeGFtaW5lZCddDQoNCg0KZGVmIGYxKHYpOg0KICAgICIiIkZpcnN0IGZsb2F0IG9mIGEgc2NhbGFyIC8gbXVsdGktdmFsdWUsIGVsc2UgTm9uZS4iIiINCiAgICB0cnk6DQogICAgICAgIHJldHVybiBmbG9hdCh2KQ0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHJldHVybiBmbG9hdChsaXN0KHYpWzBdKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICAgICAgcmV0dXJuIE5vbmUNCg0KDQpkZWYgZmxvYXRzKHYsIG4pOg0KICAgIHRyeToNCiAgICAgICAgYSA9IFtmbG9hdCh4KSBmb3IgeCBpbiB2XQ0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHJldHVybiBOb25lDQogICAgcmV0dXJuIGEgaWYgbGVuKGEpID09IG4gYW5kIGFsbChucC5pc2Zpbml0ZShhKSkgZWxzZSBOb25lDQoNCg0KZGVmIHN2YWwodik6DQogICAgIiIiSGVhZGVyIHZhbHVlIC0+IHBsYWluIHN0ciAoJ3wnIGpvaW5lZCBmb3IgbXVsdGktdmFsdWVzKSwgTm9uZSBpZiBhYnNlbnQuIiIiDQogICAgaWYgdiBpcyBOb25lOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIGlmIGlzaW5zdGFuY2UodiwgKGxpc3QsIHR1cGxlKSkgb3Igdi5fX2NsYXNzX18uX19uYW1lX18gPT0gJ011bHRpVmFsdWUnOg0KICAgICAgICByZXR1cm4gJ3wnLmpvaW4oc3RyKHgpIGZvciB4IGluIHYpDQogICAgcmV0dXJuIHN0cih2KQ0KDQoNCmRlZiByZWFkX3RhZ3MocGF0aCwgdGFncyk6DQogICAgdHJ5Og0KICAgICAgICBkcyA9IHB5ZGljb20uZGNtcmVhZChwYXRoLCBzdG9wX2JlZm9yZV9waXhlbHM9VHJ1ZSwgZm9yY2U9VHJ1ZSwgc3BlY2lmaWNfdGFncz1saXN0KHRhZ3MpKQ0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHJldHVybiBOb25lDQogICAgcmV0dXJuIGRzDQoNCg0KZGVmIHJlcHJlc2VudGF0aXZlX2hlYWRlcihwYXRoKToNCiAgICAiIiJGdWxsIChwaXhlbC1sZXNzKSBoZWFkZXIgb2Ygb25lIGZpbGUgLT4gZGljdCBvZiBSRVBfVEFHUyAoc3RyaW5ncykgKyBudW1lcmljIGhlbHBlcnMuIiIiDQogICAgdHJ5Og0KICAgICAgICBkcyA9IHB5ZGljb20uZGNtcmVhZChwYXRoLCBzdG9wX2JlZm9yZV9waXhlbHM9VHJ1ZSwgZm9yY2U9VHJ1ZSkNCiAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIG91dCA9IHt0OiBzdmFsKGdldGF0dHIoZHMsIHQsIE5vbmUpKSBmb3IgdCBpbiBSRVBfVEFHU30NCiAgICBwcyA9IGZsb2F0cyhnZXRhdHRyKGRzLCAnUGl4ZWxTcGFjaW5nJywgTm9uZSksIDIpDQogICAgb3V0Wydwc19yb3cnXSwgb3V0Wydwc19jb2wnXSA9IChwc1swXSwgcHNbMV0pIGlmIHBzIGVsc2UgKG5wLm5hbiwgbnAubmFuKQ0KICAgIG91dFsndHJhbnNmZXJfc3ludGF4J10gPSBzdHIoZ2V0YXR0cihnZXRhdHRyKGRzLCAnZmlsZV9tZXRhJywgTm9uZSksICdUcmFuc2ZlclN5bnRheFVJRCcsICcnKSkNCiAgICByZXR1cm4gb3V0DQoNCg0KZGVmIGRlY29kZV9zbGljZShwYXRoKToNCiAgICAiIiJEZWNvZGUgb25lIHNsaWNlIC0+IGZsb2F0MzIgMi1EIGFycmF5IHdpdGggUmVzY2FsZVNsb3BlL0ludGVyY2VwdCBhcHBsaWVkLg0KDQogICAgUmFpc2VzIG9uIGFueSBmYWlsdXJlIChjYWxsZXIgZGVjaWRlcyB0aGUgZmFsbGJhY2spLiBNT05PQ0hST01FMSBpcyBpbnZlcnRlZC4gTm9uLWZpbml0ZSB2YWx1ZXMgLT4gMC4NCiAgICAiIiINCiAgICAjIE9QVElNSVpBVElPTjogQnlwYXNzaW5nIGhlYXZ5IG1ldGFkYXRhIHBhcnNpbmcuDQogICAgIyBPbmx5IHBhcnNlIHRoZSBleGFjdCB0YWdzIG5lZWRlZCB0byBkZWNvbXByZXNzIHBpeGVscyBhbmQgYXBwbHkgbGluZWFyIHJlc2NhbGUuDQogICAgIyBTYXZlcyB+MzAtNTAlIENQVSBwYXJzaW5nIHRpbWUgcGVyIHNsaWNlIChjcml0aWNhbCBmb3IgREdYIExpbnV4IG11bHRpLWNvcmUgc2NhbGluZykuDQogICAgdGFncyA9IFsnUGl4ZWxEYXRhJywgJ1Jvd3MnLCAnQ29sdW1ucycsICdCaXRzQWxsb2NhdGVkJywgJ0JpdHNTdG9yZWQnLCAnSGlnaEJpdCcsDQogICAgICAgICAgICAnUGl4ZWxSZXByZXNlbnRhdGlvbicsICdTYW1wbGVzUGVyUGl4ZWwnLCAnUGhvdG9tZXRyaWNJbnRlcnByZXRhdGlvbicsDQogICAgICAgICAgICAnUmVzY2FsZVNsb3BlJywgJ1Jlc2NhbGVJbnRlcmNlcHQnXQ0KICAgIGRzID0gcHlkaWNvbS5kY21yZWFkKHBhdGgsIGZvcmNlPVRydWUsIHNwZWNpZmljX3RhZ3M9dGFncykNCiAgICBhID0gZHMucGl4ZWxfYXJyYXkNCiAgICBpZiBhLm5kaW0gIT0gMjoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ2V4cGVjdGVkIGEgMi1EIHNsaWNlLCBnb3QgbmRpbT17YS5uZGltfScpDQogICAgYSA9IGEuYXN0eXBlKG5wLmZsb2F0MzIpDQogICAgc2xvcGUgPSBmMShnZXRhdHRyKGRzLCAnUmVzY2FsZVNsb3BlJywgTm9uZSkpDQogICAgaWNwdCA9IGYxKGdldGF0dHIoZHMsICdSZXNjYWxlSW50ZXJjZXB0JywgTm9uZSkpDQogICAgYSA9IGEgKiAoc2xvcGUgaWYgc2xvcGUgZWxzZSAxLjApICsgKGljcHQgaWYgaWNwdCBlbHNlIDAuMCkNCiAgICBpZiBzdHIoZ2V0YXR0cihkcywgJ1Bob3RvbWV0cmljSW50ZXJwcmV0YXRpb24nLCAnJykpLnVwcGVyKCkgPT0gJ01PTk9DSFJPTUUxJzoNCiAgICAgICAgYSA9IGEubWF4KCkgLSBhDQogICAgaWYgbm90IG5wLmlzZmluaXRlKGEpLmFsbCgpOg0KICAgICAgICBhID0gbnAubmFuX3RvX251bShhLCBuYW49MC4wLCBwb3NpbmY9MC4wLCBuZWdpbmY9MC4wKQ0KICAgIHJldHVybiBhDQoNCg0KZGVmIGxpc3RfZGljb21zKHNlcmllc19kaXIpOg0KICAgIHRyeToNCiAgICAgICAgd2l0aCBvcy5zY2FuZGlyKHNlcmllc19kaXIpIGFzIGl0Og0KICAgICAgICAgICAgcmV0dXJuIHNvcnRlZChlLm5hbWUgZm9yIGUgaW4gaXQgaWYgZS5uYW1lLmxvd2VyKCkuZW5kc3dpdGgoJy5kY20nKSkNCiAgICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgcmV0dXJuIFtdDQoNCg0KZGVmIGRlY29kZV9yYXcocGF0aCk6DQogICAgIiIiRmFzdCBkZWNvZGU6IFJBVyBpbnRlZ2VyIHBpeGVscyArIHJlc2NhbGUgcGFyYW1zLg0KICAgIE9wdGltaXplZCB0byBkeW5hbWljYWxseSB1c2UgZGljb21zZGwgKEMrKyBiYXNlZCkgb24gREdYIGlmIGF2YWlsYWJsZSwgcHJvdmlkaW5nIGEgbWFzc2l2ZSBzcGVlZHVwICh+MW1zIHZzIDEybXMpLiIiIg0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IGRpY29tc2RsDQogICAgICAgIGRzID0gZGljb21zZGwub3BlbihwYXRoKQ0KICAgICAgICAjIFNPVEEgRml4OiBNdXN0IHVzZSBzdG9yZWR2YWx1ZT1UcnVlIHRvIGdldCByYXcgaW50ZWdlcnMsIG90aGVyd2lzZSBkaWNvbXNkbCBkb3VibGUtYXBwbGllcyBzbG9wZS9pY3B0IQ0KICAgICAgICBhID0gZHMucGl4ZWxEYXRhKHN0b3JlZHZhbHVlPVRydWUpDQogICAgICAgIA0KICAgICAgICAjIFBob3RvbWV0cmljIEludGVycHJldGF0aW9uIGNoZWNrIGlzIENSSVRJQ0FMDQogICAgICAgIHRyeToNCiAgICAgICAgICAgICMgU09UQSBGaXg6IGRpY29tc2RsIGRvZXMgbm90IGhhdmUgLmluZm8oKS4gVXNlIGRpcmVjdCBhdHRyaWJ1dGUgYWNjZXNzLg0KICAgICAgICAgICAgcGhvdG8gPSBnZXRhdHRyKGRzLCAnUGhvdG9tZXRyaWNJbnRlcnByZXRhdGlvbicsICcnKQ0KICAgICAgICAgICAgaWYgJ01PTk9DSFJPTUUxJyBpbiBzdHIocGhvdG8pLnVwcGVyKCk6DQogICAgICAgICAgICAgICAgYSA9IGEubWF4KCkgLSBhDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICBwYXNzDQogICAgICAgICAgICANCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgc2xvcGUgPSBmbG9hdChkcy5SZXNjYWxlU2xvcGUpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICBzbG9wZSA9IDEuMA0KICAgICAgICAgICAgDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGljcHQgPSBmbG9hdChkcy5SZXNjYWxlSW50ZXJjZXB0KQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICAgICAgaWNwdCA9IDAuMA0KICAgICAgICAgICAgDQogICAgICAgIHJldHVybiBhLCBzbG9wZSwgaWNwdA0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHBhc3MNCg0KICAgICMgRmFsbGJhY2sgdG8gc3BlY2lmaWNfdGFncyBvcHRpbWl6ZWQgcHlkaWNvbQ0KICAgIHRhZ3MgPSBbJ1BpeGVsRGF0YScsICdSb3dzJywgJ0NvbHVtbnMnLCAnQml0c0FsbG9jYXRlZCcsICdCaXRzU3RvcmVkJywgJ0hpZ2hCaXQnLA0KICAgICAgICAgICAgJ1BpeGVsUmVwcmVzZW50YXRpb24nLCAnU2FtcGxlc1BlclBpeGVsJywgJ1Bob3RvbWV0cmljSW50ZXJwcmV0YXRpb24nLA0KICAgICAgICAgICAgJ1Jlc2NhbGVTbG9wZScsICdSZXNjYWxlSW50ZXJjZXB0J10NCiAgICBkcyA9IHB5ZGljb20uZGNtcmVhZChwYXRoLCBmb3JjZT1UcnVlLCBzcGVjaWZpY190YWdzPXRhZ3MpDQogICAgYSA9IGRzLnBpeGVsX2FycmF5DQogICAgaWYgYS5uZGltICE9IDI6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZidleHBlY3RlZCBhIDItRCBzbGljZSwgZ290IG5kaW09e2EubmRpbX0nKQ0KICAgIGlmIHN0cihnZXRhdHRyKGRzLCAnUGhvdG9tZXRyaWNJbnRlcnByZXRhdGlvbicsICcnKSkudXBwZXIoKSA9PSAnTU9OT0NIUk9NRTEnOg0KICAgICAgICBhID0gYS5tYXgoKSAtIGENCiAgICBzbG9wZSA9IGdldGF0dHIoZHMsICdSZXNjYWxlU2xvcGUnLCAxLjApDQogICAgaWNwdCA9IGdldGF0dHIoZHMsICdSZXNjYWxlSW50ZXJjZXB0JywgMC4wKQ0KICAgIHJldHVybiBhLCBmbG9hdChzbG9wZSkgaWYgc2xvcGUgaXMgbm90IE5vbmUgZWxzZSAxLjAsIGZsb2F0KGljcHQpIGlmIGljcHQgaXMgbm90IE5vbmUgZWxzZSAwLjANCg==',
    'src/data/preprocess/geometry.py': 'IiIiU2xpY2Ugb3JkZXJpbmcsIG9yaWVudGF0aW9uLCBsYXRlcmFsaXR5Lg0KDQpXaHkgdGhpcyBleGlzdHMgKHZlcmlmaWVkIG9uIHRoZSBkYXRhc2V0KTogZmlsZSBuYW1lcyBhcmUgcHlkaWNvbS1nZW5lcmF0ZWQgcmFuZG9tIFVJRHMsIHNvIG5laXRoZXINCmBzb3J0ZWQob3MubGlzdGRpcigpKWAgbm9yIGFueSBmaWxlbmFtZSBvcmRlciBjYXJyaWVzIGFuYXRvbWljYWwgcG9zaXRpb24gKG9ic2VydmVkIFAoZmlyc3QgZmlsZSA9PSBzcGF0aWFsDQpjZW50cmUpID0gMy42NyUgdnMgMy41MSUgZXhwZWN0ZWQgZm9yIHJhbmRvbSBvcmRlciwgeiA9IDEuMzYsIG9uIGFsbCAyNCwzNzEgc2VyaWVzKS4gU2xpY2Ugb3JkZXIgbXVzdCBjb21lIGZyb20NCkltYWdlUG9zaXRpb25QYXRpZW50IHByb2plY3RlZCBvbiB0aGUgc2xpY2Ugbm9ybWFsLg0KIiIiDQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCg0KX0xFVCA9IHsoMCwgMSk6ICdMJywgKDAsIC0xKTogJ1InLCAoMSwgMSk6ICdQJywgKDEsIC0xKTogJ0EnLCAoMiwgMSk6ICdTJywgKDIsIC0xKTogJ0knfQ0KX0FYSVMgPSB7J0wnOiAwLCAnUic6IDAsICdQJzogMSwgJ0EnOiAxLCAnUyc6IDIsICdJJzogMn0NClBMQU5FUyA9ICgnU2FnaXR0YWwnLCAnQ29yb25hbCcsICdBeGlhbCcpDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gb3JpZW50YXRpb24NCmRlZiBvcmllbnRfaW5mbyhpb3ApOg0KICAgICIiIi0+IChwbGFuZSBmcm9tIHNsaWNlIG5vcm1hbCwgfG5vcm1hbHwgYWxpZ25tZW50IHdpdGggaXRzIGF4aXMsIDItbGV0dGVyIExQUyBjb2RlKS4NCg0KICAgIENvZGUgbGV0dGVyIDEgPSBkaXJlY3Rpb24gaW1hZ2UgQ09MVU1OUyBpbmNyZWFzZSB0b3dhcmQ7IGxldHRlciAyID0gZGlyZWN0aW9uIGltYWdlIFJPV1MgaW5jcmVhc2UgdG93YXJkLg0KICAgIERhdGFzZXQgc3RhbmRhcmQ6IFNhZ2l0dGFsICdQSScsIENvcm9uYWwgJ0xJJywgQXhpYWwgJ0xQJy4NCiAgICAiIiINCiAgICBpZiBpb3AgaXMgTm9uZSBvciBsZW4oaW9wKSAhPSA2Og0KICAgICAgICByZXR1cm4gTm9uZSwgZmxvYXQoJ25hbicpLCBOb25lDQogICAgciwgYyA9IG5wLmFzYXJyYXkoaW9wWzozXSwgZmxvYXQpLCBucC5hc2FycmF5KGlvcFszOl0sIGZsb2F0KQ0KICAgIG4gPSBucC5jcm9zcyhyLCBjKQ0KICAgIG5uID0gbnAubGluYWxnLm5vcm0obikNCiAgICBpZiBub3QgbnAuaXNmaW5pdGUobm4pIG9yIG5uIDwgMWUtNjoNCiAgICAgICAgcmV0dXJuIE5vbmUsIGZsb2F0KCduYW4nKSwgTm9uZQ0KICAgIG4gPSBuIC8gbm4NCiAgICBwbGFuZSA9IFBMQU5FU1tpbnQobnAuYXJnbWF4KG5wLmFicyhuKSkpXQ0KICAgIGNvZGUgPSAnJw0KICAgIGZvciB2IGluIChyLCBjKToNCiAgICAgICAgYXggPSBpbnQobnAuYXJnbWF4KG5wLmFicyh2KSkpDQogICAgICAgIGNvZGUgKz0gX0xFVFsoYXgsIDEgaWYgdltheF0gPj0gMCBlbHNlIC0xKV0NCiAgICByZXR1cm4gcGxhbmUsIGZsb2F0KG5wLmFicyhuKS5tYXgoKSksIGNvZGUNCg0KDQpkZWYgc2xpY2Vfbm9ybWFsKGlvcCk6DQogICAgciwgYyA9IG5wLmFzYXJyYXkoaW9wWzozXSwgZmxvYXQpLCBucC5hc2FycmF5KGlvcFszOl0sIGZsb2F0KQ0KICAgIG4gPSBucC5jcm9zcyhyLCBjKQ0KICAgIHJldHVybiBuIC8gKG5wLmxpbmFsZy5ub3JtKG4pICsgMWUtMTIpDQoNCg0KZGVmIGNlbnRlcl94eXooaXBwLCBpb3AsIHBzLCByb3dzLCBjb2xzKToNCiAgICAiIiJQYXRpZW50LXNwYWNlIGNlbnRyZSBvZiBhIHNsaWNlIChtbSkuIiIiDQogICAgaXBwLCBpb3AgPSBucC5hc2FycmF5KGlwcCwgZmxvYXQpLCBucC5hc2FycmF5KGlvcCwgZmxvYXQpDQogICAgcmV0dXJuIGlwcCArIGlvcFs6M10gKiBwc1sxXSAqIChjb2xzIC0gMSkgLyAyLjAgKyBpb3BbMzpdICogcHNbMF0gKiAocm93cyAtIDEpIC8gMi4wDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gb3JkZXJpbmcNCmRlZiBvcmRlcl9yZWNvcmRzKHJlY3MsIG9yZGVyX21vZGU9J3Bvc2l0aW9uJyk6DQogICAgIiIiT3JkZXIgb25lIHNlcmllcycgc2xpY2VzIGJ5IGFuYXRvbWljYWwgcG9zaXRpb24uDQoNCiAgICByZWNzOiBsaXN0IG9mIGRpY3RzIHtuYW1lLCBpcHAsIGlvcCwgaW5zdCwgcm93cywgY29scywgcHN9ICAoTm9uZSB3aGVyZSBhIHRhZyBpcyBtaXNzaW5nKQ0KICAgIG9yZGVyX21vZGU6DQogICAgICAncG9zaXRpb24nIDoga2V5ID0gc2lnbihuW2F4aXNdKSAqIGRvdChpcHAsIG4pLCBhc2NlbmRpbmcgIC0+IGluY3JlYXNpbmcgYWxvbmcgK0wgLyArUCAvICtTDQogICAgICAgICAgICAgICAgICAgKGRpcmVjdGlvbiBpcyBpbmRlcGVuZGVudCBvZiB0aGUgdmVuZG9yJ3MgSU9QIHNpZ24gY29udmVudGlvbikNCiAgICAgICdub3JtYWwnICAgOiBrZXkgPSBkb3QoaXBwLCByIHggYykgYXNjZW5kaW5nIChwdWJsaWMtbm90ZWJvb2sgY29udmVudGlvbiwgbm8gc2lnbiBub3JtYWxpc2F0aW9uKQ0KICAgIEZhbGxzIGJhY2sgdG8gSW5zdGFuY2VOdW1iZXIsIHRoZW4gbmF0dXJhbCBuYW1lIG9yZGVyLCBhbmQgcmVwb3J0cyB3aGljaCBvbmUgd2FzIHVzZWQuDQogICAgRmlsZXMgd2hvc2UgbWF0cml4IGRpZmZlcnMgZnJvbSB0aGUgbWFqb3JpdHkgc2hhcGUgYXJlIGRyb3BwZWQgKGFuZCBjb3VudGVkKS4NCiAgICBSZXR1cm5zIGRpY3Qob3JkZXJlZCwgcG9zLCBtZXRob2QsIG5fZHJvcHBlZF9zaGFwZSwgcGxhbmUsIGNvZGUsIGFsaWduLCBzcGFjaW5nX21lZCwgel9leHRlbnQsIHNoYXBlLCBwcywNCiAgICAgICAgICAgICAgICAgY2VudGVyX3gsIHJlcF9pbmRleCkgb3IgTm9uZSB3aGVuIHRoZXJlIGFyZSBubyByZWFkYWJsZSBmaWxlcy4NCiAgICAiIiINCiAgICByZWNzID0gW3IgZm9yIHIgaW4gcmVjcyBpZiByIGlzIG5vdCBOb25lXQ0KICAgIGlmIG5vdCByZWNzOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHNoYXBlcyA9IENvdW50ZXIoKHJbJ3Jvd3MnXSwgclsnY29scyddKSBmb3IgciBpbiByZWNzKQ0KICAgIHNoYXBlID0gc2hhcGVzLm1vc3RfY29tbW9uKDEpWzBdWzBdDQogICAgbl9kcm9wID0gc3VtKDEgZm9yIHIgaW4gcmVjcyBpZiAoclsncm93cyddLCByWydjb2xzJ10pICE9IHNoYXBlKQ0KICAgIHJlY3MgPSBbciBmb3IgciBpbiByZWNzIGlmIChyWydyb3dzJ10sIHJbJ2NvbHMnXSkgPT0gc2hhcGVdDQogICAgaW9wcyA9IENvdW50ZXIodHVwbGUobnAucm91bmQoclsnaW9wJ10sIDMpKSBmb3IgciBpbiByZWNzIGlmIHJbJ2lvcCddIGlzIG5vdCBOb25lKQ0KICAgIGlvcCA9IG5wLmFycmF5KGlvcHMubW9zdF9jb21tb24oMSlbMF1bMF0pIGlmIGlvcHMgZWxzZSBOb25lDQogICAgcGxhbmUsIGFsaWduLCBjb2RlID0gb3JpZW50X2luZm8oaW9wKSBpZiBpb3AgaXMgbm90IE5vbmUgZWxzZSAoTm9uZSwgZmxvYXQoJ25hbicpLCBOb25lKQ0KDQogICAgaGF2ZV9wb3MgPSBbciBmb3IgciBpbiByZWNzIGlmIHJbJ2lwcCddIGlzIG5vdCBOb25lXQ0KICAgIG4gPSBsZW4ocmVjcykNCiAgICBuc2lnbiA9IDEuMA0KICAgICMgZXZlcnkgYnJhbmNoIGJ1aWxkcyB0dXBsZXMgKGtleSwgdGllYnJlYWssIG5hbWUsIHJlYyk7IGBwb3NgIGlzIGluIG1tIG9ubHkgZm9yIG1ldGhvZCA9PSAncG9zaXRpb24nDQogICAgaWYgaW9wIGlzIG5vdCBOb25lIGFuZCBsZW4oaGF2ZV9wb3MpID49IG1heCgyLCBpbnQoMC44ICogbikpOg0KICAgICAgICBucm0gPSBzbGljZV9ub3JtYWwoaW9wKQ0KICAgICAgICBheCA9IGludChucC5hcmdtYXgobnAuYWJzKG5ybSkpKQ0KICAgICAgICBuc2lnbiA9IDEuMCBpZiBucm1bYXhdID49IDAgZWxzZSAtMS4wDQogICAgICAgIHNnbiA9IG5zaWduIGlmIG9yZGVyX21vZGUgPT0gJ3Bvc2l0aW9uJyBlbHNlIDEuMA0KICAgICAgICBzcGFyZSA9IGZsb2F0KG5wLm1lZGlhbihbc2duICogZmxvYXQobnAuZG90KHJbJ2lwcCddLCBucm0pKSBmb3IgciBpbiBoYXZlX3Bvc10pKQ0KICAgICAgICBrZXllZCA9IFsoc2duICogZmxvYXQobnAuZG90KHJbJ2lwcCddLCBucm0pKSBpZiByWydpcHAnXSBpcyBub3QgTm9uZSBlbHNlIHNwYXJlLA0KICAgICAgICAgICAgICAgICAgclsnaW5zdCddIGlmIHJbJ2luc3QnXSBpcyBub3QgTm9uZSBlbHNlIGZsb2F0KCdpbmYnKSwgclsnbmFtZSddLCByKSBmb3IgciBpbiByZWNzXQ0KICAgICAgICBtZXRob2QgPSAncG9zaXRpb24nDQogICAgZWxpZiBzdW0oclsnaW5zdCddIGlzIG5vdCBOb25lIGZvciByIGluIHJlY3MpID49IG1heCgyLCBpbnQoMC44ICogbikpOg0KICAgICAgICBrZXllZCA9IFsoclsnaW5zdCddIGlmIHJbJ2luc3QnXSBpcyBub3QgTm9uZSBlbHNlIGZsb2F0KCdpbmYnKSwgMC4wLCByWyduYW1lJ10sIHIpIGZvciByIGluIHJlY3NdDQogICAgICAgIG1ldGhvZCA9ICdpbnN0YW5jZScNCiAgICBlbHNlOg0KICAgICAgICBrZXllZCA9IFsoMC4wLCAwLjAsIHJbJ25hbWUnXSwgcikgZm9yIHIgaW4gcmVjc10NCiAgICAgICAgbWV0aG9kID0gJ25hbWUnDQogICAga2V5ZWQuc29ydChrZXk9bGFtYmRhIHQ6ICh0WzBdLCB0WzFdLCB0WzJdKSkNCiAgICBvcmRlcmVkID0gW2tbMl0gZm9yIGsgaW4ga2V5ZWRdDQogICAgcG9zID0gbnAuYXJyYXkoW2tbMF0gZm9yIGsgaW4ga2V5ZWRdLCBkdHlwZT1ucC5mbG9hdDY0KSBpZiBtZXRob2QgIT0gJ25hbWUnIGVsc2UgbnAuYXJhbmdlKG4sIGR0eXBlPW5wLmZsb2F0NjQpDQogICAgZHAgPSBucC5kaWZmKHBvcykgaWYgbGVuKHBvcykgPiAxIGVsc2UgbnAuYXJyYXkoW10pDQogICAgc3BhY2luZyA9IGZsb2F0KG5wLm1lZGlhbihucC5hYnMoZHApKSkgaWYgbGVuKGRwKSBlbHNlIGZsb2F0KCduYW4nKQ0KICAgIHJlcCA9IGxlbihvcmRlcmVkKSAvLyAyDQogICAgcmVwX3JlYyA9IGtleWVkW3JlcF1bM10NCiAgICBwcyA9IHJlcF9yZWNbJ3BzJ10gaWYgcmVwX3JlY1sncHMnXSBlbHNlIChmbG9hdCgnbmFuJyksIGZsb2F0KCduYW4nKSkNCiAgICBjeCA9IGZsb2F0KCduYW4nKQ0KICAgIGlmIHJlcF9yZWNbJ2lwcCddIGlzIG5vdCBOb25lIGFuZCBpb3AgaXMgbm90IE5vbmUgYW5kIHJlcF9yZWNbJ3BzJ106DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGN4ID0gZmxvYXQoY2VudGVyX3h5eihyZXBfcmVjWydpcHAnXSwgaW9wLCByZXBfcmVjWydwcyddLCBzaGFwZVswXSwgc2hhcGVbMV0pWzBdKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICAgICAgY3ggPSBmbG9hdCgnbmFuJykNCiAgICByZXR1cm4gZGljdChvcmRlcmVkPW9yZGVyZWQsIHBvcz1wb3MsIG1ldGhvZD1tZXRob2QsIG5fZHJvcHBlZF9zaGFwZT1uX2Ryb3AsIHBsYW5lPXBsYW5lLCBjb2RlPWNvZGUsDQogICAgICAgICAgICAgICAgYWxpZ249YWxpZ24sIHNwYWNpbmdfbWVkPXNwYWNpbmcsIHpfZXh0ZW50PWZsb2F0KHBvc1stMV0gLSBwb3NbMF0pIGlmIGxlbihwb3MpID4gMSBlbHNlIDAuMCwNCiAgICAgICAgICAgICAgICBzaGFwZT1zaGFwZSwgcHM9cHMsIGNlbnRlcl94PWN4LCByZXBfaW5kZXg9cmVwLCBuc2lnbj1uc2lnbikNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBsYXRlcmFsaXR5DQpkZWYgX3NpZGVfZnJvbV90YWcodik6DQogICAgaWYgdiBpcyBOb25lOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHMgPSBzdHIodikuc3RyaXAoKS51cHBlcigpDQogICAgcmV0dXJuIHNbMF0gaWYgcyBhbmQgc1swXSBpbiAoJ0wnLCAnUicpIGVsc2UgTm9uZQ0KDQoNCmRlZiByZXNvbHZlX2xhdGVyYWxpdHkocm93cywgbWluX29mZnNldD1Ob25lKToNCiAgICAiIiJTdHVkeS1sZXZlbCBzaWRlIG9mIHRoZSBrbmVlLg0KDQogICAgcm93czogaXRlcmFibGUgb2YgZGljdHMgd2l0aCBvcHRpb25hbCAnTGF0ZXJhbGl0eScsICdJbWFnZUxhdGVyYWxpdHknIChzdHJpbmdzKSBhbmQgJ2NlbnRlcl94JyAobW0pLg0KICAgIFByaW9yaXR5OiBESUNPTSB0YWcgKG1ham9yaXR5IHZvdGUpIC0+IHBhdGllbnQteCBvZiB0aGUgaW1hZ2UgY2VudHJlIChtZWRpYW4gb3ZlciBzZXJpZXMsIHx4fCA+PSBtaW5fb2Zmc2V0Og0KICAgIHg8MCAtPiBSaWdodCwgZWxzZSBMZWZ0KSAtPiB1bnJlc29sdmVkICgnVScpLg0KICAgIFJldHVybnMgZGljdChzaWRlIGluIHsnTCcsJ1InLCdVJ30sIHNvdXJjZSwgeF9tZWQsIGRpc2FncmVlKS4NCiAgICAiIiINCiAgICBtaW5fb2Zmc2V0ID0gY29uZmlnLkxBVF9NSU5fT0ZGU0VUX01NIGlmIG1pbl9vZmZzZXQgaXMgTm9uZSBlbHNlIG1pbl9vZmZzZXQNCiAgICB0YWdzID0gW10NCiAgICB4cyA9IFtdDQogICAgZm9yIHIgaW4gcm93czoNCiAgICAgICAgZm9yIGsgaW4gKCdMYXRlcmFsaXR5JywgJ0ltYWdlTGF0ZXJhbGl0eScpOg0KICAgICAgICAgICAgcyA9IF9zaWRlX2Zyb21fdGFnKHIuZ2V0KGspKQ0KICAgICAgICAgICAgaWYgczoNCiAgICAgICAgICAgICAgICB0YWdzLmFwcGVuZChzKQ0KICAgICAgICB4ID0gci5nZXQoJ2NlbnRlcl94JykNCiAgICAgICAgaWYgeCBpcyBub3QgTm9uZSBhbmQgbnAuaXNmaW5pdGUoeCk6DQogICAgICAgICAgICB4cy5hcHBlbmQoZmxvYXQoeCkpDQogICAgeG0gPSBmbG9hdChucC5tZWRpYW4oeHMpKSBpZiB4cyBlbHNlIGZsb2F0KCduYW4nKQ0KICAgIGdlbyA9IE5vbmUNCiAgICBpZiBucC5pc2Zpbml0ZSh4bSkgYW5kIGFicyh4bSkgPj0gbWluX29mZnNldDoNCiAgICAgICAgZ2VvID0gJ1InIGlmIHhtIDwgMCBlbHNlICdMJw0KICAgIGlmIHRhZ3M6DQogICAgICAgIHNpZGUgPSBDb3VudGVyKHRhZ3MpLm1vc3RfY29tbW9uKDEpWzBdWzBdDQogICAgICAgIHJldHVybiBkaWN0KHNpZGU9c2lkZSwgc291cmNlPSd0YWcnLCB4X21lZD14bSwgZGlzYWdyZWU9Ym9vbChnZW8gYW5kIGdlbyAhPSBzaWRlKSkNCiAgICBpZiBnZW86DQogICAgICAgIHJldHVybiBkaWN0KHNpZGU9Z2VvLCBzb3VyY2U9J2dlb21ldHJ5JywgeF9tZWQ9eG0sIGRpc2FncmVlPUZhbHNlKQ0KICAgIHJldHVybiBkaWN0KHNpZGU9J1UnLCBzb3VyY2U9J25vbmUnLCB4X21lZD14bSwgZGlzYWdyZWU9RmFsc2UpDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY2Fub25pY2FsaXNhdGlvbg0KZGVmIHJlb3JpZW50X3N0YWNrKHN0YWNrLCBjb2RlLCBwbGFuZSk6DQogICAgIiIiTWFwIGEgW0QsSCxXXSBzdGFjayB0byB0aGUgZGF0YXNldC1zdGFuZGFyZCBvcmllbnRhdGlvbiBjb2RlIG9mIGBwbGFuZWAgKENBTk9OX09SSUVOVCkuDQoNCiAgICBIYW5kbGVzIGF4aXMgc3dhcHMgKHRyYW5zcG9zZSkgYW5kIHNpZ24gZmxpcHMuIEEgbm8tb3AgZm9yIGFsbCAyNCwzNzEgdHJhaW4gc2VyaWVzIChzaW5nbGUgY29kZSBwZXIgcGxhbmUpLiIiIg0KICAgIHRhcmdldCA9IGNvbmZpZy5DQU5PTl9PUklFTlQuZ2V0KHBsYW5lKQ0KICAgIGlmIG5vdCBjb2RlIG9yIGxlbihjb2RlKSAhPSAyIG9yIG5vdCB0YXJnZXQgb3IgY29kZSA9PSB0YXJnZXQ6DQogICAgICAgIHJldHVybiBzdGFjaywgRmFsc2UNCiAgICByLCBjID0gY29kZQ0KICAgIHRyLCB0YyA9IHRhcmdldA0KICAgIGlmIF9BWElTW3JdICE9IF9BWElTW3RyXToNCiAgICAgICAgc3RhY2sgPSBzdGFjay50cmFuc3Bvc2UoMCwgMiwgMSkNCiAgICAgICAgciwgYyA9IGMsIHINCiAgICBpZiByICE9IHRyOg0KICAgICAgICBzdGFjayA9IHN0YWNrWzosIDosIDo6LTFdDQogICAgaWYgYyAhPSB0YzoNCiAgICAgICAgc3RhY2sgPSBzdGFja1s6LCA6Oi0xLCA6XQ0KICAgIHJldHVybiBucC5hc2NvbnRpZ3VvdXNhcnJheShzdGFjayksIFRydWUNCg0KDQpkZWYgY2Fub25pY2FsX3NpZGUoc3RhY2ssIHBsYW5lLCBzaWRlKToNCiAgICAiIiJNaXJyb3IgcmlnaHQga25lZXMgc28gdGhleSBsb29rIGxpa2UgbGVmdCBrbmVlcy4NCg0KICAgIENvcm9uYWwvQXhpYWw6IGZsaXAgaW1hZ2UgY29sdW1ucy4gU2FnaXR0YWw6IHJldmVyc2Ugc2xpY2Ugb3JkZXIgKHNsaWNlcyBydW4gbWVkaWFsPC0+bGF0ZXJhbCkuIiIiDQogICAgaWYgc2lkZSAhPSAnUic6DQogICAgICAgIHJldHVybiBzdGFjaw0KICAgIGlmIHBsYW5lIGluICgnQ29yb25hbCcsICdBeGlhbCcpOg0KICAgICAgICByZXR1cm4gbnAuYXNjb250aWd1b3VzYXJyYXkoc3RhY2tbOiwgOiwgOjotMV0pDQogICAgcmV0dXJuIG5wLmFzY29udGlndW91c2FycmF5KHN0YWNrWzo6LTFdKQ0K',
    'src/data/preprocess/index.py': 'IiIiU2VyaWVzIGluZGV4OiBvbmUgcm93IHBlciBzZXJpZXMgb24gZGlzaywgd2l0aCBhbmF0b21pY2FsIHNsaWNlIG9yZGVyLCBnZW9tZXRyeSBhbmQgYSByZXByZXNlbnRhdGl2ZSBoZWFkZXIuDQoNCkRlc2lnbiBydWxlcyAoZnJvbSB0aGUgZGF0YXNldCBhdWRpdCArIHRoZSBoaWRkZW4tcmVydW4gZmFpbHVyZXMgZG9jdW1lbnRlZCBpbiB0aGUgcHVibGljIG5vdGVib29rcyk6DQogICogdGhlIERJUkVDVE9SWSBUUkVFIGlzIHRoZSB0cnV0aCBmb3Igd2hpY2ggc2VyaWVzL2ZpbGVzIGV4aXN0OyB0aGUgQ1NWcyBvbmx5IGNvbnRyaWJ1dGUgZmxhZ3MNCiAgICAodGVzdF9zZXJpZXMuY3N2IGluIHRoZSBoaWRkZW4gc2V0IG1heSBkaWZmZXIgZnJvbSB0aGUgcGxhY2Vob2xkZXIgLT4gbmV2ZXIgYXNzdW1lIGl0IGlzIGNvbXBsZXRlKQ0KICAqIGEgZmFpbHVyZSBpbnNpZGUgb25lIHNlcmllcyBuZXZlciBhYm9ydHMgdGhlIHJ1bjogdGhlIHJvdyBpcyBrZXB0IHdpdGggYGVycmAgc2V0IGFuZCBhbiBlbXB0eSBmaWxlIGxpc3QNCiAgKiB0aGUgYnVpbGQgaXMgY2h1bmtlZCArIGNoZWNrcG9pbnRlZCwgc28gYSA0MC1taW51dGUgc2NhbiBzdXJ2aXZlcyBhIHNlc3Npb24gcmVzdGFydA0KIiIiDQppbXBvcnQgb3MNCmltcG9ydCB0aW1lDQppbXBvcnQgcGlja2xlDQppbXBvcnQgbXVsdGlwcm9jZXNzaW5nIGFzIG1wDQpmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgUHJvY2Vzc1Bvb2xFeGVjdXRvciwgVGhyZWFkUG9vbEV4ZWN1dG9yDQoNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQppbXBvcnQgc3JjLmNvcmUuY29uZmlnIGFzIGNvbmZpZw0KZnJvbSAuIGltcG9ydCBkaWNvbWlvIGFzIGRpbw0KZnJvbSAuIGltcG9ydCBnZW9tZXRyeSBhcyBnZW8NCg0KU1BMSVRfRElSUyA9IHsndHJhaW4nOiAndHJhaW5fc2VyaWVzJywgJ3Rlc3QnOiAndGVzdF9zZXJpZXMnfQ0KDQoNCmRlZiBkaXNjb3Zlcl9yb290KGV4cGxpY2l0PU5vbmUpOg0KICAgICIiIkZpbmQgdGhlIGNvbXBldGl0aW9uIGZvbGRlciAoaGFuZGxlcyAva2FnZ2xlL2lucHV0L2NvbXBldGl0aW9ucy88c2x1Zz4gYW5kIC9rYWdnbGUvaW5wdXQvPHNsdWc+KS4iIiINCiAgICBjYW5kcyA9IFtleHBsaWNpdCwgb3MuZW52aXJvbi5nZXQoJ0tORUVfREFUQScpXSArIGxpc3QoY29uZmlnLlJPT1RfQ0FORElEQVRFUykNCiAgICBmb3IgYyBpbiBjYW5kczoNCiAgICAgICAgaWYgYyBhbmQgb3MucGF0aC5pc2RpcihjKSBhbmQgKG9zLnBhdGguaXNkaXIob3MucGF0aC5qb2luKGMsICd0cmFpbl9zZXJpZXMnKSkgb3INCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG9zLnBhdGguaXNkaXIob3MucGF0aC5qb2luKGMsICd0ZXN0X3NlcmllcycpKSk6DQogICAgICAgICAgICByZXR1cm4gYw0KICAgIGJhc2UgPSAnL2thZ2dsZS9pbnB1dCcNCiAgICBpZiBvcy5wYXRoLmlzZGlyKGJhc2UpOg0KICAgICAgICBmb3IgcCBpbiBzb3J0ZWQob3MubGlzdGRpcihiYXNlKSk6DQogICAgICAgICAgICBmb3IgcSBpbiAob3MucGF0aC5qb2luKGJhc2UsIHApLCAqW29zLnBhdGguam9pbihiYXNlLCBwLCB4KSBmb3IgeCBpbiBzb3J0ZWQob3MubGlzdGRpcihvcy5wYXRoLmpvaW4oYmFzZSwgcCkpKVs6MjBdDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaWYgb3MucGF0aC5pc2Rpcihvcy5wYXRoLmpvaW4oYmFzZSwgcCwgeCkpXSk6DQogICAgICAgICAgICAgICAgaWYgb3MucGF0aC5pc2Rpcihvcy5wYXRoLmpvaW4ocSwgJ3Rlc3Rfc2VyaWVzJykpIG9yIG9zLnBhdGguaXNkaXIob3MucGF0aC5qb2luKHEsICd0cmFpbl9zZXJpZXMnKSk6DQogICAgICAgICAgICAgICAgICAgIHJldHVybiBxDQogICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoJ2NvbXBldGl0aW9uIHJvb3Qgbm90IGZvdW5kOyBwYXNzIGl0IGV4cGxpY2l0bHkgb3Igc2V0IEtORUVfREFUQScpDQoNCg0KZGVmIGxpc3Rfc2VyaWVzX2RpcnMocm9vdCwgc3BsaXQpOg0KICAgIGJhc2UgPSBvcy5wYXRoLmpvaW4ocm9vdCwgU1BMSVRfRElSUy5nZXQoc3BsaXQsIHNwbGl0KSkNCiAgICBpdGVtcyA9IFtdDQogICAgaWYgbm90IG9zLnBhdGguaXNkaXIoYmFzZSk6DQogICAgICAgIGZvciBhbHQgaW4gKHNwbGl0LCBmIntzcGxpdH1fc2VyaWVzIiwgZiJ7c3BsaXR9X2ltYWdlcyIpOg0KICAgICAgICAgICAgYWx0X3BhdGggPSBvcy5wYXRoLmpvaW4ocm9vdCwgYWx0KQ0KICAgICAgICAgICAgaWYgb3MucGF0aC5pc2RpcihhbHRfcGF0aCk6DQogICAgICAgICAgICAgICAgYmFzZSA9IGFsdF9wYXRoDQogICAgICAgICAgICAgICAgYnJlYWsNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIHJldHVybiBpdGVtcw0KICAgIHdpdGggb3Muc2NhbmRpcihiYXNlKSBhcyBpdDoNCiAgICAgICAgc3R1ZGllcyA9IHNvcnRlZChlLm5hbWUgZm9yIGUgaW4gaXQgaWYgZS5pc19kaXIoKSkNCiAgICBmb3Igc3QgaW4gc3R1ZGllczoNCiAgICAgICAgc2QgPSBvcy5wYXRoLmpvaW4oYmFzZSwgc3QpDQogICAgICAgIHdpdGggb3Muc2NhbmRpcihzZCkgYXMgaXQyOg0KICAgICAgICAgICAgZm9yIGUgaW4gc29ydGVkKGl0Miwga2V5PWxhbWJkYSB4OiB4Lm5hbWUpOg0KICAgICAgICAgICAgICAgIGlmIGUuaXNfZGlyKCk6DQogICAgICAgICAgICAgICAgICAgIGl0ZW1zLmFwcGVuZCgoc3BsaXQsIHN0LCBlLm5hbWUsIGUucGF0aCkpDQogICAgcmV0dXJuIGl0ZW1zDQoNCg0KZGVmIHNjYW5fc2VyaWVzKGl0ZW0pOg0KICAgICIiIkZ1bGwgc2xpY2Ugb3JkZXJpbmcgKyByZXByZXNlbnRhdGl2ZSBoZWFkZXIgZm9yIG9uZSBzZXJpZXMgZGlyZWN0b3J5LiIiIg0KICAgIHNwbGl0LCBzdHVkeSwgc2VyaWVzLCBkID0gaXRlbQ0KICAgIHJvdyA9IGRpY3Qoc3BsaXQ9c3BsaXQsIFN0dWR5SW5zdGFuY2VVSUQ9c3R1ZHksIFNlcmllc0luc3RhbmNlVUlEPXNlcmllcywgZGlyPWQsIG5fZmlsZXM9MCwNCiAgICAgICAgICAgICAgIG9yZGVyZWRfZmlsZXM9W10sIHBvc2l0aW9ucz1ucC56ZXJvcygwKSwgZXJyPU5vbmUpDQogICAgdHJ5Og0KICAgICAgICBuYW1lcyA9IGRpby5saXN0X2RpY29tcyhkKQ0KICAgICAgICByb3dbJ25fZmlsZXMnXSA9IGxlbihuYW1lcykNCiAgICAgICAgcmVjcywgYmFkID0gW10sIDANCiAgICAgICAgZm9yIG5tIGluIG5hbWVzOg0KICAgICAgICAgICAgZHMgPSBkaW8ucmVhZF90YWdzKG9zLnBhdGguam9pbihkLCBubSksIGRpby5PUkRFUl9UQUdTKQ0KICAgICAgICAgICAgaWYgZHMgaXMgTm9uZSBvciAnUm93cycgbm90IGluIGRzIG9yICdDb2x1bW5zJyBub3QgaW4gZHM6DQogICAgICAgICAgICAgICAgYmFkICs9IDENCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgcHMgPSBkaW8uZmxvYXRzKGdldGF0dHIoZHMsICdQaXhlbFNwYWNpbmcnLCBOb25lKSwgMikNCiAgICAgICAgICAgIHJlY3MuYXBwZW5kKGRpY3QobmFtZT1ubSwgaXBwPWRpby5mbG9hdHMoZ2V0YXR0cihkcywgJ0ltYWdlUG9zaXRpb25QYXRpZW50JywgTm9uZSksIDMpLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpb3A9ZGlvLmZsb2F0cyhnZXRhdHRyKGRzLCAnSW1hZ2VPcmllbnRhdGlvblBhdGllbnQnLCBOb25lKSwgNiksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGluc3Q9ZGlvLmYxKGdldGF0dHIoZHMsICdJbnN0YW5jZU51bWJlcicsIE5vbmUpKSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcm93cz1pbnQoZHMuUm93cyksIGNvbHM9aW50KGRzLkNvbHVtbnMpLCBwcz10dXBsZShwcykgaWYgcHMgZWxzZSBOb25lKSkNCiAgICAgICAgcm93WyduX2JhZCddID0gYmFkDQogICAgICAgIG8gPSBnZW8ub3JkZXJfcmVjb3JkcyhyZWNzLCAncG9zaXRpb24nKQ0KICAgICAgICBpZiBvIGlzIE5vbmU6DQogICAgICAgICAgICByb3dbJ2VyciddID0gJ25vX3JlYWRhYmxlX3NsaWNlcycNCiAgICAgICAgICAgIHJldHVybiByb3cNCiAgICAgICAgcm93LnVwZGF0ZShvcmRlcmVkX2ZpbGVzPW9bJ29yZGVyZWQnXSwgcG9zaXRpb25zPW9bJ3BvcyddLmFzdHlwZShucC5mbG9hdDMyKSwgb3JkZXJfbWV0aG9kPW9bJ21ldGhvZCddLA0KICAgICAgICAgICAgICAgICAgIG5fZHJvcHBlZF9zaGFwZT1vWyduX2Ryb3BwZWRfc2hhcGUnXSwgcGxhbmVfZ2VvPW9bJ3BsYW5lJ10sIG9yaWVudF9jb2RlPW9bJ2NvZGUnXSwNCiAgICAgICAgICAgICAgICAgICBwbGFuZV9hbGlnbj1vWydhbGlnbiddLCBzcGFjaW5nX21lZD1vWydzcGFjaW5nX21lZCddLCB6X2V4dGVudD1vWyd6X2V4dGVudCddLA0KICAgICAgICAgICAgICAgICAgIHJvd3M9b1snc2hhcGUnXVswXSwgY29scz1vWydzaGFwZSddWzFdLCBwc19yb3c9b1sncHMnXVswXSwgcHNfY29sPW9bJ3BzJ11bMV0sDQogICAgICAgICAgICAgICAgICAgY2VudGVyX3g9b1snY2VudGVyX3gnXSwgbnNpZ249b1snbnNpZ24nXSwgbl9zbGljZXM9bGVuKG9bJ29yZGVyZWQnXSkpDQogICAgICAgIHJlcCA9IGRpby5yZXByZXNlbnRhdGl2ZV9oZWFkZXIob3MucGF0aC5qb2luKGQsIG9bJ29yZGVyZWQnXVtvWydyZXBfaW5kZXgnXV0pKQ0KICAgICAgICBpZiByZXA6DQogICAgICAgICAgICByb3cudXBkYXRlKHtrOiB2IGZvciBrLCB2IGluIHJlcC5pdGVtcygpIGlmIGsgbm90IGluICgncHNfcm93JywgJ3BzX2NvbCcpfSkNCiAgICAgICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShyb3dbJ3BzX3JvdyddKSBhbmQgbnAuaXNmaW5pdGUocmVwWydwc19yb3cnXSk6DQogICAgICAgICAgICAgICAgcm93Wydwc19yb3cnXSwgcm93Wydwc19jb2wnXSA9IHJlcFsncHNfcm93J10sIHJlcFsncHNfY29sJ10NCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgbmV2ZXIgZmF0YWwNCiAgICAgICAgcm93WydlcnInXSA9IGYne3R5cGUoZSkuX19uYW1lX199OiB7c3RyKGUpWzo4MF19Jw0KICAgIHJldHVybiByb3cNCg0KDQpkZWYgYnVpbGRfaW5kZXgocm9vdCwgc3BsaXRzPSgndHJhaW4nLCAndGVzdCcpLCB3b3JrZXJzPU5vbmUsIGNodW5rPTEwMDAsIGNhY2hlX2Rpcj1Ob25lLCBsaW1pdD0wLCBmb3JjZT1GYWxzZSwNCiAgICAgICAgICAgICAgICBwcm9ncmVzcz1UcnVlKToNCiAgICAiIiJTY2FuIGV2ZXJ5IHNlcmllcyBvbiBkaXNrLiBSZXR1cm5zIGEgRGF0YUZyYW1lIChvYmplY3QgY29sdW1uczogb3JkZXJlZF9maWxlcywgcG9zaXRpb25zKS4iIiINCiAgICB3b3JrZXJzID0gd29ya2VycyBvciBtYXgoMiwgKG9zLmNwdV9jb3VudCgpIG9yIDQpICogMikNCiAgICBpdGVtcyA9IFtdDQogICAgZm9yIHNwIGluIHNwbGl0czoNCiAgICAgICAgaXRlbXMgKz0gbGlzdF9zZXJpZXNfZGlycyhyb290LCBzcCkNCiAgICBpZiBsaW1pdDoNCiAgICAgICAgaXRlbXMgPSBpdGVtc1s6bGltaXRdDQogICAgY2h1bmtzID0gW2l0ZW1zW2k6aSArIGNodW5rXSBmb3IgaSBpbiByYW5nZSgwLCBsZW4oaXRlbXMpLCBjaHVuayldDQogICAgaWYgY2FjaGVfZGlyOiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgc3RhbGUtY2hlY2twb2ludCBndWFyZDogY2h1bmtzIGFyZSBvbmx5IHZhbGlkIGZvciBUSElTIGl0ZW0gbGlzdA0KICAgICAgICBpbXBvcnQgZ2xvYg0KICAgICAgICBpbXBvcnQgaGFzaGxpYg0KICAgICAgICBzaWcgPSBoYXNobGliLm1kNSgoJ3wnLmpvaW4oaVszXSBmb3IgaSBpbiBpdGVtcykgKyBmJyN7Y2h1bmt9JykuZW5jb2RlKCkpLmhleGRpZ2VzdCgpDQogICAgICAgIHNwID0gb3MucGF0aC5qb2luKGNhY2hlX2RpciwgJ3NpZ25hdHVyZS50eHQnKQ0KICAgICAgICBvcy5tYWtlZGlycyhjYWNoZV9kaXIsIGV4aXN0X29rPVRydWUpDQogICAgICAgIGlmIG5vdCAob3MucGF0aC5leGlzdHMoc3ApIGFuZCBvcGVuKHNwKS5yZWFkKCkgPT0gc2lnKToNCiAgICAgICAgICAgIGZvciBmIGluIGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4oY2FjaGVfZGlyLCAnaW5kZXhfKi5wa2wnKSk6DQogICAgICAgICAgICAgICAgb3MucmVtb3ZlKGYpDQogICAgICAgICAgICBvcGVuKHNwLCAndycpLndyaXRlKHNpZykNCiAgICBwYXJ0cywgdDAgPSBbXSwgdGltZS50aW1lKCkNCiAgICBleCA9IE5vbmUNCiAgICB0cnk6DQogICAgICAgIGV4ID0gUHJvY2Vzc1Bvb2xFeGVjdXRvcihtYXhfd29ya2Vycz13b3JrZXJzLCBtcF9jb250ZXh0PW1wLmdldF9jb250ZXh0KCdmb3JrJykpDQogICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgZXggPSBUaHJlYWRQb29sRXhlY3V0b3IobWF4X3dvcmtlcnM9d29ya2VycykNCiAgICBmb3IgaywgY2ggaW4gZW51bWVyYXRlKGNodW5rcyk6DQogICAgICAgIHAgPSBvcy5wYXRoLmpvaW4oY2FjaGVfZGlyLCBmJ2luZGV4X3trOjA0ZH0ucGtsJykgaWYgY2FjaGVfZGlyIGVsc2UgTm9uZQ0KICAgICAgICBpZiBwIGFuZCBvcy5wYXRoLmV4aXN0cyhwKSBhbmQgbm90IGZvcmNlOg0KICAgICAgICAgICAgcGFydHMuYXBwZW5kKHBkLnJlYWRfcGlja2xlKHApKQ0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgZGYgPSBwZC5EYXRhRnJhbWUobGlzdChleC5tYXAoc2Nhbl9zZXJpZXMsIGNoLCBjaHVua3NpemU9OCkpKQ0KICAgICAgICBpZiBwOg0KICAgICAgICAgICAgb3MubWFrZWRpcnMoY2FjaGVfZGlyLCBleGlzdF9vaz1UcnVlKQ0KICAgICAgICAgICAgZGYudG9fcGlja2xlKHApDQogICAgICAgIHBhcnRzLmFwcGVuZChkZikNCiAgICAgICAgaWYgcHJvZ3Jlc3M6DQogICAgICAgICAgICBwcmludChmJyAgaW5kZXggY2h1bmsge2sgKyAxfS97bGVuKGNodW5rcyl9ICB7KGsgKyAxKSAqIGNodW5rIC8gbWF4KHRpbWUudGltZSgpIC0gdDAsIDFlLTkpOi4xZn0gc2VyaWVzL3MnLCBmbHVzaD1UcnVlKQ0KICAgIGV4LnNodXRkb3duKCkNCiAgICBvdXQgPSBwZC5jb25jYXQocGFydHMsIGlnbm9yZV9pbmRleD1UcnVlKSBpZiBwYXJ0cyBlbHNlIHBkLkRhdGFGcmFtZSgpDQogICAgcmV0dXJuIG91dA0KDQoNCmRlZiBhdHRhY2hfY3N2X2ZsYWdzKGluZGV4X2RmLCByb290KToNCiAgICAiIiJMZWZ0LWpvaW4gdGhlIGNvbXBldGl0aW9uIENTViBmbGFncyAocGxhbmUgLyBmbHVpZCAvIGZhdC1zYXQpIG9udG8gdGhlIG9uLWRpc2sgaW5kZXguIiIiDQogICAgaWYgaW5kZXhfZGYuZW1wdHkgb3IgJ1Nlcmllc0luc3RhbmNlVUlEJyBub3QgaW4gaW5kZXhfZGYuY29sdW1uczoNCiAgICAgICAgZm9yIGMgaW4gKCdTZXJpZXNJbnN0YW5jZVVJRCcsICdBbmF0b21pY2FsX1BsYW5lJywgJ0ZsdWlkX1NlbnNpdGl2ZScsICdGYXRfU3VwcHJlc3Npb24nKToNCiAgICAgICAgICAgIGlmIGMgbm90IGluIGluZGV4X2RmLmNvbHVtbnM6DQogICAgICAgICAgICAgICAgaW5kZXhfZGZbY10gPSBbXSBpZiBpbmRleF9kZi5lbXB0eSBlbHNlIChucC5uYW4gaWYgYyAhPSAnQW5hdG9taWNhbF9QbGFuZScgZWxzZSBOb25lKQ0KICAgICAgICByZXR1cm4gaW5kZXhfZGYNCiAgICBmcmFtZXMgPSBbXQ0KICAgIGZvciBuYW1lIGluICgndHJhaW5fc2VyaWVzLmNzdicsICd0ZXN0X3Nlcmllcy5jc3YnKToNCiAgICAgICAgcCA9IG9zLnBhdGguam9pbihyb290LCBuYW1lKQ0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhwKToNCiAgICAgICAgICAgIGZyYW1lcy5hcHBlbmQocGQucmVhZF9jc3YocCkpDQogICAgaWYgbm90IGZyYW1lczoNCiAgICAgICAgZm9yIGMgaW4gKCdBbmF0b21pY2FsX1BsYW5lJywgJ0ZsdWlkX1NlbnNpdGl2ZScsICdGYXRfU3VwcHJlc3Npb24nKToNCiAgICAgICAgICAgIGluZGV4X2RmW2NdID0gbnAubmFuIGlmIGMgIT0gJ0FuYXRvbWljYWxfUGxhbmUnIGVsc2UgTm9uZQ0KICAgICAgICByZXR1cm4gaW5kZXhfZGYNCiAgICBjc3YgPSBwZC5jb25jYXQoZnJhbWVzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkuZHJvcF9kdXBsaWNhdGVzKCdTZXJpZXNJbnN0YW5jZVVJRCcpDQogICAga2VlcCA9IFtjIGZvciBjIGluICgnU2VyaWVzSW5zdGFuY2VVSUQnLCAnQW5hdG9taWNhbF9QbGFuZScsICdGbHVpZF9TZW5zaXRpdmUnLCAnRmF0X1N1cHByZXNzaW9uJykgaWYgYyBpbiBjc3YuY29sdW1uc10NCiAgICBvdXQgPSBpbmRleF9kZi5tZXJnZShjc3Zba2VlcF0sIG9uPSdTZXJpZXNJbnN0YW5jZVVJRCcsIGhvdz0nbGVmdCcpDQogICAgZm9yIGMgaW4gKCdBbmF0b21pY2FsX1BsYW5lJywgJ0ZsdWlkX1NlbnNpdGl2ZScsICdGYXRfU3VwcHJlc3Npb24nKToNCiAgICAgICAgaWYgYyBub3QgaW4gb3V0LmNvbHVtbnM6DQogICAgICAgICAgICBvdXRbY10gPSBucC5uYW4gaWYgYyAhPSAnQW5hdG9taWNhbF9QbGFuZScgZWxzZSBOb25lDQogICAgcmV0dXJuIG91dA0KDQoNCmRlZiBzYXZlX2luZGV4KGRmLCBwYXRoKToNCiAgICB3aXRoIG9wZW4ocGF0aCwgJ3diJykgYXMgZmg6DQogICAgICAgIHBpY2tsZS5kdW1wKGRmLCBmaCwgcHJvdG9jb2w9NCkNCg0KDQpkZWYgbG9hZF9pbmRleChwYXRoKToNCiAgICB3aXRoIG9wZW4ocGF0aCwgJ3JiJykgYXMgZmg6DQogICAgICAgIHJldHVybiBwaWNrbGUubG9hZChmaCkNCg0KDQpkZWYgaW5kZXhfcmVwb3J0KGRmKToNCiAgICAiIiJIdW1hbi1yZWFkYWJsZSBkYXRhLWNvbnRyYWN0IHN0YXRpc3RpY3MgKHByaW50ZWQgYnkgdG9vbHMvYnVpbGRfaW5kZXgucHkpLiIiIg0KICAgIHJlcCA9IHt9DQogICAgaWYgZGYuZW1wdHk6DQogICAgICAgIHJlcFsnc2VyaWVzJ10gPSAwDQogICAgICAgIHJlcFsnc2VyaWVzX3dpdGhfZXJyb3InXSA9IDANCiAgICAgICAgcmVwWydlbXB0eV9zZXJpZXMnXSA9IDANCiAgICAgICAgcmV0dXJuIHJlcA0KICAgIG9rID0gZGZbZGYuZXJyLmlzbmEoKV0gaWYgJ2VycicgaW4gZGYgZWxzZSBkZg0KICAgIHJlcFsnc2VyaWVzJ10gPSBsZW4oZGYpDQogICAgcmVwWydzZXJpZXNfd2l0aF9lcnJvciddID0gaW50KGRmLmVyci5ub3RuYSgpLnN1bSgpKSBpZiAnZXJyJyBpbiBkZiBlbHNlIDANCiAgICByZXBbJ2VtcHR5X3NlcmllcyddID0gaW50KChkZi5uX2ZpbGVzID09IDApLnN1bSgpKSBpZiAnbl9maWxlcycgaW4gZGYgZWxzZSAwDQogICAgaWYgJ29yZGVyX21ldGhvZCcgaW4gb2s6DQogICAgICAgIHJlcFsnb3JkZXJfbWV0aG9kJ10gPSBvay5vcmRlcl9tZXRob2QudmFsdWVfY291bnRzKCkudG9fZGljdCgpDQogICAgaWYgJ29yaWVudF9jb2RlJyBpbiBvayBhbmQgJ3BsYW5lX2dlbycgaW4gb2s6DQogICAgICAgIHJlcFsnb3JpZW50X2NvZGVzJ10gPSB7Zid7cH06e2N9JzogaW50KG4pIGZvciAocCwgYyksIG4gaW4gb2suZ3JvdXBieSgncGxhbmVfZ2VvJykub3JpZW50X2NvZGUudmFsdWVfY291bnRzKCkuaXRlbXMoKX0NCiAgICAgICAgYmFkID0gMA0KICAgICAgICBmb3IgcGwsIGNvZGUgaW4gY29uZmlnLkNBTk9OX09SSUVOVC5pdGVtcygpOg0KICAgICAgICAgICAgc3ViID0gb2tbb2sucGxhbmVfZ2VvID09IHBsXQ0KICAgICAgICAgICAgYmFkICs9IGludCgoc3ViLm9yaWVudF9jb2RlICE9IGNvZGUpLnN1bSgpKQ0KICAgICAgICByZXBbJ25vbl9jYW5vbmljYWxfb3JpZW50YXRpb24nXSA9IGJhZA0KICAgIGlmICdBbmF0b21pY2FsX1BsYW5lJyBpbiBvazoNCiAgICAgICAgbSA9IG9rW29rLkFuYXRvbWljYWxfUGxhbmUubm90bmEoKSAmIG9rLnBsYW5lX2dlby5ub3RuYSgpXQ0KICAgICAgICByZXBbJ2Nzdl9wbGFuZV92c19nZW9tZXRyeV9taXNtYXRjaCddID0gaW50KChtLkFuYXRvbWljYWxfUGxhbmUgIT0gbS5wbGFuZV9nZW8pLnN1bSgpKQ0KICAgIGlmICduX2Ryb3BwZWRfc2hhcGUnIGluIG9rOg0KICAgICAgICByZXBbJ3Nlcmllc193aXRoX21peGVkX3NoYXBlcyddID0gaW50KChvay5uX2Ryb3BwZWRfc2hhcGUgPiAwKS5zdW0oKSkNCiAgICBpZiAnbl9iYWQnIGluIG9rOg0KICAgICAgICByZXBbJ3Nlcmllc193aXRoX3VucmVhZGFibGVfc2xpY2VzJ10gPSBpbnQoKG9rLm5fYmFkID4gMCkuc3VtKCkpDQogICAgcmV0dXJuIHJlcA0K',
    'src/data/preprocess/loader.py': 'IiIiQ2FjaGUgLT4gdHJhaW5pbmcvZXZhbCBzYW1wbGUgKG51bXB5IG9ubHksIHNvIGl0IGlzIHRlc3RhYmxlIHdpdGhvdXQgdG9yY2g7IGRhdGFzZXQucHkganVzdCB3cmFwcyBpdCkuIiIiDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCmZyb20gLiBpbXBvcnQgc2FtcGxpbmcNCg0KDQpkZWYgbWFrZV9zYW1wbGUoY2FjaGUsIGksIGNmZywgdHJhaW49RmFsc2UsIG5fdXNlPU5vbmUsIHJuZz1Ob25lLCBhdWc9Tm9uZSk6DQogICAgIiIiT25lIHN0dWR5IGZyb20gdGhlIG1lbW1hcCBjYWNoZS4NCg0KICAgIFJldHVybnMgaW1ncyB1aW50OCBbUywgVywgRywgSCwgV10sIHNsb3RfbWFzayB1aW50OCBbU10sIHdpbl9tYXNrIGJvb2wgW1MsIFddLg0KICAgIHRyYWluOiBpbmRlcGVuZGVudCBzdHJhdGlmaWVkLXJhbmRvbSB3aW5kb3dzIHBlciBzbG90ICsgb25lIGdlb21ldHJpYy9pbnRlbnNpdHkgYXVnbWVudGF0aW9uIHBlciBzbG90Lg0KICAgIGV2YWwgOiB0aGUgc2FtZSBldmVubHkgc3BhY2VkIHdpbmRvd3MgZm9yIGV2ZXJ5IHNsb3QgKG5fdXNlPU5vbmUgLT4gYWxsIHdpbmRvd3MpLg0KICAgIGF1ZyAgOiBpZiBOb25lLCBkZWZhdWx0cyB0byB0cmFpbi4gSWYgZXhwbGljaXRseSBib29sLCBmb3JjZXMgYXVnbWVudGF0aW9uIG9uIG9yIG9mZiAoZS5nLiBmb3IgVFRBKS4iIiINCiAgICBTLCBHLCBIID0gY29uZmlnLk5fU0xPVFMsIGNmZy5ncm91cCwgY2ZnLmltZ19zaXplDQogICAgRCwgc3RyaWRlID0gY2ZnLnN0YWNrX2RlcHRoLCBjZmcud2luX3N0cmlkZQ0KICAgIFcgPSBzYW1wbGluZy5uX3dpbmRvd3MoRCwgRywgc3RyaWRlKSBpZiBuX3VzZSBpcyBOb25lIGVsc2UgbWluKG5fdXNlLCBzYW1wbGluZy5uX3dpbmRvd3MoRCwgRywgc3RyaWRlKSkNCiAgICBpbWdzID0gbnAuemVyb3MoKFMsIFcsIEcsIEgsIEgpLCBucC51aW50OCkNCiAgICB3bWFzayA9IG5wLnplcm9zKChTLCBXKSwgYm9vbCkNCiAgICBybmcgPSBybmcgaWYgcm5nIGlzIG5vdCBOb25lIGVsc2UgbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApDQogICAgc2hvdWxkX2F1ZyA9IGF1ZyBpZiBhdWcgaXMgbm90IE5vbmUgZWxzZSB0cmFpbg0KICAgIGZvciBzIGluIHJhbmdlKFMpOg0KICAgICAgICBpZiBub3QgY2FjaGUuc2xvdFtpLCBzXToNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHN0YXJ0cyA9IHNhbXBsaW5nLndpbmRvd19zdGFydHMoRCwgRywgc3RyaWRlLCBXLCB0cmFpbiwgcm5nKQ0KICAgICAgICB3aW4sIHd2ID0gc2FtcGxpbmcuZ2F0aGVyX3dpbmRvd3MoY2FjaGUuaW1hZ2VzW2ksIHNdLCBjYWNoZS52YWxpZFtpLCBzXSwgc3RhcnRzLCBHKQ0KICAgICAgICBpZiBzaG91bGRfYXVnOg0KICAgICAgICAgICAgd2luID0gc2FtcGxpbmcuYXVnbWVudF9zbG90KHdpbiwgcm5nLCBjb25maWcuQVVHX1JPVF9ERUcsIGNvbmZpZy5BVUdfU0NBTEUsIGNvbmZpZy5BVUdfU0hJRlQsIGNvbmZpZy5BVUdfSU5URU5TSVRZKQ0KICAgICAgICBuID0gbGVuKHN0YXJ0cykNCiAgICAgICAgaW1nc1tzLCA6bl0sIHdtYXNrW3MsIDpuXSA9IHdpbiwgd3YNCiAgICBzbG90ID0gKG5wLmFzYXJyYXkoY2FjaGUuc2xvdFtpXSkgPiAwKSAmIHdtYXNrLmFueShheGlzPTEpDQogICAgcmV0dXJuIGltZ3MsIHNsb3QuYXN0eXBlKG5wLnVpbnQ4KSwgd21hc2sNCg0KDQpkZWYgbGFiZWxfYXJyYXlzKGRmLCBzdHVkaWVzX2luZGV4KToNCiAgICAiIiJUYXJnZXRzL3dlaWdodHMgYXMgZGVuc2UgZmxvYXQzMiBhcnJheXMgYWxpZ25lZCB0byBkZiByb3dzIChubyBwYW5kYXMgaW4gdGhlIGhvdCBsb29wKS4NCg0KICAgIE1pc3NpbmcgbGFiZWwgKE5hTikgLT4gdGFyZ2V0IDAgYW5kIHdlaWdodCAwIChtYXNrZWQpLCBmaXhpbmcgdGhlIE5hTi1sb3NzIGJ1ZyAoOTguNyUgb2Ygc3R1ZGllcyBhcmUgdW5sYWJlbGVkKS4NCiAgICBBbiBvcHRpb25hbCAnPHRhcmdldD5fd2VpZ2h0JyBjb2x1bW4gc2NhbGVzIGxhYmVsbGVkIGVudHJpZXMgKGUuZy4gcmVwb3J0LWV4dHJhY3RvciBjb25maWRlbmNlKS4iIiINCiAgICB2YWxpZF9tYXNrID0gZGZbJ1N0dWR5SW5zdGFuY2VVSUQnXS5pc2luKHN0dWRpZXNfaW5kZXgpDQogICAgaWYgbm90IHZhbGlkX21hc2suYWxsKCk6DQogICAgICAgIGRmID0gZGZbdmFsaWRfbWFza10ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQ0KICAgIFkgPSBucC56ZXJvcygobGVuKGRmKSwgbGVuKGNvbmZpZy5UQVJHRVRTKSksIG5wLmZsb2F0MzIpDQogICAgV3QgPSBucC56ZXJvc19saWtlKFkpDQogICAgZm9yIGosIHQgaW4gZW51bWVyYXRlKGNvbmZpZy5UQVJHRVRTKToNCiAgICAgICAgaWYgdCBub3QgaW4gZGYuY29sdW1uczoNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHkgPSBkZlt0XS5hc3R5cGUoZmxvYXQpLnZhbHVlcw0KICAgICAgICBsYWIgPSBucC5pc2Zpbml0ZSh5KQ0KICAgICAgICBZWzosIGpdID0gbnAud2hlcmUobGFiLCB5LCAwLjApDQogICAgICAgIHcgPSBkZltmJ3t0fV93ZWlnaHQnXS5hc3R5cGUoZmxvYXQpLnZhbHVlcyBpZiBmJ3t0fV93ZWlnaHQnIGluIGRmLmNvbHVtbnMgZWxzZSBucC5vbmVzKGxlbihkZikpDQogICAgICAgIFd0WzosIGpdID0gbnAud2hlcmUobGFiLCBucC5uYW5fdG9fbnVtKHcsIG5hbj0xLjApLCAwLjApDQogICAgcm93cyA9IG5wLmFycmF5KFtzdHVkaWVzX2luZGV4W3NdIGZvciBzIGluIGRmWydTdHVkeUluc3RhbmNlVUlEJ11dLCBkdHlwZT1ucC5pbnQ2NCkNCiAgICByZXR1cm4gcm93cywgWSwgV3QNCg==',
    'src/data/preprocess/nlp_extractor.py': '...',
    'src/data/preprocess/pipeline.py': 'IiIiT25lIGNvZGUgcGF0aCBmb3IgZXZlcnl0aGluZzogb2ZmbGluZSBjYWNoZSBidWlsZCwgdHJhaW5pbmcgZmFsbGJhY2ssIGFuZCB0aGUgaW5mZXJlbmNlIG5vdGVib29rLg0KDQpwcmVwYXJlX3NlcmllcyAvIHByZXBhcmVfc3R1ZHkgTkVWRVIgcmFpc2UgKGhpZGRlbi1yZXJ1biBsZXNzb246IGRhdGEtZGVwZW5kZW50IGFib3J0cyBhcmUgd2hhdCBicm9rZSBlYXJsaWVyDQpwdWJsaWMgdmVyc2lvbnMpLiBFdmVyeSBwcm9ibGVtIGJlY29tZXMgYW4gZW50cnkgaW4gYGluZm9gIGFuZCBhIG1hc2tlZCBzbG90LCBuZXZlciBhIGNyYXNoIG9yIGEgTmFOLg0KIiIiDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCmZyb20gLiBpbXBvcnQgZ2VvbWV0cnkgYXMgZ2VvDQpmcm9tIC4gaW1wb3J0IHBpeGVscw0KDQoNCmRlZiBpbmRleF90b19yZWNvcmRzKGRmKToNCiAgICAiIiJEYXRhRnJhbWUgLT4ge1Nlcmllc0luc3RhbmNlVUlEOiBkaWN0fS4gRG9uZSBvbmNlOyBhdm9pZHMgcGFuZGFzIGluIHRoZSBob3QgcGF0aC4iIiINCiAgICByZXR1cm4ge3JbJ1Nlcmllc0luc3RhbmNlVUlEJ106IHIgZm9yIHIgaW4gZGYudG9fZGljdCgncmVjb3JkcycpfSBpZiBsZW4oZGYpIGVsc2Uge30NCg0KDQpkZWYgc3R1ZHlfc2lkZXMoaW5kZXhfZGYpOg0KICAgICIiIntzdHVkeTogbGF0ZXJhbGl0eSBkaWN0fSB1c2luZyB0aGUgRElDT00gdGFnIGZpcnN0LCBwYXRpZW50LXggZ2VvbWV0cnkgc2Vjb25kLiIiIg0KICAgIGlmIGluZGV4X2RmLmVtcHR5IG9yICdTdHVkeUluc3RhbmNlVUlEJyBub3QgaW4gaW5kZXhfZGYuY29sdW1uczoNCiAgICAgICAgcmV0dXJuIHt9DQogICAgb3V0ID0ge30NCiAgICBjb2xzID0gW2MgZm9yIGMgaW4gKCdTdHVkeUluc3RhbmNlVUlEJywgJ0xhdGVyYWxpdHknLCAnSW1hZ2VMYXRlcmFsaXR5JywgJ2NlbnRlcl94JykgaWYgYyBpbiBpbmRleF9kZi5jb2x1bW5zXQ0KICAgIGZvciBzdCwgZyBpbiBpbmRleF9kZltjb2xzXS5ncm91cGJ5KCdTdHVkeUluc3RhbmNlVUlEJywgc29ydD1GYWxzZSk6DQogICAgICAgIG91dFtzdF0gPSBnZW8ucmVzb2x2ZV9sYXRlcmFsaXR5KGcudG9fZGljdCgncmVjb3JkcycpKQ0KICAgIHJldHVybiBvdXQNCg0KDQpkZWYgcHJlcGFyZV9zZXJpZXMocmVjLCBwbGFuZSwgc2lkZSwgY2ZnKToNCiAgICAiIiItPiAodWludDggW0QsUyxTXSwgdmFsaWQgW0RdLCBpbmZvKSBvciAoTm9uZSwgTm9uZSwgaW5mbykuIiIiDQogICAgc3RhY2ssIHZhbGlkLCBpbmZvID0gcGl4ZWxzLmJ1aWxkX3N0YWNrKHJlYywgY2ZnKQ0KICAgIGlmIHN0YWNrIGlzIE5vbmU6DQogICAgICAgIHJldHVybiBOb25lLCBOb25lLCBpbmZvDQogICAgaW5mb1sncmVvcmllbnRlZCddID0gRmFsc2UNCiAgICBpZiBjZmcucmVvcmllbnQ6DQogICAgICAgIHN0YWNrLCBjaGFuZ2VkID0gZ2VvLnJlb3JpZW50X3N0YWNrKHN0YWNrLCByZWMuZ2V0KCdvcmllbnRfY29kZScpLCBwbGFuZSkNCiAgICAgICAgaW5mb1sncmVvcmllbnRlZCddID0gYm9vbChjaGFuZ2VkKSAgICAgICAgICAjIChhIHRyYW5zcG9zaXRpb24gbGVhdmVzIHRoZSBkZXB0aCBheGlzIHVudG91Y2hlZCkNCiAgICBpZiBjZmcubGF0X2Nhbm9uIGFuZCBzaWRlID09ICdSJzoNCiAgICAgICAgc3RhY2sgPSBnZW8uY2Fub25pY2FsX3NpZGUoc3RhY2ssIHBsYW5lLCAnUicpDQogICAgICAgIGlmIHBsYW5lID09ICdTYWdpdHRhbCc6DQogICAgICAgICAgICB2YWxpZCA9IHZhbGlkWzo6LTFdLmNvcHkoKQ0KICAgICAgICBpbmZvWydtaXJyb3JlZCddID0gVHJ1ZQ0KICAgIHJldHVybiBzdGFjaywgdmFsaWQsIGluZm8NCg0KDQpkZWYgcHJlcGFyZV9zdHVkeShzdHVkeSwgc2xvdF9yb3csIHJlY29yZHMsIHNpZGUsIGNmZyk6DQogICAgIiIic2xvdF9yb3c6IHtzbG90X25hbWU6IFNlcmllc0luc3RhbmNlVUlEIG9yIE5vbmV9Lg0KDQogICAgLT4gZGljdChzdGFjayB1aW50OCBbUyxELEgsV10sIHZhbGlkIGJvb2wgW1MsRF0sIHNsb3RfbWFzayB1aW50OCBbU10sIGluZm8ge3Nsb3Q6IGluZm99LCBzaWRlKSIiIg0KICAgIFMsIEQsIEggPSBjb25maWcuTl9TTE9UUywgY2ZnLnN0YWNrX2RlcHRoLCBjZmcuaW1nX3NpemUNCiAgICBzdGFjayA9IG5wLnplcm9zKChTLCBELCBILCBIKSwgbnAudWludDgpDQogICAgdmFsaWQgPSBucC56ZXJvcygoUywgRCksIGJvb2wpDQogICAgbWFzayA9IG5wLnplcm9zKFMsIG5wLnVpbnQ4KQ0KICAgIGluZm9zID0ge30NCiAgICBmb3IgaywgKG5hbWUsIHBsYW5lLCBfLCBfKSBpbiBlbnVtZXJhdGUoY29uZmlnLlNMT1RTKToNCiAgICAgICAgdWlkID0gc2xvdF9yb3cuZ2V0KG5hbWUpDQogICAgICAgIGlmIG5vdCB1aWQgb3IgdWlkIG5vdCBpbiByZWNvcmRzOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgc3QsIHYsIGluZm8gPSBwcmVwYXJlX3NlcmllcyhyZWNvcmRzW3VpZF0sIHBsYW5lLCBzaWRlLCBjZmcpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgYmVsdCBhbmQgYnJhY2VzDQogICAgICAgICAgICBzdCwgdiwgaW5mbyA9IE5vbmUsIE5vbmUsIGRpY3QoZXJyb3JzPVtmJ3t0eXBlKGUpLl9fbmFtZV9ffToge3N0cihlKVs6NjBdfSddKQ0KICAgICAgICBpbmZvc1tuYW1lXSA9IGluZm8NCiAgICAgICAgaWYgc3QgaXMgTm9uZToNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHN0YWNrW2tdLCB2YWxpZFtrXSwgbWFza1trXSA9IHN0LCB2LCAxDQogICAgcmV0dXJuIGRpY3Qoc3RhY2s9c3RhY2ssIHZhbGlkPXZhbGlkLCBzbG90X21hc2s9bWFzaywgaW5mbz1pbmZvcywgc2lkZT1zaWRlKQ0K',
    'src/data/preprocess/pixels.py': 'IiIiUGl4ZWwgcGlwZWxpbmU6IGRlY29kZSAtPiBwaHlzaWNhbCBjcm9wIC0+IHJvYnVzdCBub3JtYWxpc2UgLT4gcmVzYW1wbGUgLT4gdWludDggc3RhY2suDQoNClRocm91Z2hwdXQgZGVzaWduIChlYWNoIGNob2ljZSBtZWFzdXJlZC9qdXN0aWZpZWQgaW4gdG9vbHMvYmVuY2gucHkpOg0KICAqIGRlY29kZSBSQVcgaW50ZWdlcnMsIGNyb3AgRklSU1QsIGNvbnZlcnQgb25seSB0aGUgY3JvcHBlZCB3aW5kb3cgdG8gZmxvYXQzMg0KICAqIHNsYWIgYXZlcmFnaW5nIG9mIHRoaW4gMy1EIHNsaWNlcyBoYXBwZW5zIGF0IGNyb3AgcmVzb2x1dGlvbiwgb25jZSBwZXIgdGFyZ2V0IHNsaWNlDQogICogcGVyY2VudGlsZXMgb24gYSAyeDItc3Vic2FtcGxlZCBzdGFjayAoZXhhY3QgcGVyY2VudGlsZXMgZm9yIHRoZSAncHVibGljJyBwcmVzZXQpDQogICogY3YyIHJlc2l6ZSAoU0lNRCkgaW5zdGVhZCBvZiBwZXItc2xpY2UgdG9yY2ggY2FsbHM7IGN2Mi9CTEFTIHRocmVhZHMgcGlubmVkIHRvIDEgaW5zaWRlIHdvcmtlcnMNCkRhdGFzZXQtZHJpdmVuIHJ1bGVzIChhdWRpdCk6IHBoeXNpY2FsIG1tIGNyb3Agd2l0aCBlYWNoIHNlcmllcycgb3duIHJvdy9jb2x1bW4gc3BhY2luZyAoNy4zJSBvZiBtYXRyaWNlcyBhcmUNCm5vbi1zcXVhcmUsIDAuMzklIGhhdmUgbm9uLXNxdWFyZSBwaXhlbHMsIDY0MHgxMjgwIHdpZGUgZnJhbWVzIGV4aXN0KSwgYm9yZGVyLW1lZGlhbiBwYWRkaW5nICg3MiUgb2Ygc2VyaWVzIGhhdmUgYQ0Kbm9uLXplcm8gbm9pc2UgZmxvb3IpLCBuZWdhdGl2ZXMgY2xpcHBlZCAoMjQxIENhbm9uIHNlcmllcyksIHBlci1zZXJpZXMgcm9idXN0IHNjYWxlICgyMnggc3ByZWFkKSwgaGVhdnkgdGFpbHMNCig4LjIlIG9mIHNlcmllcyBoYXZlIHA5OS45ID4gMnggcDk5KS4NCiIiIg0KaW1wb3J0IG9zDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBjdjINCg0KZnJvbSAuIGltcG9ydCBkaWNvbWlvIGFzIGRpbw0KDQpjdjIuc2V0TnVtVGhyZWFkcygwKSAgICAgICAgICAgICAgICAgICAgIyBwYXJhbGxlbGlzbSBjb21lcyBmcm9tIHdvcmtlciBwcm9jZXNzZXM7IGF2b2lkIG92ZXJzdWJzY3JpcHRpb24NClBTX0ZBTExCQUNLID0gMC4zMTI1ICAgICAgICAgICAgICAgICAgICAjIGRhdGFzZXQgbWVkaWFuIGluLXBsYW5lIHNwYWNpbmcgKG1tKSwgb25seSBpZiB0aGUgaGVhZGVyIGhhcyBub25lDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0geiBzYW1wbGluZw0KZGVmIHpfZ3JvdXBzKHBvcywgY2ZnLCBzcGFjaW5nLCBtZXRob2Q9J3Bvc2l0aW9uJyk6DQogICAgIiIiV2hpY2ggbmF0aXZlIHNsaWNlcyBmZWVkIGVhY2ggb2YgdGhlIGNmZy5zdGFja19kZXB0aCBzdG9yZWQgc2xpY2VzLg0KDQogICAgUmV0dXJucyAoZ3JvdXBzOiBsaXN0W2xpc3RbaW50XV0sIHZhbGlkOiBib29sW0RdKS4NCiAgICAgICdpbmRleCcgKHB1YmxpYyk6IGxpbnNwYWNlIG92ZXIgY2ZnLmJhbmQgb2YgdGhlIG9yZGVyZWQgc2xpY2UgaW5kaWNlcywgZHVwbGljYXRlcyBwYWRkZWQNCiAgICAgICdtbSc6IHBoeXNpY2FsIGdyaWQgY2ZnLnpfc3RlcF9tbSBjZW50cmVkIG9uIHRoZSBzZXJpZXM7IG5lYXJlc3QgbmF0aXZlIHNsaWNlLCBvciB0aGUgbWVhbiBvZiBhbGwgbmF0aXZlDQogICAgICAgICAgICBzbGljZXMgaW5zaWRlICstc3RlcC8yIHdoZW4gdGhlIG5hdGl2ZSBzcGFjaW5nIGlzIGZpbmVyIHRoYW4gMC43NSpzdGVwICgzLUQgLyB0aGluLXNsaWNlIHNlcmllcykNCiAgICAiIiINCiAgICBuID0gbGVuKHBvcykNCiAgICBEID0gY2ZnLnN0YWNrX2RlcHRoDQogICAgaWYgY2ZnLnpfbW9kZSA9PSAnaW5kZXgnIG9yIG1ldGhvZCAhPSAncG9zaXRpb24nOg0KICAgICAgICAjIHB1YmxpYyBsaW5zcGFjZSBvdmVyIHRoZSBiYW5kOyBhbHNvIHRoZSBzYWZlIGZhbGxiYWNrIHdoZW4gc2xpY2Ugb3JkZXIgY2FtZSBmcm9tIEluc3RhbmNlTnVtYmVyL25hbWUNCiAgICAgICAgIyAocG9zaXRpb25zIGFyZSB0aGVuIGluZGV4IHVuaXRzLCBub3QgbW0sIHNvIGEgbWlsbGltZXRyZSBncmlkIHdvdWxkIGJlIG1lYW5pbmdsZXNzKQ0KICAgICAgICBiYW5kID0gY2ZnLmJhbmQgaWYgY2ZnLnpfbW9kZSA9PSAnaW5kZXgnIGVsc2UgKDAuMCwgMS4wKQ0KICAgICAgICBsbywgaGkgPSBpbnQoYmFuZFswXSAqIChuIC0gMSkpLCBpbnQoYmFuZFsxXSAqIChuIC0gMSkpDQogICAgICAgIGlkeCA9IG5wLnVuaXF1ZShucC5saW5zcGFjZShsbywgaGksIEQpLmFzdHlwZShpbnQpKSBpZiBoaSA+IGxvIGVsc2UgbnAuYXJyYXkoW24gLy8gMl0pDQogICAgICAgIHdoaWxlIGxlbihpZHgpIDwgRDoNCiAgICAgICAgICAgIGlkeCA9IG5wLmFwcGVuZChpZHgsIGlkeFstMV0pDQogICAgICAgIHJldHVybiBbW2ludChpKV0gZm9yIGkgaW4gaWR4WzpEXV0sIG5wLm9uZXMoRCwgYm9vbCkNCiAgICBwb3MgPSBucC5hc2FycmF5KHBvcywgbnAuZmxvYXQ2NCkNCiAgICBzdGVwID0gY2ZnLnpfc3RlcF9tbQ0KICAgIG1pZCA9IDAuNSAqIChwb3NbMF0gKyBwb3NbLTFdKQ0KICAgIHRhcmdldHMgPSBtaWQgKyAobnAuYXJhbmdlKEQpIC0gKEQgLSAxKSAvIDIuMCkgKiBzdGVwDQogICAgc3AgPSBzcGFjaW5nIGlmIG5wLmlzZmluaXRlKHNwYWNpbmcpIGFuZCBzcGFjaW5nID4gMCBlbHNlIHN0ZXANCiAgICB0b2wgPSAwLjc1ICogbWF4KHNwLCBzdGVwKQ0KICAgIG5lYXIgPSBucC5jbGlwKG5wLnNlYXJjaHNvcnRlZChwb3MsIHRhcmdldHMpLCAxLCBtYXgobiAtIDEsIDEpKQ0KICAgIGlmIG4gPT0gMToNCiAgICAgICAgbmVhciA9IG5wLnplcm9zKEQsIGludCkNCiAgICBlbHNlOg0KICAgICAgICBsZWZ0LCByaWdodCA9IHBvc1tuZWFyIC0gMV0sIHBvc1tuZWFyXQ0KICAgICAgICBuZWFyID0gbnAud2hlcmUobnAuYWJzKHRhcmdldHMgLSBsZWZ0KSA8PSBucC5hYnMocmlnaHQgLSB0YXJnZXRzKSwgbmVhciAtIDEsIG5lYXIpDQogICAgZGlzdCA9IG5wLmFicyhwb3NbbmVhcl0gLSB0YXJnZXRzKQ0KICAgIHZhbGlkID0gZGlzdCA8PSB0b2wNCiAgICBncm91cHMgPSBbXQ0KICAgIHNsYWIgPSBzcCA8IDAuNzUgKiBzdGVwDQogICAgZm9yIHQsIGsgaW4gemlwKHRhcmdldHMsIG5lYXIpOg0KICAgICAgICBpZiBzbGFiOg0KICAgICAgICAgICAgbG9faSA9IGludChucC5zZWFyY2hzb3J0ZWQocG9zLCB0IC0gc3RlcCAvIDIuMCwgJ2xlZnQnKSkNCiAgICAgICAgICAgIGhpX2kgPSBpbnQobnAuc2VhcmNoc29ydGVkKHBvcywgdCArIHN0ZXAgLyAyLjAsICdyaWdodCcpKQ0KICAgICAgICAgICAgZ3JvdXBzLmFwcGVuZChsaXN0KHJhbmdlKGxvX2ksIGhpX2kpKSBpZiBoaV9pID4gbG9faSBlbHNlIFtpbnQoayldKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgZ3JvdXBzLmFwcGVuZChbaW50KGspXSkNCiAgICByZXR1cm4gZ3JvdXBzLCB2YWxpZA0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGNyb3AgZ2VvbWV0cnkNCmRlZiBfYm9yZGVyX21lZGlhbihhKToNCiAgICBoLCB3ID0gYS5zaGFwZQ0KICAgIGIgPSBtYXgoMiwgaW50KDAuMDIgKiBtaW4oaCwgdykpKQ0KICAgIHJpbmcgPSBucC5jb25jYXRlbmF0ZShbYVs6Yl0ucmF2ZWwoKSwgYVstYjpdLnJhdmVsKCksIGFbYjotYiwgOmJdLnJhdmVsKCksIGFbYjotYiwgLWI6XS5yYXZlbCgpXSkNCiAgICByZXR1cm4gZmxvYXQobnAubWVkaWFuKHJpbmcpKQ0KDQoNCmRlZiBfZmdfY2VudGVyKHJhd3MpOg0KICAgICIiIkZvcmVncm91bmQgYmJveCBjZW50cmUgKHJvdywgY29sKSBpbiBmdWxsLWZyYW1lIHBpeGVscywgZnJvbSB0aGUgbWVhbiBvZiB0aGUgZGVjb2RlZCBzbGljZXMuIiIiDQogICAgcmVmID0gbnAubWVhbihbbnAubWF4aW11bShhWzo6NCwgOjo0XS5hc3R5cGUobnAuZmxvYXQzMiksIDApIGZvciBhIGluIHJhd3NdLCBheGlzPTApDQogICAgcDk5ID0gbnAucGVyY2VudGlsZShyZWYsIDk5KQ0KICAgIGlmIHA5OSA8PSAwOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIGZnID0gcmVmID4gMC4xICogcDk5DQogICAgaWYgbm90IGZnLmFueSgpOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHJyLCBjYyA9IG5wLndoZXJlKGZnLmFueSgxKSlbMF0sIG5wLndoZXJlKGZnLmFueSgwKSlbMF0NCiAgICByZXR1cm4gNC4wICogKHJyWzBdICsgcnJbLTFdKSAvIDIuMCwgNC4wICogKGNjWzBdICsgY2NbLTFdKSAvIDIuMA0KDQoNCmRlZiBjcm9wX3BsYW4oc2hhcGUsIHBzX3JvdywgcHNfY29sLCBjZmcsIHJhd3M9Tm9uZSk6DQogICAgIiIiLT4gKHkwLCB4MCwgSGMsIFdjKSBvZiB0aGUgcGh5c2ljYWwgY3JvcCB3aW5kb3cgaW4gZnVsbC1mcmFtZSBwaXhlbHMgKG1heSBleGNlZWQgdGhlIGZyYW1lIC0+IHBhZGRlZCksDQogICAgb3IgTm9uZSB3aGVuIHRoZSBwdWJsaWMgcnVsZSBza2lwcyB0aGUgY3JvcC4iIiINCiAgICBILCBXID0gc2hhcGUNCiAgICBvayA9IG5wLmlzZmluaXRlKHBzX3JvdykgYW5kIG5wLmlzZmluaXRlKHBzX2NvbCkgYW5kIHBzX3JvdyA+IDAgYW5kIHBzX2NvbCA+IDANCiAgICBpZiBjZmcucGFkX21vZGUgPT0gJ3B1YmxpYyc6ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIHB1YmxpYyBydWxlLCB2ZXJiYXRpbQ0KICAgICAgICBpZiBub3QgKG9rIGFuZCBwc19yb3cgPiAwKToNCiAgICAgICAgICAgIHJldHVybiBOb25lDQogICAgICAgIHdhbnQgPSBpbnQocm91bmQoY2ZnLmNyb3BfbW0gLyBwc19yb3cpKQ0KICAgICAgICBpZiAxNiA8IHdhbnQgPCBtaW4oSCwgVyk6DQogICAgICAgICAgICBjeSwgY3ggPSBIIC8vIDIsIFcgLy8gMg0KICAgICAgICAgICAgaGFsZiA9IHdhbnQgLy8gMg0KICAgICAgICAgICAgcmV0dXJuIG1heCgwLCBjeSAtIGhhbGYpLCBtYXgoMCwgY3ggLSBoYWxmKSwgbWluKEgsIGN5ICsgaGFsZikgLSBtYXgoMCwgY3kgLSBoYWxmKSwgXA0KICAgICAgICAgICAgICAgIG1pbihXLCBjeCArIGhhbGYpIC0gbWF4KDAsIGN4IC0gaGFsZikNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICBpZiBub3Qgb2s6DQogICAgICAgIHBzX3JvdyA9IHBzX2NvbCA9IFBTX0ZBTExCQUNLDQogICAgSGMgPSBtYXgoMTYsIGludChyb3VuZChjZmcuY3JvcF9tbSAvIHBzX3JvdykpKQ0KICAgIFdjID0gbWF4KDE2LCBpbnQocm91bmQoY2ZnLmNyb3BfbW0gLyBwc19jb2wpKSkNCiAgICBjeSwgY3ggPSAoSCAtIDEpIC8gMi4wLCAoVyAtIDEpIC8gMi4wDQogICAgYXNwZWN0ID0gKFcgKiBwc19jb2wpIC8gKEggKiBwc19yb3cpDQogICAgaWYgY2ZnLndpZGVfY2VudGVyIGFuZCByYXdzIGFuZCAoYXNwZWN0ID4gY2ZnLndpZGVfcmF0aW8gb3IgYXNwZWN0IDwgMS4wIC8gY2ZnLndpZGVfcmF0aW8pOg0KICAgICAgICBjID0gX2ZnX2NlbnRlcihyYXdzKQ0KICAgICAgICBpZiBjIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgaWYgYXNwZWN0ID4gMToNCiAgICAgICAgICAgICAgICBjeCA9IGNbMV0NCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgY3kgPSBjWzBdDQogICAgeTAgPSBpbnQocm91bmQoY3kgLSAoSGMgLSAxKSAvIDIuMCkpDQogICAgeDAgPSBpbnQocm91bmQoY3ggLSAoV2MgLSAxKSAvIDIuMCkpDQogICAgcmV0dXJuIHkwLCB4MCwgSGMsIFdjDQoNCg0KZGVmIF9jcm9wX2Zsb2F0KHJhdywgc2xvcGUsIGljcHQsIHBsYW4sIGJnX2ZuKToNCiAgICAiIiJDcm9wICh3aXRoIGJvcmRlci1tZWRpYW4gcGFkZGluZykgYW5kIGNvbnZlcnQgT05MWSB0aGUgd2luZG93IHRvIGZsb2F0MzIuIiIiDQogICAgSCwgVyA9IHJhdy5zaGFwZQ0KICAgIGlmIHBsYW4gaXMgTm9uZToNCiAgICAgICAgcmV0dXJuIHJhdy5hc3R5cGUobnAuZmxvYXQzMikgKiBzbG9wZSArIGljcHQNCiAgICB5MCwgeDAsIEhjLCBXYyA9IHBsYW4NCiAgICB5czAsIHhzMCwgeXMxLCB4czEgPSBtYXgoeTAsIDApLCBtYXgoeDAsIDApLCBtaW4oeTAgKyBIYywgSCksIG1pbih4MCArIFdjLCBXKQ0KICAgIHdpbiA9IHJhd1t5czA6eXMxLCB4czA6eHMxXS5hc3R5cGUobnAuZmxvYXQzMikgKiBzbG9wZSArIGljcHQNCiAgICBpZiAoeXMxIC0geXMwLCB4czEgLSB4czApID09IChIYywgV2MpOg0KICAgICAgICByZXR1cm4gd2luDQogICAgb3V0ID0gbnAuZnVsbCgoSGMsIFdjKSwgYmdfZm4ocmF3LCBzbG9wZSwgaWNwdCksIG5wLmZsb2F0MzIpDQogICAgb3V0W3lzMCAtIHkwOnlzMSAtIHkwLCB4czAgLSB4MDp4czEgLSB4MF0gPSB3aW4NCiAgICByZXR1cm4gb3V0DQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gbm9ybWFsaXNlIC8gcmVzaXplDQpkZWYgbm9ybWFsaXNlKHZvbCwgdmFsaWQsIGNmZyk6DQogICAgIiIiU3RhY2std2lkZSByb2J1c3Qgc2NhbGluZyB0byBbMCwxXSAoYWxsIHNsaWNlcyBzaGFyZSBvbmUgd2luZG93OyBpbnZhbGlkIHNsaWNlcyB1bnRvdWNoZWQpLiIiIg0KICAgIGlmIGNmZy5jbGlwX25lZ2F0aXZlOg0KICAgICAgICBucC5tYXhpbXVtKHZvbCwgMCwgb3V0PXZvbCkNCiAgICB2ID0gdm9sW3ZhbGlkXQ0KICAgIHN1YiA9IHYgaWYgY2ZnLm5hbWUgPT0gJ3B1YmxpYycgZWxzZSB2WzosIDo6MiwgOjoyXQ0KICAgIGxvLCBoaSA9IG5wLnBlcmNlbnRpbGUoc3ViLCBbY2ZnLm5vcm1fbG8sIGNmZy5ub3JtX2hpXSkNCiAgICB2b2wgLT0gbG8NCiAgICB2b2wgLz0gbWF4KGZsb2F0KGhpIC0gbG8pLCAxZS02KQ0KICAgIG5wLmNsaXAodm9sLCAwLjAsIDEuMCwgb3V0PXZvbCkNCiAgICByZXR1cm4gdm9sDQoNCg0KZGVmIHJlc2l6ZV9zdGFjayh2b2wsIHNpemUsIG1vZGUpOg0KICAgICIiIltELGgsd10gZmxvYXQgLT4gW0Qsc2l6ZSxzaXplXSBmbG9hdC4gJ2FyZWEnID0gYW50aWFsaWFzZWQgd2hlbiBzaHJpbmtpbmcsIGJpbGluZWFyIHdoZW4gZW5sYXJnaW5nLiIiIg0KICAgIEQsIGgsIHcgPSB2b2wuc2hhcGUNCiAgICBpZiBtb2RlID09ICdhcmVhJyBhbmQgKGggPiBzaXplIG9yIHcgPiBzaXplKToNCiAgICAgICAgaW50ZXJwID0gY3YyLklOVEVSX0FSRUENCiAgICBlbHNlOg0KICAgICAgICBpbnRlcnAgPSBjdjIuSU5URVJfTElORUFSDQogICAgb3V0ID0gbnAuZW1wdHkoKEQsIHNpemUsIHNpemUpLCBucC5mbG9hdDMyKQ0KICAgIGZvciBpIGluIHJhbmdlKEQpOg0KICAgICAgICBvdXRbaV0gPSBjdjIucmVzaXplKHZvbFtpXSwgKHNpemUsIHNpemUpLCBpbnRlcnBvbGF0aW9uPWludGVycCkNCiAgICByZXR1cm4gb3V0DQoNCg0KZGVmIHRvX3VpbnQ4KHgpOg0KICAgIHggKj0gMjU1LjANCiAgICBucC5yaW50KHgsIG91dD14KQ0KICAgIHJldHVybiB4LmFzdHlwZShucC51aW50OCkNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBvbmUgc2VyaWVzIC0+IHN0YWNrDQpkZWYgYnVpbGRfc3RhY2socmVjLCBjZmcpOg0KICAgICIiInJlYzogaW5kZXggcm93IChkaWN0LWxpa2UpIHdpdGggZGlyLCBvcmRlcmVkX2ZpbGVzLCBwb3NpdGlvbnMsIG5zaWduLCBwc19yb3csIHBzX2NvbCwgc3BhY2luZ19tZWQuDQoNCiAgICAtPiAodWludDggW0QsUyxTXSwgdmFsaWQgYm9vbCBbRF0sIGluZm8pIG9yIChOb25lLCBOb25lLCBpbmZvKSB3aGVuIG5vdGhpbmcgY291bGQgYmUgZGVjb2RlZC4NCiAgICBOZXZlciByYWlzZXM6IGV2ZXJ5IHNsaWNlIGZhaWx1cmUgaXMgcmVjb3JkZWQgaW4gaW5mb1snZXJyb3JzJ10uIiIiDQogICAgaW5mbyA9IGRpY3QoZXJyb3JzPVtdLCBwYWRkZWQ9RmFsc2UsIHBzX21pc3Npbmc9RmFsc2UsIG5fZGVjb2RlZD0wKQ0KICAgIGZpbGVzID0gbGlzdChyZWNbJ29yZGVyZWRfZmlsZXMnXSkNCiAgICBwb3MgPSBucC5hc2FycmF5KHJlY1sncG9zaXRpb25zJ10sIG5wLmZsb2F0NjQpDQogICAgaWYgbm90IGZpbGVzOg0KICAgICAgICBpbmZvWydlcnJvcnMnXS5hcHBlbmQoJ25vX2ZpbGVzJykNCiAgICAgICAgcmV0dXJuIE5vbmUsIE5vbmUsIGluZm8NCiAgICBpZiBjZmcub3JkZXJfbW9kZSA9PSAnbm9ybWFsJyBhbmQgZmxvYXQocmVjLmdldCgnbnNpZ24nLCAxLjApKSA8IDA6ICAgICAjIHB1YmxpYyBzaWduIGNvbnZlbnRpb24NCiAgICAgICAgZmlsZXMsIHBvcyA9IGZpbGVzWzo6LTFdLCAtcG9zWzo6LTFdDQogICAgbWV0aG9kID0gc3RyKHJlYy5nZXQoJ29yZGVyX21ldGhvZCcsICdwb3NpdGlvbicpKQ0KICAgIGdyb3VwcywgdmFsaWQgPSB6X2dyb3Vwcyhwb3MsIGNmZywgZmxvYXQocmVjLmdldCgnc3BhY2luZ19tZWQnLCBucC5uYW4pKSwgbWV0aG9kKQ0KICAgIGluZm9bJ3pfZmFsbGJhY2snXSA9IGNmZy56X21vZGUgPT0gJ21tJyBhbmQgbWV0aG9kICE9ICdwb3NpdGlvbicNCiAgICBuZWVkID0gc29ydGVkKHtpIGZvciBnIGluIGdyb3VwcyBmb3IgaSBpbiBnfSkNCiAgICByYXdzID0ge30NCiAgICBmb3IgaSBpbiBuZWVkOg0KICAgICAgICB0cnk6DQogICAgICAgICAgICByYXdzW2ldID0gZGlvLmRlY29kZV9yYXcob3MucGF0aC5qb2luKHJlY1snZGlyJ10sIGZpbGVzW2ldKSkNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICAgICAgaW5mb1snZXJyb3JzJ10uYXBwZW5kKGYne2ZpbGVzW2ldfToge3R5cGUoZSkuX19uYW1lX199JykNCiAgICBpbmZvWyduX2RlY29kZWQnXSA9IGxlbihyYXdzKQ0KICAgIGlmIG5vdCByYXdzOg0KICAgICAgICByZXR1cm4gTm9uZSwgTm9uZSwgaW5mbw0KICAgIHNoYXBlID0gbWF4KHtyWzBdLnNoYXBlIGZvciByIGluIHJhd3MudmFsdWVzKCl9LCBrZXk9bGFtYmRhIHM6IHN1bShyWzBdLnNoYXBlID09IHMgZm9yIHIgaW4gcmF3cy52YWx1ZXMoKSkpDQogICAgcmF3cyA9IHtpOiByIGZvciBpLCByIGluIHJhd3MuaXRlbXMoKSBpZiByWzBdLnNoYXBlID09IHNoYXBlfQ0KICAgIHBzX3JvdywgcHNfY29sID0gZmxvYXQocmVjLmdldCgncHNfcm93JywgbnAubmFuKSksIGZsb2F0KHJlYy5nZXQoJ3BzX2NvbCcsIG5wLm5hbikpDQogICAgaW5mb1sncHNfbWlzc2luZyddID0gbm90IChucC5pc2Zpbml0ZShwc19yb3cpIGFuZCBucC5pc2Zpbml0ZShwc19jb2wpIGFuZCBwc19yb3cgPiAwIGFuZCBwc19jb2wgPiAwKQ0KICAgIHBsYW4gPSBjcm9wX3BsYW4oc2hhcGUsIHBzX3JvdywgcHNfY29sLCBjZmcsIFtyWzBdIGZvciByIGluIHJhd3MudmFsdWVzKCldIGlmIGNmZy53aWRlX2NlbnRlciBlbHNlIE5vbmUpDQogICAgaWYgcGxhbiBpcyBub3QgTm9uZToNCiAgICAgICAgaW5mb1sncGFkZGVkJ10gPSBwbGFuWzBdIDwgMCBvciBwbGFuWzFdIDwgMCBvciBwbGFuWzBdICsgcGxhblsyXSA+IHNoYXBlWzBdIG9yIHBsYW5bMV0gKyBwbGFuWzNdID4gc2hhcGVbMV0NCiAgICBkZWYgX2JnKHJhdywgc2xvcGUsIGljcHQpOg0KICAgICAgICAjIEZJWCBCOiBNZWRpYW5zIGNvbW11dGUgd2l0aCBtb25vdG9uaWMgbGluZWFyIHRyYW5zZm9ybWF0aW9ucy4NCiAgICAgICAgIyBFeHRyYWN0IGJvcmRlciBmcm9tIFJBVyBpbnRlZ2VyIGFycmF5LCBjb21wdXRlIG1lZGlhbiwgVEhFTiBhcHBseSBzbG9wZS9pY3B0Lg0KICAgICAgICAjIFNhdmVzIGNvbnZlcnRpbmcgdGhlIGVudGlyZSA4MDAsMDAwKyBwaXhlbCBpbWFnZSB0byBmbG9hdDMyLg0KICAgICAgICByZXR1cm4gZmxvYXQoX2JvcmRlcl9tZWRpYW4ocmF3KSkgKiBzbG9wZSArIGljcHQNCg0KICAgIEQgPSBsZW4oZ3JvdXBzKQ0KICAgIGNyb3BzID0ge30NCiAgICBmb3IgaSwgKHJhdywgc2xvcGUsIGljcHQpIGluIHJhd3MuaXRlbXMoKToNCiAgICAgICAgY3JvcHNbaV0gPSBfY3JvcF9mbG9hdChyYXcsIHNsb3BlLCBpY3B0LCBwbGFuLCBfYmcpDQogICAgcmVmX2kgPSBuZXh0KGl0ZXIoY3JvcHMpKQ0KICAgIHZvbCA9IG5wLnplcm9zKChELCkgKyBjcm9wc1tyZWZfaV0uc2hhcGUsIG5wLmZsb2F0MzIpDQogICAgb2sgPSBucC56ZXJvcyhELCBib29sKQ0KICAgIGZvciBkLCBnIGluIGVudW1lcmF0ZShncm91cHMpOg0KICAgICAgICBnb3QgPSBbY3JvcHNbaV0gZm9yIGkgaW4gZyBpZiBpIGluIGNyb3BzXQ0KICAgICAgICBpZiBub3QgZ290Og0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgdm9sW2RdID0gZ290WzBdIGlmIGxlbihnb3QpID09IDEgZWxzZSBucC5tZWFuKGdvdCwgYXhpcz0wLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgICAgICBva1tkXSA9IFRydWUNCiAgICB2YWxpZCA9IHZhbGlkICYgb2sNCiAgICBpZiBjZmcuel9tb2RlID09ICdpbmRleCcgb3IgbWV0aG9kICE9ICdwb3NpdGlvbic6ICAgIyBmYWlsZWQgc2xpY2VzIGJvcnJvdyB0aGUgbmVhcmVzdCBkZWNvZGVkIG5laWdoYm91cg0KICAgICAgICBmb3IgZCBpbiBucC53aGVyZSh+b2spWzBdOg0KICAgICAgICAgICAgaiA9IGludChucC5hcmdtaW4obnAud2hlcmUob2ssIG5wLmFicyhucC5hcmFuZ2UoRCkgLSBkKSwgMTAgKiogNikpKQ0KICAgICAgICAgICAgdm9sW2RdID0gdm9sW2pdDQogICAgICAgIHZhbGlkID0gbnAub25lcyhELCBib29sKQ0KICAgIGlmIG5vdCB2YWxpZC5hbnkoKToNCiAgICAgICAgcmV0dXJuIE5vbmUsIE5vbmUsIGluZm8NCiAgICB2b2wgPSBub3JtYWxpc2Uodm9sLCB2YWxpZCwgY2ZnKQ0KICAgIHZvbCA9IHJlc2l6ZV9zdGFjayh2b2wsIGNmZy5pbWdfc2l6ZSwgY2ZnLnJlc2l6ZV9tb2RlKQ0KICAgIG91dCA9IHRvX3VpbnQ4KHZvbCkNCiAgICBvdXRbfnZhbGlkXSA9IDANCiAgICByZXR1cm4gb3V0LCB2YWxpZCwgaW5mbw0K',
    'src/data/preprocess/qc.py': 'IiIiRGF0YS1jb250cmFjdCBjaGVja3MuIENoZWFwIGVub3VnaCB0byBydW4gb24gZXZlcnkgYnVpbGQ7IHRoZWlyIG51bWJlcnMgYXJlIHRoZSBhY2NlcHRhbmNlIGdhdGVzLiIiIg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmltcG9ydCBzcmMuY29yZS5jb25maWcgYXMgY29uZmlnDQoNCg0KZGVmIGNhY2hlX3Nhbml0eShjYWNoZSwgbl9zYW1wbGU9MjAwLCBzZWVkPTApOg0KICAgICIiIlNhbXBsZSBidWlsdCBzdHVkaWVzIGFuZCB2ZXJpZnkgdGhlIGNhY2hlIGlzIG5vdCBzaWxlbnRseSBicm9rZW4gKGNvbnN0YW50IGltYWdlcywgZW1wdHktYnV0LWZsYWdnZWQgc2xvdHMsDQogICAgYWxsLWludmFsaWQgZGVwdGgsIHdyb25nIGR0eXBlKS4iIiINCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkNCiAgICBkb25lID0gbnAud2hlcmUobnAuYXNhcnJheShjYWNoZS5kb25lKSA9PSAxKVswXQ0KICAgIGlmIGxlbihkb25lKSA9PSAwOg0KICAgICAgICByZXR1cm4gZGljdChzdHVkaWVzX2NoZWNrZWQ9MCkNCiAgICBwaWNrID0gcm5nLmNob2ljZShkb25lLCBzaXplPW1pbihuX3NhbXBsZSwgbGVuKGRvbmUpKSwgcmVwbGFjZT1GYWxzZSkNCiAgICBiYWQgPSBkaWN0KGNvbnN0YW50X3Nsb3Q9W10sIGZsYWdnZWRfYnV0X2JsYW5rPVtdLCBsb3dfdmFsaWRfZGVwdGg9W10sIHVuZmxhZ2dlZF9idXRfZmlsbGVkPVtdKQ0KICAgIG5fc2xvdHMgPSAwDQogICAgc3RkcywgbWVhbnMsIHZmcmFjID0gW10sIFtdLCBbXQ0KICAgIGZvciBpIGluIHBpY2s6DQogICAgICAgIGZvciBzIGluIHJhbmdlKGNvbmZpZy5OX1NMT1RTKToNCiAgICAgICAgICAgIGZpbGxlZCA9IGJvb2woY2FjaGUuc2xvdFtpLCBzXSkNCiAgICAgICAgICAgIHN0ID0gbnAuYXNhcnJheShjYWNoZS5pbWFnZXNbaSwgcywgOjoyLCA6OjQsIDo6NF0pICAgICAgICAgICMgY2hlYXAgc3Vic2FtcGxlDQogICAgICAgICAgICBpZiBmaWxsZWQ6DQogICAgICAgICAgICAgICAgbl9zbG90cyArPSAxDQogICAgICAgICAgICAgICAgdiA9IGZsb2F0KG5wLm1lYW4oY2FjaGUudmFsaWRbaSwgc10pKQ0KICAgICAgICAgICAgICAgIHZmcmFjLmFwcGVuZCh2KQ0KICAgICAgICAgICAgICAgIGlmIHN0Lm1heCgpID09IDA6DQogICAgICAgICAgICAgICAgICAgIGJhZFsnZmxhZ2dlZF9idXRfYmxhbmsnXS5hcHBlbmQoKGNhY2hlLnN0dWRpZXNbaV0sIHMpKQ0KICAgICAgICAgICAgICAgIGlmIHN0LnN0ZCgpIDwgMS4wOg0KICAgICAgICAgICAgICAgICAgICBiYWRbJ2NvbnN0YW50X3Nsb3QnXS5hcHBlbmQoKGNhY2hlLnN0dWRpZXNbaV0sIHMpKQ0KICAgICAgICAgICAgICAgIGlmIHYgPCAwLjU6DQogICAgICAgICAgICAgICAgICAgIGJhZFsnbG93X3ZhbGlkX2RlcHRoJ10uYXBwZW5kKChjYWNoZS5zdHVkaWVzW2ldLCBzKSkNCiAgICAgICAgICAgICAgICBzdGRzLmFwcGVuZChmbG9hdChzdC5zdGQoKSkpDQogICAgICAgICAgICAgICAgbWVhbnMuYXBwZW5kKGZsb2F0KHN0Lm1lYW4oKSkpDQogICAgICAgICAgICBlbGlmIHN0Lm1heCgpID4gMDoNCiAgICAgICAgICAgICAgICBiYWRbJ3VuZmxhZ2dlZF9idXRfZmlsbGVkJ10uYXBwZW5kKChjYWNoZS5zdHVkaWVzW2ldLCBzKSkNCiAgICBvdXQgPSBkaWN0KHN0dWRpZXNfY2hlY2tlZD1sZW4ocGljayksIHNsb3RzX2NoZWNrZWQ9bl9zbG90cywNCiAgICAgICAgICAgICAgIG1lYW5fdmFsaWRfZGVwdGg9ZmxvYXQobnAubWVhbih2ZnJhYykpIGlmIHZmcmFjIGVsc2UgZmxvYXQoJ25hbicpLA0KICAgICAgICAgICAgICAgc2xvdF9tZWFuX3A1X3A5NT1bZmxvYXQobnAucGVyY2VudGlsZShtZWFucywgNSkpLCBmbG9hdChucC5wZXJjZW50aWxlKG1lYW5zLCA5NSkpXSBpZiBtZWFucyBlbHNlIE5vbmUsDQogICAgICAgICAgICAgICBzbG90X3N0ZF9wNV9wOTU9W2Zsb2F0KG5wLnBlcmNlbnRpbGUoc3RkcywgNSkpLCBmbG9hdChucC5wZXJjZW50aWxlKHN0ZHMsIDk1KSldIGlmIHN0ZHMgZWxzZSBOb25lKQ0KICAgIG91dC51cGRhdGUoe2s6IGxlbih2KSBmb3IgaywgdiBpbiBiYWQuaXRlbXMoKX0pDQogICAgb3V0WydmYWlsdXJlcyddID0ge2s6IHZbOjVdIGZvciBrLCB2IGluIGJhZC5pdGVtcygpIGlmIHZ9DQogICAgcmV0dXJuIG91dA0KDQoNCmRlZiBsYXRlcmFsaXR5X3JlcG9ydChzaWRlcyk6DQogICAgaWYgbm90IHNpZGVzOg0KICAgICAgICByZXR1cm4gZGljdChzdHVkaWVzPTAsIGJ5X3NpZGU9e30sIGJ5X3NvdXJjZT17fSwgdGFnX2dlb21ldHJ5X2Rpc2FncmVlPTAsIHVucmVzb2x2ZWRfZnJhYz0wLjApDQogICAgZGYgPSBwZC5EYXRhRnJhbWUoc2lkZXMpLlQNCiAgICByZXAgPSBkaWN0KHN0dWRpZXM9bGVuKGRmKSwgYnlfc2lkZT1kZlsnc2lkZSddLnZhbHVlX2NvdW50cygpLnRvX2RpY3QoKSBpZiAnc2lkZScgaW4gZGYgZWxzZSB7fSwNCiAgICAgICAgICAgICAgIGJ5X3NvdXJjZT1kZlsnc291cmNlJ10udmFsdWVfY291bnRzKCkudG9fZGljdCgpIGlmICdzb3VyY2UnIGluIGRmIGVsc2Uge30sDQogICAgICAgICAgICAgICB0YWdfZ2VvbWV0cnlfZGlzYWdyZWU9aW50KGRmWydkaXNhZ3JlZSddLnN1bSgpKSBpZiAnZGlzYWdyZWUnIGluIGRmIGVsc2UgMCkNCiAgICByZXBbJ3VucmVzb2x2ZWRfZnJhYyddID0gZmxvYXQoKGRmWydzaWRlJ10gPT0gJ1UnKS5tZWFuKCkpIGlmICdzaWRlJyBpbiBkZiBlbHNlIDAuMA0KICAgIHJldHVybiByZXANCg==',
    'src/data/preprocess/runner.py': 'IiIiTGlicmFyeS1sZXZlbCBlbnRyeSBwb2ludHMgdXNlZCBieSB0aGUgbm90ZWJvb2s6IGluZGV4IC0+IHNsb3RzL2xhdGVyYWxpdHkgLT4gY2FjaGUgLT4gUUMuDQoNCkV2ZXJ5dGhpbmcgaXMgY2hlY2twb2ludGVkL3Jlc3VtYWJsZTsgZXZlcnkgc3RlcCBwcmludHMgdGhlIG51bWJlcnMgdGhhdCBzZXJ2ZSBhcyBhY2NlcHRhbmNlIGdhdGVzLiIiIg0KaW1wb3J0IG9zDQppbXBvcnQganNvbg0KaW1wb3J0IHRpbWUNCmltcG9ydCBzaHV0aWwNCmltcG9ydCB0ZW1wZmlsZQ0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCmZyb20gLiBpbXBvcnQgaW5kZXggYXMgcGl4DQpmcm9tIC4gaW1wb3J0IHNsb3RzIGFzIHBzbG90cw0KZnJvbSAuIGltcG9ydCBwaXBlbGluZQ0KZnJvbSAuIGltcG9ydCBjYWNoZSBhcyBwY2FjaGUNCmZyb20gLiBpbXBvcnQgcWMNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGVudmlyb25tZW50DQpkZWYgcGlja19zY3JhdGNoKGNhbmRpZGF0ZXM9Tm9uZSwgdmVyYm9zZT1UcnVlKToNCiAgICAiIiJMYXJnZXN0IHdyaXRhYmxlIHNjcmF0Y2ggZGlzayBPVVRTSURFIC9rYWdnbGUvd29ya2luZyAod2hpY2ggaXMgY2FwcGVkIGF0IH4yMCBHQiBhbmQgcGVyc2lzdGVkIGFzIG91dHB1dCkuIiIiDQogICAgY2FuZHMgPSBjYW5kaWRhdGVzIG9yIFsnL2thZ2dsZS90ZW1wJywgJy9rYWdnbGUvdG1wJywgJy90bXAnLCB0ZW1wZmlsZS5nZXR0ZW1wZGlyKCldDQogICAgYmVzdCwgdGFibGUgPSBOb25lLCBbXQ0KICAgIGZvciBjIGluIGRpY3QuZnJvbWtleXMoY2FuZHMpOg0KICAgICAgICB0cnk6DQogICAgICAgICAgICBpZiBub3Qgb3MucGF0aC5pc2RpcihjKSBvciBub3Qgb3MuYWNjZXNzKGMsIG9zLldfT0spOiAgICAgICMgb25seSBleGlzdGluZywgd3JpdGFibGUgZGlycyAobm8gc2lkZSBlZmZlY3RzKQ0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBmcmVlID0gc2h1dGlsLmRpc2tfdXNhZ2UoYykuZnJlZSAvIDFlOQ0KICAgICAgICAgICAgdGFibGUuYXBwZW5kKChjLCByb3VuZChmcmVlLCAxKSkpDQogICAgICAgICAgICBpZiBiZXN0IGlzIE5vbmUgb3IgZnJlZSA+IGJlc3RbMV06DQogICAgICAgICAgICAgICAgYmVzdCA9IChjLCBmcmVlKQ0KICAgICAgICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgaWYgdmVyYm9zZToNCiAgICAgICAgcHJpbnQoJ3NjcmF0Y2ggY2FuZGlkYXRlcyAocGF0aCwgZnJlZSBHQik6JywgdGFibGUpDQogICAgaWYgYmVzdCBpcyBOb25lOg0KICAgICAgICByYWlzZSBPU0Vycm9yKCdubyB3cml0YWJsZSBzY3JhdGNoIGRpcmVjdG9yeSBmb3VuZCcpDQogICAgcmV0dXJuIGJlc3RbMF0NCg0KDQpkZWYgZXN0aW1hdGVfY2FjaGVfZ2Iobl9zdHVkaWVzLCBjZmcpOg0KICAgIHJldHVybiBuX3N0dWRpZXMgKiBjb25maWcuTl9TTE9UUyAqIGNmZy5zdGFja19kZXB0aCAqIGNmZy5pbWdfc2l6ZSAqKiAyIC8gMWU5DQoNCg0KZGVmIGZpdF9jYWNoZV9jZmcobl9zdHVkaWVzLCBmcmVlX2diLCBwcmVzZXQ9J3YyJywgbWFyZ2luPTAuOTIpOg0KICAgICIiIkxhcmdlc3Qgc3RhbmRhcmQgKGltZ19zaXplLCBzdGFja19kZXB0aCkgdGhhdCBmaXRzIHRoZSBmcmVlIGRpc2suIFRoZSAncHVibGljJyBwcmVzZXQgaXMgZml4ZWQgKEQ9MTIsIDMzNiBweCkuIiIiDQogICAgaWYgcHJlc2V0ID09ICdwdWJsaWMnOg0KICAgICAgICByZXR1cm4gY29uZmlnLmdldF9jZmcoJ3B1YmxpYycpDQogICAgbGFkZGVyID0gWygzMzYsIDI0KSwgKDMzNiwgMjApLCAoMzIwLCAyMCksICgyODgsIDI0KSwgKDI4OCwgMjApLCAoMjU2LCAyMCksICgyNTYsIDE2KSwgKDIyNCwgMTYpXQ0KICAgIGZvciBpbWcsIGRlcHRoIGluIGxhZGRlcjoNCiAgICAgICAgY2ZnID0gY29uZmlnLmdldF9jZmcocHJlc2V0LCBpbWdfc2l6ZT1pbWcsIHN0YWNrX2RlcHRoPWRlcHRoKQ0KICAgICAgICBpZiBlc3RpbWF0ZV9jYWNoZV9nYihuX3N0dWRpZXMsIGNmZykgPD0gZnJlZV9nYiAqIG1hcmdpbjoNCiAgICAgICAgICAgIHJldHVybiBjZmcNCiAgICByZXR1cm4gY29uZmlnLmdldF9jZmcocHJlc2V0LCBpbWdfc2l6ZT0yMjQsIHN0YWNrX2RlcHRoPTEyKQ0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gMS4gaW5kZXgNCmRlZiBydW5faW5kZXgocm9vdCwgb3V0X2Rpciwgc3BsaXRzPSgndHJhaW4nLCAndGVzdCcpLCB3b3JrZXJzPU5vbmUsIGxpbWl0PTAsIHByZXNldD0ndjInLCBwcm9ncmVzcz1UcnVlKToNCiAgICAiIiJTY2FuIGV2ZXJ5IHNlcmllcyBvbiBkaXNrLCBqb2luIENTViBmbGFncywgYXNzaWduIHNsb3RzLCByZXNvbHZlIGxhdGVyYWxpdHkuIFNhdmVzIHRvIG91dF9kaXI6DQogICAgaW5kZXgucGtsLCBzbG90cy5wa2wsIHNpZGVzLmpzb24sIHNlcmllc190YWJsZS5jc3YuZ3osIHJlcG9ydC5qc29uLiAtPiBkaWN0KGFubiwgdGFiLCBzaWRlcywgcmVwb3J0KS4iIiINCiAgICBjZmcgPSBjb25maWcuZ2V0X2NmZyhwcmVzZXQpDQogICAgb3MubWFrZWRpcnMob3V0X2RpciwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICB0MCA9IHRpbWUudGltZSgpDQogICAgZGYgPSBwaXguYnVpbGRfaW5kZXgocm9vdCwgdHVwbGUoc3BsaXRzKSwgd29ya2VycywgMTAwMCwgb3MucGF0aC5qb2luKG91dF9kaXIsICdjaHVua3MnKSwgbGltaXQsIHByb2dyZXNzPXByb2dyZXNzKQ0KICAgIGRmID0gcGl4LmF0dGFjaF9jc3ZfZmxhZ3MoZGYsIHJvb3QpDQogICAgcmVwID0gcGl4LmluZGV4X3JlcG9ydChkZikNCiAgICByZXBbJ3NjYW5fbWludXRlcyddID0gcm91bmQoKHRpbWUudGltZSgpIC0gdDApIC8gNjAsIDIpDQogICAgYW5uLCB0YWIgPSBwc2xvdHMuYXNzaWduX2FsbChkZiwgY2ZnLnNsb3RfcHJlZmVyXzJkLCBjZmcuc2xvdF9mc19wcmlvcml0eSwgY2ZnLnNsb3RfY3N2X2ZhbGxiYWNrKQ0KICAgIHNpZGVzID0gcGlwZWxpbmUuc3R1ZHlfc2lkZXMoZGYpDQogICAgcmVwWydzbG90X2NvdmVyYWdlX3BjdCddID0gKHRhYi5ub3RuYSgpLm1lYW4oKSAqIDEwMCkucm91bmQoMSkudG9fZGljdCgpDQogICAgcmVwWydzbG90c19maWxsZWRfcGVyX3N0dWR5J10gPSB7aW50KGspOiBpbnQodikgZm9yIGssIHYgaW4gdGFiLm5vdG5hKCkuc3VtKGF4aXM9MSkudmFsdWVfY291bnRzKCkuc29ydF9pbmRleCgpLml0ZW1zKCl9DQogICAgcmVwWydsYXRlcmFsaXR5J10gPSBxYy5sYXRlcmFsaXR5X3JlcG9ydChzaWRlcykNCiAgICByZXBbJ2hlYWRlcl92c19jc3ZfZmF0c2F0X2NvbmZsaWN0cyddID0gaW50KGFublsnZnNfY29uZmxpY3QnXS5zdW0oKSkgaWYgJ2ZzX2NvbmZsaWN0JyBpbiBhbm4gZWxzZSAwDQogICAgcGl4LnNhdmVfaW5kZXgoYW5uLCBvcy5wYXRoLmpvaW4ob3V0X2RpciwgJ2luZGV4LnBrbCcpKQ0KICAgIHBkLnRvX3BpY2tsZSh0YWIsIG9zLnBhdGguam9pbihvdXRfZGlyLCAnc2xvdHMucGtsJykpDQogICAganNvbi5kdW1wKHNpZGVzLCBvcGVuKG9zLnBhdGguam9pbihvdXRfZGlyLCAnc2lkZXMuanNvbicpLCAndycpKQ0KICAgIGFubi5kcm9wKGNvbHVtbnM9WydvcmRlcmVkX2ZpbGVzJywgJ3Bvc2l0aW9ucyddLCBlcnJvcnM9J2lnbm9yZScpLnRvX2Nzdihvcy5wYXRoLmpvaW4ob3V0X2RpciwgJ3Nlcmllc190YWJsZS5jc3YuZ3onKSwgaW5kZXg9RmFsc2UpDQogICAganNvbi5kdW1wKHJlcCwgb3Blbihvcy5wYXRoLmpvaW4ob3V0X2RpciwgJ3JlcG9ydC5qc29uJyksICd3JyksIGluZGVudD0xLCBkZWZhdWx0PXN0cikNCiAgICBsYXQgPSByZXBbJ2xhdGVyYWxpdHknXQ0KICAgIHByaW50KGpzb24uZHVtcHMocmVwLCBpbmRlbnQ9MSwgZGVmYXVsdD1zdHIpKQ0KICAgIHByaW50KCdcbkdBVEVTIChpbmRleCk6JykNCiAgICBwcmludChmIiAgc2VyaWVzIHdpdGggcmVhZCBlcnJvcnMgICAgICAgICAgOiB7cmVwWydzZXJpZXNfd2l0aF9lcnJvciddfSAgKGV4cGVjdCB+MCkiKQ0KICAgIHByaW50KGYiICBub24tY2Fub25pY2FsIG9yaWVudGF0aW9uIHNlcmllcyA6IHtyZXAuZ2V0KCdub25fY2Fub25pY2FsX29yaWVudGF0aW9uJyl9ICAoZXhwZWN0IDApIikNCiAgICBwcmludChmIiAgQ1NWIHBsYW5lIHZzIERJQ09NIGdlb21ldHJ5ICAgICAgOiB7cmVwLmdldCgnY3N2X3BsYW5lX3ZzX2dlb21ldHJ5X21pc21hdGNoJyl9IG1pc21hdGNoZXMgKGV4cGVjdCAwKSIpDQogICAgcHJpbnQoZiIgIGxhdGVyYWxpdHkgdW5yZXNvbHZlZCAgICAgICAgICAgIDoge2xhdFsndW5yZXNvbHZlZF9mcmFjJ106LjIlfSAgKGdhdGUgPCAxJSkiKQ0KICAgIHByaW50KGYiICBsYXRlcmFsaXR5IHRhZy12cy1nZW9tZXRyeSBjbGFzaCA6IHtsYXRbJ3RhZ19nZW9tZXRyeV9kaXNhZ3JlZSddfSBzdHVkaWVzICAoZ2F0ZSA8IDElIG9mIHRhZ2dlZCkiKQ0KICAgIHJldHVybiBkaWN0KGFubj1hbm4sIHRhYj10YWIsIHNpZGVzPXNpZGVzLCByZXBvcnQ9cmVwKQ0KDQoNCmRlZiBnZXRfaW5kZXgocm9vdCwgb3V0X2Rpciwgc3BsaXRzPSgndHJhaW4nLCAndGVzdCcpLCB3b3JrZXJzPU5vbmUsIGZvcmNlPUZhbHNlLCAqKmt3KToNCiAgICAiIiJMb2FkIHRoZSBzYXZlZCBpbmRleCBpZiBwcmVzZW50IChyZXN0YXJ0LWZyaWVuZGx5KSwgZWxzZSBidWlsZCBpdC4gLT4gZGljdChhbm4sIHRhYiwgc2lkZXNbLCByZXBvcnRdKS4iIiINCiAgICBpZiBub3QgZm9yY2UgYW5kIG9zLnBhdGguZXhpc3RzKG9zLnBhdGguam9pbihvdXRfZGlyLCAnaW5kZXgucGtsJykpIGFuZCBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4ob3V0X2RpciwgJ3Nsb3RzLnBrbCcpKToNCiAgICAgICAgYW5uLCB0YWIsIHNpZGVzID0gbG9hZF9pbmRleF9kaXIob3V0X2RpcikNCiAgICAgICAgcHJpbnQoZidsb2FkZWQgZXhpc3RpbmcgaW5kZXg6IHtsZW4oYW5uKTosfSBzZXJpZXMsIHtsZW4odGFiKTosfSBzdHVkaWVzICAoe291dF9kaXJ9KScpDQogICAgICAgIHJldHVybiBkaWN0KGFubj1hbm4sIHRhYj10YWIsIHNpZGVzPXNpZGVzLCByZXBvcnQ9anNvbi5sb2FkKG9wZW4ob3MucGF0aC5qb2luKG91dF9kaXIsICdyZXBvcnQuanNvbicpKSkpDQogICAgcmV0dXJuIHJ1bl9pbmRleChyb290LCBvdXRfZGlyLCBzcGxpdHMsIHdvcmtlcnMsICoqa3cpDQoNCg0KZGVmIGxvYWRfaW5kZXhfZGlyKG91dF9kaXIpOg0KICAgIHJldHVybiAocGl4LmxvYWRfaW5kZXgob3MucGF0aC5qb2luKG91dF9kaXIsICdpbmRleC5wa2wnKSksIHBkLnJlYWRfcGlja2xlKG9zLnBhdGguam9pbihvdXRfZGlyLCAnc2xvdHMucGtsJykpLA0KICAgICAgICAgICAganNvbi5sb2FkKG9wZW4ob3MucGF0aC5qb2luKG91dF9kaXIsICdzaWRlcy5qc29uJykpKSkNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIDIuIGNhY2hlDQpkZWYgcnVuX2NhY2hlKGluZGV4LCBzcGxpdCwgcHJlZml4LCBwcmVzZXQ9J3YyJywgd29ya2Vycz1Ob25lLCBsaW1pdF9zdHVkaWVzPTAsIGZyZXNoPUZhbHNlLCBjZmc9Tm9uZSwgc3R1ZGllcz1Ob25lLA0KICAgICAgICAgICAgICAqKmNmZ19vdmVycmlkZXMpOg0KICAgICIiIkJ1aWxkL3Jlc3VtZSB0aGUgbWVtbWFwIGNhY2hlIGZvciBhIHNwbGl0LiBgaW5kZXhgIGlzIHRoZSBydW5faW5kZXgoKSBkaWN0IG9yIGFuIGluZGV4IGRpcmVjdG9yeS4NCiAgICAtPiAoY2FjaGUsIHN0YXRzKS4iIiINCiAgICBhbm4sIHRhYiwgc2lkZXMgPSAoaW5kZXhbJ2FubiddLCBpbmRleFsndGFiJ10sIGluZGV4WydzaWRlcyddKSBpZiBpc2luc3RhbmNlKGluZGV4LCBkaWN0KSBlbHNlIGxvYWRfaW5kZXhfZGlyKGluZGV4KQ0KICAgIGNmZyA9IGNmZyBvciBjb25maWcuZ2V0X2NmZyhwcmVzZXQsICoqY2ZnX292ZXJyaWRlcykNCiAgICBpZiBzdHVkaWVzIGlzIE5vbmU6DQogICAgICAgIHN0dWRpZXMgPSBzb3J0ZWQoYW5uW2Fublsnc3BsaXQnXSA9PSBzcGxpdF1bJ1N0dWR5SW5zdGFuY2VVSUQnXS51bmlxdWUoKSkNCiAgICBpZiBsaW1pdF9zdHVkaWVzOg0KICAgICAgICBzdHVkaWVzID0gc3R1ZGllc1s6bGltaXRfc3R1ZGllc10NCiAgICBuZWVkID0gZXN0aW1hdGVfY2FjaGVfZ2IobGVuKHN0dWRpZXMpLCBjZmcpDQogICAgcHJpbnQoZid7bGVuKHN0dWRpZXMpfSBzdHVkaWVzIC0+IGNhY2hlIHtuZWVkOi4xZn0gR0IgIChwcmVzZXQ9e2NmZy5uYW1lfSwgRD17Y2ZnLnN0YWNrX2RlcHRofSwge2NmZy5pbWdfc2l6ZX1weCkgYXQge3ByZWZpeH0nKQ0KICAgIHN1YiA9IGFublthbm5bJ1N0dWR5SW5zdGFuY2VVSUQnXS5pc2luKHNldChzdHVkaWVzKSldIGlmICgnU3R1ZHlJbnN0YW5jZVVJRCcgaW4gYW5uLmNvbHVtbnMpIGVsc2UgYW5uDQogICAgcmVjb3JkcyA9IHBpcGVsaW5lLmluZGV4X3RvX3JlY29yZHMoc3ViKQ0KICAgIHNsb3Rfcm93cyA9IHtzOiB0YWIubG9jW3NdLnRvX2RpY3QoKSBmb3IgcyBpbiBzdHVkaWVzIGlmIHMgaW4gdGFiLmluZGV4fQ0KICAgIGNhY2hlLCBzdGF0cyA9IHBjYWNoZS5idWlsZF9jYWNoZShwcmVmaXgsIHN0dWRpZXMsIHNsb3Rfcm93cywgcmVjb3Jkcywgc2lkZXMsIGNmZywgd29ya2VycywgcmVzdW1lPW5vdCBmcmVzaCkNCiAgICBwcmludChqc29uLmR1bXBzKHN0YXRzLCBpbmRlbnQ9MSkpDQogICAgcmV0dXJuIGNhY2hlLCBzdGF0cw0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gMy4gUUMNCmRlZiBydW5fcWMocHJlZml4LCBvdXRfZGlyPU5vbmUsIG49MzAwLCBtb250YWdlPVRydWUpOg0KICAgICIiIkFjY2VwdGFuY2UgZ2F0ZXMgb24gYSBidWlsdCBjYWNoZSAoKyBvcHRpb25hbCBtb250YWdlIFBORzogb25lIHJvdyBwZXIgc2xvdCB0eXBlKS4iIiINCiAgICBjID0gcGNhY2hlLlN0dWR5Q2FjaGUocHJlZml4KQ0KICAgIHJlcCA9IHFjLmNhY2hlX3Nhbml0eShjLCBuKQ0KICAgIHByaW50KGpzb24uZHVtcHMocmVwLCBpbmRlbnQ9MSwgZGVmYXVsdD1zdHIpKQ0KICAgIGJhZCA9IHN1bShyZXAuZ2V0KGssIDApIGZvciBrIGluICgnY29uc3RhbnRfc2xvdCcsICdmbGFnZ2VkX2J1dF9ibGFuaycsICd1bmZsYWdnZWRfYnV0X2ZpbGxlZCcpKQ0KICAgIHByaW50KGYiXG5HQVRFUyAoY2FjaGUpOiBjb25zdGFudC9ibGFuay91bmZsYWdnZWQgc2xvdHMgPSB7YmFkfSAoZXhwZWN0IDApOyAiDQogICAgICAgICAgZiJtZWFuIHZhbGlkIGRlcHRoID0ge3JlcC5nZXQoJ21lYW5fdmFsaWRfZGVwdGgnLCBmbG9hdCgnbmFuJykpOi4yZn0gKGV4cGVjdCA+IDAuOCkiKQ0KICAgIGlmIG91dF9kaXI6DQogICAgICAgIG9zLm1ha2VkaXJzKG91dF9kaXIsIGV4aXN0X29rPVRydWUpDQogICAgICAgIGpzb24uZHVtcChyZXAsIG9wZW4ob3MucGF0aC5qb2luKG91dF9kaXIsICdjYWNoZV9zYW5pdHkuanNvbicpLCAndycpLCBpbmRlbnQ9MSwgZGVmYXVsdD1zdHIpDQogICAgaWYgbW9udGFnZSBhbmQgb3V0X2RpcjoNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgaW1wb3J0IG1hdHBsb3RsaWINCiAgICAgICAgICAgIG1hdHBsb3RsaWIudXNlKCdBZ2cnKQ0KICAgICAgICAgICAgaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdA0KICAgICAgICAgICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApDQogICAgICAgICAgICBkb25lID0gbnAud2hlcmUobnAuYXNhcnJheShjLmRvbmUpID09IDEpWzBdDQogICAgICAgICAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGNvbmZpZy5OX1NMT1RTLCA2LCBmaWdzaXplPSgxNCwgMi40ICogY29uZmlnLk5fU0xPVFMpKQ0KICAgICAgICAgICAgZm9yIHMgaW4gcmFuZ2UoY29uZmlnLk5fU0xPVFMpOg0KICAgICAgICAgICAgICAgIGNhbmQgPSBbaSBmb3IgaSBpbiBkb25lIGlmIGMuc2xvdFtpLCBzXV0NCiAgICAgICAgICAgICAgICBwaWNrID0gcm5nLmNob2ljZShjYW5kLCBtaW4oNiwgbGVuKGNhbmQpKSwgcmVwbGFjZT1GYWxzZSkgaWYgY2FuZCBlbHNlIFtdDQogICAgICAgICAgICAgICAgZm9yIGogaW4gcmFuZ2UoNik6DQogICAgICAgICAgICAgICAgICAgIGF4W3MsIGpdLmF4aXMoJ29mZicpDQogICAgICAgICAgICAgICAgZm9yIGosIGkgaW4gZW51bWVyYXRlKHBpY2spOg0KICAgICAgICAgICAgICAgICAgICBheFtzLCBqXS5pbXNob3coYy5pbWFnZXNbaSwgcywgYy5pbWFnZXMuc2hhcGVbMl0gLy8gMl0sIGNtYXA9J2dyYXknLCB2bWluPTAsIHZtYXg9MjU1KQ0KICAgICAgICAgICAgICAgICAgICBheFtzLCBqXS5zZXRfdGl0bGUoY29uZmlnLlNMT1RTW3NdWzBdLCBmb250c2l6ZT03KQ0KICAgICAgICAgICAgcGx0LnRpZ2h0X2xheW91dCgpDQogICAgICAgICAgICBwYXRoID0gb3MucGF0aC5qb2luKG91dF9kaXIsICdtb250YWdlLnBuZycpDQogICAgICAgICAgICBwbHQuc2F2ZWZpZyhwYXRoLCBkcGk9OTApDQogICAgICAgICAgICBwbHQuY2xvc2UoJ2FsbCcpDQogICAgICAgICAgICBwcmludCgnbW9udGFnZSAtPicsIHBhdGgpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGEgcGxvdHRpbmcgcHJvYmxlbSBtdXN0IG5ldmVyIGZhaWwgdGhlIHBpcGVsaW5lDQogICAgICAgICAgICBwcmludCgnbW9udGFnZSBza2lwcGVkOicsIHR5cGUoZSkuX19uYW1lX18sIGUpDQogICAgcmV0dXJuIHJlcA0K',
    'src/data/preprocess/sampling.py': 'IiIiV2luZG93IHNlbGVjdGlvbiArIGF1Z21lbnRhdGlvbiAodGhlIHRyYWluaW5nIGhvdCBsb29wOyBudW1weS9PcGVuQ1Ygb25seSkuDQoNClRocm91Z2hwdXQgcnVsZXM6DQogICogY2hvb3NlIHRoZSB3aW5kb3dzIEZJUlNULCB0aGVuIGF1Z21lbnQgb25seSB0aGUgc2xpY2VzIHRob3NlIHdpbmRvd3MgdXNlICh0cmFpbmluZyB1c2VzIH40IG9mIDExIHdpbmRvd3MsDQogICAgaS5lLiB+MTIgb2YgdGhlIDI0IHN0b3JlZCBzbGljZXMgLT4gfjJ4IGxlc3Mgd2FycGluZykNCiAgKiBvbmUgYWZmaW5lIG1hdHJpeCBhbmQgb25lIDI1Ni1lbnRyeSBMVVQgcGVyIHNsb3QsIGFwcGxpZWQgd2l0aCBjdjIgKFNJTUQpIHRvIGFsbCBvZiBpdHMgc2xpY2VzDQogICogbm8gZmxpcHM6IGtuZWVzIGFyZSBjYW5vbmljYWxpc2VkIHRvICdsZWZ0JyBhdCBwcmVwcm9jZXNzaW5nIHRpbWUsIHNvIGEgaG9yaXpvbnRhbCBmbGlwIHdvdWxkIGRlc3Ryb3kgdGhlDQogICAgbWVkaWFsL2xhdGVyYWwgaW1hZ2Utc2lkZSBjb25zaXN0ZW5jeSB0aGF0IHRoZSBzbG90IGhlYWQgcmVsaWVzIG9uDQoiIiINCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IGN2Mg0KDQpjdjIuc2V0TnVtVGhyZWFkcygwKQ0KDQoNCmRlZiBuX3dpbmRvd3MoZGVwdGgsIGdyb3VwLCBzdHJpZGUpOg0KICAgIHJldHVybiBtYXgoMSwgKGRlcHRoIC0gZ3JvdXApIC8vIHN0cmlkZSArIDEpDQoNCg0KZGVmIHdpbmRvd19zdGFydHMoZGVwdGgsIGdyb3VwLCBzdHJpZGUsIG5fdXNlPU5vbmUsIHRyYWluPUZhbHNlLCBybmc9Tm9uZSk6DQogICAgIiIiU3RhcnQgaW5kaWNlcyAoaW50byB0aGUgc3RvcmVkIHN0YWNrKSBvZiB0aGUgd2luZG93cyB0byB1c2UuDQoNCiAgICBldmFsIDogZXZlbmx5IHNwYWNlZCBzdWJzZXQgKG9yIGFsbCkgICAgICAgICAgIHRyYWluOiBvbmUgcmFuZG9tIHdpbmRvdyBwZXIgZXF1YWwtd2lkdGggYmluIChzdHJhdGlmaWVkLCBzbw0KICAgIHRoZSB3aG9sZSBkZXB0aCBpcyBjb3ZlcmVkIGV2ZXJ5IHN0ZXApIHdpdGggYSByYW5kb20gcGhhc2UuIiIiDQogICAgVyA9IG5fd2luZG93cyhkZXB0aCwgZ3JvdXAsIHN0cmlkZSkNCiAgICBzdGFydHMgPSBucC5hcmFuZ2UoVykgKiBzdHJpZGUNCiAgICBpZiBuX3VzZSBpcyBOb25lIG9yIG5fdXNlID49IFc6DQogICAgICAgIHJldHVybiBzdGFydHMNCiAgICBpZiB0cmFpbjoNCiAgICAgICAgcm5nID0gcm5nIG9yIG5wLnJhbmRvbS5kZWZhdWx0X3JuZygpDQogICAgICAgICMgVVBHUkFERSAzOiBaLWF4aXMgaml0dGVyICh0aHJvdWdoLXBsYW5lIHRyYW5zbGF0aW9uIGF1Z21lbnRhdGlvbikuDQogICAgICAgICMgU2hpZnRzIHRoZSBlbnRpcmUgd2luZG93IGdyaWQgYnkgYSByYW5kb20gb2Zmc2V0IGJlZm9yZSBiaW5uaW5nLg0KICAgICAgICAjIEZvcmNlcyB0aGUgbW9kZWwgdG8gbGVhcm4gYW5hdG9teSByYXRoZXIgdGhhbiBtZW1vcml6aW5nIHNsaWNlIHBvc2l0aW9ucy4NCiAgICAgICAgel9qaXR0ZXIgPSBpbnQocm5nLmludGVnZXJzKDAsIG1heCgxLCBzdHJpZGUgLy8gMiArIDEpKSkNCiAgICAgICAgaml0dGVyZWQgPSBucC5jbGlwKHN0YXJ0cyArIHpfaml0dGVyLCAwLCBkZXB0aCAtIGdyb3VwKQ0KICAgICAgICBlZGdlcyA9IG5wLmxpbnNwYWNlKDAsIFcsIG5fdXNlICsgMSkNCiAgICAgICAgcGljayA9IFtpbnQocm5nLmludGVnZXJzKGludChucC5mbG9vcihlZGdlc1tpXSkpLCBtYXgoaW50KG5wLmZsb29yKGVkZ2VzW2ldKSkgKyAxLCBpbnQobnAuY2VpbChlZGdlc1tpICsgMV0pKSkpKSBmb3IgaSBpbiByYW5nZShuX3VzZSldDQogICAgICAgIHJldHVybiBqaXR0ZXJlZFtucC5jbGlwKHBpY2ssIDAsIFcgLSAxKV0NCiAgICByZXR1cm4gc3RhcnRzW25wLnVuaXF1ZShucC5yaW50KG5wLmxpbnNwYWNlKDAsIFcgLSAxLCBuX3VzZSkpLmFzdHlwZShpbnQpKV0NCg0KDQpkZWYgZ2F0aGVyX3dpbmRvd3Moc3RhY2ssIHZhbGlkLCBzdGFydHMsIGdyb3VwKToNCiAgICAiIiJzdGFjayBbRCxILFddIChtYXkgYmUgYSBtZW1tYXAgdmlldykgLT4gd2luZG93cyBbbixncm91cCxILFddIHVpbnQ4IGFuZCB3aW5kb3cgdmFsaWRpdHkgW25dLg0KDQogICAgQSB3aW5kb3cgaXMgdmFsaWQgb25seSBpZiBhbGwgb2YgaXRzIGNoYW5uZWxzIGFyZSB2YWxpZCBzbGljZXMuDQoNCiAgICBDT05USUdVT1VTIE1FTU1BUCBSRUFEIE9QVElNSVpBVElPTjoNCiAgICBJbnN0ZWFkIG9mIG5vbi1jb250aWd1b3VzIGluZGV4aW5nIHN0YWNrW2lkeF0gdGhhdCB0cmlnZ2VycyBzY2F0dGVyZWQgZGlzayBwYWdlIGZhdWx0cw0KICAgIGFjcm9zcyB0aGUgMjI3IEdCIG1lbW1hcCBmaWxlLCByZWFkIHRoZSBib3VuZGluZyBzbGljZSBzdGFja1tsbzpoaV0gYXMgYSBzaW5nbGUgY29udGlndW91cw0KICAgIGJsb2NrIGludG8gUkFNLCB0aGVuIGV4dHJhY3Qgd2luZG93cyBsb2NhbGx5IGF0IG1lbW9yeSBidXMgc3BlZWRzICg+MTAwIEdCL3MpLg0KICAgICIiIg0KICAgIGlmIGxlbihzdGFydHMpID09IDA6DQogICAgICAgIHJldHVybiBucC56ZXJvcygoMCwgZ3JvdXAsIHN0YWNrLnNoYXBlWzFdLCBzdGFjay5zaGFwZVsyXSksIGR0eXBlPXN0YWNrLmR0eXBlKSwgbnAuemVyb3MoMCwgZHR5cGU9Ym9vbCkNCiAgICBpZHggPSBucC5hc2FycmF5KHN0YXJ0cylbOiwgTm9uZV0gKyBucC5hcmFuZ2UoZ3JvdXApW05vbmUsIDpdDQogICAgbG8gPSBpbnQoaWR4Lm1pbigpKQ0KICAgIGhpID0gaW50KGlkeC5tYXgoKSkgKyAxDQogICAgYmxvY2sgPSBucC5hc2NvbnRpZ3VvdXNhcnJheShzdGFja1tsbzpoaV0pDQogICAgd2luID0gYmxvY2tbaWR4IC0gbG9dDQogICAgd3YgPSBucC5hc2FycmF5KHZhbGlkKVtpZHhdLmFsbChheGlzPTEpDQogICAgcmV0dXJuIHdpbiwgd3YNCg0KDQpkZWYgX2x1dChybmcsIGEpOg0KICAgIGMgPSAxLjAgKyBybmcudW5pZm9ybSgtYSwgYSkNCiAgICBiID0gcm5nLnVuaWZvcm0oLWEgLyAyLjAsIGEgLyAyLjApDQogICAgZyA9IDEuMCArIHJuZy51bmlmb3JtKC1hLCBhKQ0KICAgIHggPSBucC5hcmFuZ2UoMjU2LCBkdHlwZT1ucC5mbG9hdDMyKSAvIDI1NS4wDQogICAgeSA9IG5wLmNsaXAoKG5wLnBvd2VyKHgsIGcpIC0gMC41KSAqIGMgKyAwLjUgKyBiLCAwLjAsIDEuMCkNCiAgICByZXR1cm4gbnAucmludCh5ICogMjU1LjApLmFzdHlwZShucC51aW50OCkNCg0KDQpkZWYgYXVnbWVudF9zbG90KHdpbiwgcm5nLCByb3RfZGVnPTguMCwgc2NhbGU9MC4wOCwgc2hpZnQ9MC4wNSwgaW50ZW5zaXR5PTAuMSk6DQogICAgIiIid2luIFtuLGcsSCxXXSB1aW50OCAtPiBzYW1lIHNoYXBlLiBPbmUgcmFuZG9tIHJvdGF0aW9uL3NjYWxlL3NoaWZ0ICsgb25lIGludGVuc2l0eSBMVVQgZm9yIHRoZSB3aG9sZSBzbG90LiIiIg0KICAgIG4sIGcsIEgsIFcgPSB3aW4uc2hhcGUNCiAgICBhbmcgPSBybmcudW5pZm9ybSgtcm90X2RlZywgcm90X2RlZykNCiAgICBzYyA9IDEuMCArIHJuZy51bmlmb3JtKC1zY2FsZSwgc2NhbGUpDQogICAgTSA9IGN2Mi5nZXRSb3RhdGlvbk1hdHJpeDJEKCgoVyAtIDEpIC8gMi4wLCAoSCAtIDEpIC8gMi4wKSwgYW5nLCBzYykNCiAgICBNWzAsIDJdICs9IHJuZy51bmlmb3JtKC1zaGlmdCwgc2hpZnQpICogVw0KICAgIE1bMSwgMl0gKz0gcm5nLnVuaWZvcm0oLXNoaWZ0LCBzaGlmdCkgKiBIDQogICAgbHV0ID0gX2x1dChybmcsIGludGVuc2l0eSkgaWYgaW50ZW5zaXR5ID4gMCBlbHNlIE5vbmUNCiAgICBvdXQgPSBucC5lbXB0eV9saWtlKHdpbikNCiAgICBmb3IgaSBpbiByYW5nZShuKToNCiAgICAgICAgZm9yIGogaW4gcmFuZ2UoZyk6DQogICAgICAgICAgICB4ID0gY3YyLndhcnBBZmZpbmUod2luW2ksIGpdLCBNLCAoVywgSCksIGZsYWdzPWN2Mi5JTlRFUl9MSU5FQVIsIGJvcmRlck1vZGU9Y3YyLkJPUkRFUl9SRUZMRUNUXzEwMSkNCiAgICAgICAgICAgIG91dFtpLCBqXSA9IGN2Mi5MVVQoeCwgbHV0KSBpZiBsdXQgaXMgbm90IE5vbmUgZWxzZSB4DQogICAgcmV0dXJuIG91dA0K',
    'src/data/preprocess/slots.py': 'IiIiU2VyaWVzIC0+IGFjcXVpc2l0aW9uLXNsb3QgYXNzaWdubWVudC4NCg0KQW5ub3RhdGlvbiBjb25zdGFudHMgYXJlIHRoZSBwdWJsaWMgMC45NDMgbm90ZWJvb2sncyAocmVncmVzc2lvbi10ZXN0ZWQgYWdhaW5zdCBpdCBvbiBhbGwgNCw0MDcgdHJhaW4gc3R1ZGllczoNClNBRy9DT1IvQVggZmx1aWQtRlMgc2xvdHMgZmlsbGVkIGZvciA5NC4zIC8gOTUuNSAvIDk5LjQgJSBvZiBzdHVkaWVzKS4NCg0KV2hhdCB0aGUgYXVkaXQgYWRkZWQ6DQogICogdGhlIENTViBgRmF0X1N1cHByZXNzaW9uYCBmbGFnIGFncmVlcyB3aXRoIHRoZSBoZWFkZXItZGVyaXZlZCBmYXQtc2F0IGV2aWRlbmNlIGZvciA5OS4xNCUgb2Ygc2VyaWVzLCBhbmQNCiAgICBgRmx1aWRfU2Vuc2l0aXZlYCBpcyBJREVOVElDQUwgdG8gaXQgaW4gdHJhaW4gKGl0IHRyYWNrcyBmYXQtc3VwcHJlc3Npb24sIG5vdCBQRC9UMiBjb250cmFzdCksIHNvIHRoZSBDU1YgZmx1aWQNCiAgICBmbGFnIGlzIG9ubHkgdXNlZCBhcyBhIGxhc3QtcmVzb3J0IHByb3h5Ow0KICAqIHRoZSBtZXRhZGF0YS1wb29yIGNvaG9ydCAoMjM4IHN0dWRpZXMsIG5vIFRSL1RFL1NjYW5PcHRpb25zKSBzdGlsbCBoYXMgZGVzY3JpcHRpb25zLCBzbyBoZWFkZXIgcnVsZXMgd29yazsNCiAgKiBoaWRkZW4tc2V0IHJvYnVzdG5lc3M6IGV2ZXJ5IG1pc3NpbmcgaW5wdXQgZGVncmFkZXMgdG8gYSB3ZWFrZXIgc2lnbmFsIGluc3RlYWQgb2YgYW4gZXhjZXB0aW9uOw0KICAqIDMuMCUgb2YgJ21vc3Qgc2xpY2VzJyBwaWNrcyBhcmUgMy1EIHZvbHVtZXMgLT4gYHByZWZlcl8yZGAgcGlja3MgYSAyLUQgc2VyaWVzIHdoZW5ldmVyIG9uZSBleGlzdHMuDQoiIiINCmltcG9ydCByZQ0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmltcG9ydCBzcmMuY29yZS5jb25maWcgYXMgY29uZmlnDQoNCkZBVFNBVF9PUFRTID0geydGUycsICdGQVRTQVQnLCAnRkFUX1NBVCcsICdGU0FUJ30NCl9GQVRTQVRfUlggPSByZS5jb21waWxlKCdcXGJmc1xcYnxmYXRzYXR8ZmF0IHNhdHxcXGJzdGlyXFxifFxcYnNwYWlyXFxifFxcYnNwaXJcXGJ8XFxid2VcXGJ8d2F0ZXIgZXhjaXR8XFxidGlybVxcYnwnDQogICAgICAgICAgICAgICAgICAgICAgICAnXFxic3RpbmdcXGJ8XFxiZmF0c3VwXFxifHNtYXJ0IGZhdHxcXGJ3YXRlclxcYicpDQpfVDFfUlggPSByZS5jb21waWxlKCdcXGJ0MVxcYnxcXGJ0MXdcXGInKQ0KX1QyX1JYID0gcmUuY29tcGlsZSgnXFxidDJcXGJ8XFxidDJ3XFxiJykNCl9QRF9SWCA9IHJlLmNvbXBpbGUoJ1xcYnBkXFxifFxcYnBkd1xcYnxwcm90b258XFxiZHBcXGJ8ZGVucycpDQpfU0VQID0gcmUuY29tcGlsZSgnW19cXC0uXScpDQoNCk5FRURFRCA9IFsnU2VyaWVzRGVzY3JpcHRpb24nLCAnU2VxdWVuY2VOYW1lJywgJ1NjYW5PcHRpb25zJywgJ1NjYW5uaW5nU2VxdWVuY2UnLCAnUmVwZXRpdGlvblRpbWUnLCAnRWNob1RpbWUnXQ0KDQoNCmRlZiB0b19mbGFnKHgpOg0KICAgICIiIlJvYnVzdCAwLzEvTmFOIGZyb20gQ1NWIHZhbHVlcyAoaGlkZGVuLXNldCBDU1YgZm9ybWF0dGluZyBpcyBub3QgZ3VhcmFudGVlZCkuIiIiDQogICAgaWYgeCBpcyBOb25lOg0KICAgICAgICByZXR1cm4gbnAubmFuDQogICAgaWYgaXNpbnN0YW5jZSh4LCAoYm9vbCwgbnAuYm9vbF8pKToNCiAgICAgICAgcmV0dXJuIGZsb2F0KGJvb2woeCkpDQogICAgaWYgaXNpbnN0YW5jZSh4LCAoaW50LCBmbG9hdCwgbnAuaW50ZWdlciwgbnAuZmxvYXRpbmcpKToNCiAgICAgICAgcmV0dXJuIGZsb2F0KHgpIGlmIHggaW4gKDAsIDEpIGVsc2UgbnAubmFuDQogICAgcyA9IHN0cih4KS5zdHJpcCgpLmxvd2VyKCkNCiAgICBpZiBzIGluICgnMScsICd0cnVlJywgJ3llcycsICd5JywgJ3QnLCAnMS4wJyk6DQogICAgICAgIHJldHVybiAxLjANCiAgICBpZiBzIGluICgnMCcsICdmYWxzZScsICdubycsICduJywgJ2YnLCAnMC4wJyk6DQogICAgICAgIHJldHVybiAwLjANCiAgICByZXR1cm4gbnAubmFuDQoNCg0KZGVmIGFubm90YXRlKGRmLCBmc19wcmlvcml0eT0naGRyJywgY3N2X2ZhbGxiYWNrPVRydWUpOg0KICAgICIiIkFkZCBmYXRzYXQgLyB3ZWlnaHQgLyBmbHVpZCAvIGluZm9ybWF0aXZlIC8gaXMzZCBjb2x1bW5zLiBSZXR1cm5zIGEgY29weS4NCg0KICAgIGZhdHNhdDogaGVhZGVyIGV2aWRlbmNlIChkZXNjcmlwdGlvbiByZWdleCBvciBTY2FuT3B0aW9ucyB0b2tlbikgYXMgaW4gdGhlIHB1YmxpYyBwaXBlbGluZS4gV2hlbiB0aGUgaGVhZGVyDQogICAgaGFzIG5vIHVzYWJsZSB0ZXh0IGF0IGFsbCAoZHVtbXkvbWlzc2luZyBkZXNjcmlwdGlvbiBBTkQgbm8gU2Nhbk9wdGlvbnMpIHRoZSBDU1YgZmxhZyBpcyB1c2VkIGluc3RlYWQ7IHdpdGgNCiAgICBmc19wcmlvcml0eT0nY3N2JyB0aGUgQ1NWIGZsYWcgd2lucyB3aGVuZXZlciBwcmVzZW50Lg0KICAgICIiIg0KICAgIGRmID0gZGYuY29weSgpDQogICAgaWYgJ3BsYW5lJyBub3QgaW4gZGYuY29sdW1uczoNCiAgICAgICAgZGZbJ3BsYW5lJ10gPSBwbGFuZV9vZihkZikNCiAgICBmb3IgYyBpbiBORUVERUQ6DQogICAgICAgIGlmIGMgbm90IGluIGRmLmNvbHVtbnM6DQogICAgICAgICAgICBkZltjXSA9IE5vbmUNCiAgICByYXcgPSBkZlsnU2VyaWVzRGVzY3JpcHRpb24nXS5maWxsbmEoJycpLmFzdHlwZShzdHIpDQogICAgZHVtbXkgPSByYXcuc3RyLmNvbnRhaW5zKCdkdW1teXNlcmllc2Rlc2MnLCBjYXNlPUZhbHNlKQ0KICAgIGRlc2MgPSAocmF3LndoZXJlKH5kdW1teSwgJycpICsgJyAnICsgZGZbJ1NlcXVlbmNlTmFtZSddLmZpbGxuYSgnJykuYXN0eXBlKHN0cikpLnN0ci5sb3dlcigpDQogICAgZGVzYyA9IGRlc2Muc3RyLnJlcGxhY2UoX1NFUCwgJyAnLCByZWdleD1UcnVlKQ0KICAgIG9wdHMgPSBkZlsnU2Nhbk9wdGlvbnMnXS5maWxsbmEoJycpLmFzdHlwZShzdHIpLnN0ci51cHBlcigpLnN0ci5zcGxpdCgnfCcpDQogICAgb3B0c19mcyA9IG9wdHMuYXBwbHkobGFtYmRhIHRzOiBhbnkodC5zdHJpcCgpIGluIEZBVFNBVF9PUFRTIGZvciB0IGluIHRzKSkNCiAgICBmc19oZHIgPSBkZXNjLnN0ci5jb250YWlucyhfRkFUU0FUX1JYKSB8IG9wdHNfZnMNCiAgICBpbmZvcm1hdGl2ZSA9IChkZXNjLnN0ci5zdHJpcCgpICE9ICcnKSB8IChkZlsnU2Nhbk9wdGlvbnMnXS5maWxsbmEoJycpLmFzdHlwZShzdHIpLnN0ci5zdHJpcCgpICE9ICcnKQ0KDQogICAgY3N2X2ZzID0gcGQuU2VyaWVzKG5wLm5hbiwgaW5kZXg9ZGYuaW5kZXgpDQogICAgaWYgJ0ZhdF9TdXBwcmVzc2lvbicgaW4gZGYuY29sdW1uczoNCiAgICAgICAgY3N2X2ZzID0gZGZbJ0ZhdF9TdXBwcmVzc2lvbiddLm1hcCh0b19mbGFnKQ0KICAgIGVsaWYgJ0ZsdWlkX1NlbnNpdGl2ZScgaW4gZGYuY29sdW1uczoNCiAgICAgICAgIyBGSVg6IFRlc3Qgc2V0IG1pZ2h0IGRyb3AgRmF0X1N1cHByZXNzaW9uOyB1c2UgRmx1aWRfU2Vuc2l0aXZlIGFzIGlkZW50aWNhbCBwcm94eSAocGVyIEVEQSkNCiAgICAgICAgY3N2X2ZzID0gZGZbJ0ZsdWlkX1NlbnNpdGl2ZSddLm1hcCh0b19mbGFnKQ0KICAgIGlmIGZzX3ByaW9yaXR5ID09ICdjc3YnOg0KICAgICAgICBmcyA9IG5wLndoZXJlKGNzdl9mcy5ub3RuYSgpLCBjc3ZfZnMgPT0gMS4wLCBmc19oZHIpDQogICAgICAgIHNyYyA9IG5wLndoZXJlKGNzdl9mcy5ub3RuYSgpLCAnY3N2JywgJ2hkcicpDQogICAgZWxzZToNCiAgICAgICAgdXNlX2NzdiA9ICgofmluZm9ybWF0aXZlKSAmIGNzdl9mcy5ub3RuYSgpKSBpZiBjc3ZfZmFsbGJhY2sgZWxzZSBwZC5TZXJpZXMoRmFsc2UsIGluZGV4PWRmLmluZGV4KQ0KICAgICAgICBmcyA9IG5wLndoZXJlKHVzZV9jc3YsIGNzdl9mcyA9PSAxLjAsIGZzX2hkcikNCiAgICAgICAgc3JjID0gbnAud2hlcmUodXNlX2NzdiwgJ2NzdicsICdoZHInKQ0KICAgIGRmWydmYXRzYXQnXSA9IGZzLmFzdHlwZShib29sKQ0KICAgIGRmWydmc19zb3VyY2UnXSA9IHNyYw0KICAgIGRmWydmc19jb25mbGljdCddID0gY3N2X2ZzLm5vdG5hKCkgJiAoKGNzdl9mcy5maWxsbmEoMCkuYXN0eXBlKGZsb2F0KSA9PSAxLjApICE9IGRmWydmYXRzYXQnXSkNCg0KICAgIHRyID0gcGQudG9fbnVtZXJpYyhkZlsnUmVwZXRpdGlvblRpbWUnXSwgZXJyb3JzPSdjb2VyY2UnKQ0KICAgIHRlID0gcGQudG9fbnVtZXJpYyhkZlsnRWNob1RpbWUnXSwgZXJyb3JzPSdjb2VyY2UnKQ0KICAgIGdyZSA9IGRmWydTY2FubmluZ1NlcXVlbmNlJ10uZmlsbG5hKCcnKS5hc3R5cGUoc3RyKS5zdHIudXBwZXIoKS5zdHIuY29udGFpbnMoJ0dSJykNCiAgICB0MSwgdDIsIHBkdyA9IGRlc2Muc3RyLmNvbnRhaW5zKF9UMV9SWCksIGRlc2Muc3RyLmNvbnRhaW5zKF9UMl9SWCksIGRlc2Muc3RyLmNvbnRhaW5zKF9QRF9SWCkNCiAgICB3ID0gbnAud2hlcmUodDEgJiB+dDIgJiB+cGR3LCAnVDEnLCBucC53aGVyZSh0MiAmIH5wZHcsICdUMicsIG5wLndoZXJlKHBkdywgJ1BEJywgbnAud2hlcmUoDQogICAgICAgIGdyZSwgJ0dSRScsIG5wLndoZXJlKHRyIDwgODAwLCAnVDEnLCBucC53aGVyZSh0ZSA+IDYwLCAnVDInLCBucC53aGVyZSh0ciA+PSA4MDAsICdQRCcsICdVTksnKSkpKSkpKQ0KICAgIGRmWyd3ZWlnaHQnXSA9IHcNCiAgICBmbHVpZCA9IG5wLmlzaW4odywgWydQRCcsICdUMiddKQ0KICAgICMgbGFzdCByZXNvcnQ6IG5vIGhlYWRlciBjb250cmFzdCBhdCBhbGwgLT4gdGhlIENTViBmbHVpZCBmbGFnICg9PSBmYXQtc2F0IGZsYWcgaW4gdHJhaW4pIGFzIGEgcHJveHkNCiAgICBpZiBjc3ZfZmFsbGJhY2sgYW5kICdGbHVpZF9TZW5zaXRpdmUnIGluIGRmLmNvbHVtbnM6DQogICAgICAgIGNmID0gZGZbJ0ZsdWlkX1NlbnNpdGl2ZSddLm1hcCh0b19mbGFnKQ0KICAgICAgICB1bmsgPSAodyA9PSAnVU5LJykgJiBjZi5ub3RuYSgpDQogICAgICAgIGZsdWlkID0gbnAud2hlcmUodW5rLCBjZiA9PSAxLjAsIGZsdWlkKQ0KICAgIGRmWydmbHVpZCddID0gZmx1aWQuYXN0eXBlKGJvb2wpDQoNCiAgICBhY3EgPSBkZlsnTVJBY3F1aXNpdGlvblR5cGUnXS5maWxsbmEoJycpLmFzdHlwZShzdHIpLnN0ci51cHBlcigpIGlmICdNUkFjcXVpc2l0aW9uVHlwZScgaW4gZGYuY29sdW1ucyBlbHNlICcnDQogICAgaWYgJ25fc2xpY2VzJyBpbiBkZi5jb2x1bW5zOg0KICAgICAgICBucyA9IHBkLnRvX251bWVyaWMoZGZbJ25fc2xpY2VzJ10sIGVycm9ycz0nY29lcmNlJykuZmlsbG5hKDApDQogICAgZWxpZiAnbl9maWxlcycgaW4gZGYuY29sdW1uczoNCiAgICAgICAgbnMgPSBwZC50b19udW1lcmljKGRmWyduX2ZpbGVzJ10sIGVycm9ycz0nY29lcmNlJykuZmlsbG5hKDApDQogICAgZWxzZToNCiAgICAgICAgbnMgPSBwZC5TZXJpZXMoMCwgaW5kZXg9ZGYuaW5kZXgpDQogICAgdGggPSBwZC50b19udW1lcmljKGRmLmdldCgnU2xpY2VUaGlja25lc3MnLCBucC5uYW4pLCBlcnJvcnM9J2NvZXJjZScpDQogICAgZGZbJ2lzM2QnXSA9IChhY3EgPT0gJzNEJykgfCAoKG5zID4gMTIwKSAmICh0aCA8IDEuNSkpDQogICAgcmV0dXJuIGRmDQoNCg0KZGVmIHBsYW5lX29mKGRmKToNCiAgICAiIiJDU1YgcGxhbmUgKHZlcmlmaWVkIDEwMCUgPSBESUNPTSBnZW9tZXRyeSkgd2l0aCBnZW9tZXRyeSBhcyB0aGUgZmFsbGJhY2suIiIiDQogICAgcCA9IGRmWydBbmF0b21pY2FsX1BsYW5lJ10gaWYgJ0FuYXRvbWljYWxfUGxhbmUnIGluIGRmLmNvbHVtbnMgZWxzZSBwZC5TZXJpZXMoTm9uZSwgaW5kZXg9ZGYuaW5kZXgpDQogICAgZyA9IGRmWydwbGFuZV9nZW8nXSBpZiAncGxhbmVfZ2VvJyBpbiBkZi5jb2x1bW5zIGVsc2UgcGQuU2VyaWVzKE5vbmUsIGluZGV4PWRmLmluZGV4KQ0KICAgIG9rID0gcC5pc2luKFsnU2FnaXR0YWwnLCAnQ29yb25hbCcsICdBeGlhbCddKQ0KICAgIHJldHVybiBwLndoZXJlKG9rLCBnKQ0KDQoNCmRlZiBhc3NpZ25fc2xvdHMoZywgcHJlZmVyXzJkPVRydWUsIHNsb3RzPU5vbmUpOg0KICAgICIiImc6IGFubm90YXRlZCByb3dzIG9mIE9ORSBzdHVkeSAobmVlZHMgcGxhbmUsIGZhdHNhdCwgZmx1aWQsIGlzM2QsIG5fc2xpY2VzLCBTZXJpZXNJbnN0YW5jZVVJRCkuDQoNCiAgICBSZXR1cm5zIChjaG9zZW4sIGFsdHMsIGV4dHJhcyk6DQogICAgICBjaG9zZW46IHtzbG90X25hbWU6IFNlcmllc0luc3RhbmNlVUlEfQ0KICAgICAgYWx0czogICB7c2xvdF9uYW1lOiBbb3RoZXIgY2FuZGlkYXRlcyBvZiB0aGUgc2FtZSBzbG90IHR5cGUsIGJlc3QgZmlyc3RdfSAgICh2aWV3LXN3YXAgYXVnbWVudGF0aW9uIHBvb2wpDQogICAgICBleHRyYXM6IHNlcmllcyBub3QgdXNlZCBieSBhbnkgc2xvdA0KICAgIFJ1bGUgKHB1YmxpYyk6IG1vc3Qgc2xpY2VzIHdpbnMuIFdpdGggcHJlZmVyXzJkIGEgMi1EIHNlcmllcyBhbHdheXMgYmVhdHMgYSAzLUQgdm9sdW1lLg0KICAgICIiIg0KICAgIGlmICdwbGFuZScgbm90IGluIGcuY29sdW1ucyBvciAnZmF0c2F0JyBub3QgaW4gZy5jb2x1bW5zIG9yICdmbHVpZCcgbm90IGluIGcuY29sdW1ucyBvciAnaXMzZCcgbm90IGluIGcuY29sdW1uczoNCiAgICAgICAgZyA9IGFubm90YXRlKGcpDQogICAgc2xvdHMgPSBzbG90cyBvciBjb25maWcuU0xPVFMNCiAgICBjaG9zZW4sIGFsdHMsIHVzZWQgPSB7fSwge30sIHNldCgpDQogICAgaWYgJ25fc2xpY2VzJyBpbiBnLmNvbHVtbnM6DQogICAgICAgIG5zID0gcGQudG9fbnVtZXJpYyhnWyduX3NsaWNlcyddLCBlcnJvcnM9J2NvZXJjZScpLmZpbGxuYSgwKQ0KICAgIGVsaWYgJ25fZmlsZXMnIGluIGcuY29sdW1uczoNCiAgICAgICAgbnMgPSBwZC50b19udW1lcmljKGdbJ25fZmlsZXMnXSwgZXJyb3JzPSdjb2VyY2UnKS5maWxsbmEoMCkNCiAgICBlbHNlOg0KICAgICAgICBucyA9IHBkLlNlcmllcygwLCBpbmRleD1nLmluZGV4KQ0KICAgIGdnID0gZy5hc3NpZ24oX249bnMpDQogICAgZm9yIG5hbWUsIHBsYW5lLCBmbHVpZCwgZnMgaW4gc2xvdHM6DQogICAgICAgIHNlbCA9IChnZ1sncGxhbmUnXSA9PSBwbGFuZSkgJiAoZ2dbJ2ZhdHNhdCddID09IGZzKQ0KICAgICAgICBpZiBmbHVpZCBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIHNlbCAmPSAoZ2dbJ2ZsdWlkJ10gPT0gZmx1aWQpDQogICAgICAgIGMgPSBnZ1tzZWxdDQogICAgICAgIGlmIGxlbihjKSA9PSAwOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgaWYgcHJlZmVyXzJkOg0KICAgICAgICAgICAgYyA9IGMuYXNzaWduKF9rPWNbJ2lzM2QnXS5maWxsbmEoRmFsc2UpLmFzdHlwZShpbnQpKQ0KICAgICAgICAgICAgYyA9IGMuc29ydF92YWx1ZXMoWydfaycsICdfbicsICdTZXJpZXNJbnN0YW5jZVVJRCddLCBhc2NlbmRpbmc9W1RydWUsIEZhbHNlLCBUcnVlXSkNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIGMgPSBjLnNvcnRfdmFsdWVzKFsnX24nLCAnU2VyaWVzSW5zdGFuY2VVSUQnXSwgYXNjZW5kaW5nPVtGYWxzZSwgVHJ1ZV0pDQogICAgICAgIGNob3NlbltuYW1lXSA9IGMuaWxvY1swXVsnU2VyaWVzSW5zdGFuY2VVSUQnXQ0KICAgICAgICBhbHRzW25hbWVdID0gW3MgZm9yIHMgaW4gY1snU2VyaWVzSW5zdGFuY2VVSUQnXS50b2xpc3QoKVsxOl1dDQogICAgICAgIHVzZWQuYWRkKGNob3NlbltuYW1lXSkNCiAgICBleHRyYXMgPSBbcyBmb3IgcyBpbiBnZ1snU2VyaWVzSW5zdGFuY2VVSUQnXSBpZiBzIG5vdCBpbiB1c2VkXQ0KICAgIHJldHVybiBjaG9zZW4sIGFsdHMsIGV4dHJhcw0KDQoNCmRlZiBhc3NpZ25fYWxsKGluZGV4X2RmLCBwcmVmZXJfMmQ9VHJ1ZSwgZnNfcHJpb3JpdHk9J2hkcicsIGNzdl9mYWxsYmFjaz1UcnVlKToNCiAgICAiIiJBbm5vdGF0ZSB0aGUgd2hvbGUgaW5kZXggYW5kIGFzc2lnbiBzbG90cyBmb3IgZXZlcnkgc3R1ZHkgKGZ1bGx5IHZlY3RvcmlzZWQ6IH5PKG4gbG9nIG4pLCBubyBwZXItc3R1ZHkgbG9vcCkuDQoNCiAgICBSZXR1cm5zIChhbm5vdGF0ZWRfZGYsIHNsb3RfdGFibGUpOg0KICAgICAgc2xvdF90YWJsZTogRGF0YUZyYW1lIGluZGV4ZWQgYnkgU3R1ZHlJbnN0YW5jZVVJRCwgb25lIGNvbHVtbiBwZXIgc2xvdCBob2xkaW5nIGEgU2VyaWVzSW5zdGFuY2VVSUQgb3IgTm9uZTsNCiAgICAgIHNsb3RfdGFibGUuYXR0cnNbJ2FsdHMnXToge3Nsb3Q6IHtzdHVkeTogW2FsdGVybmF0ZSBTZXJpZXNJbnN0YW5jZVVJRHNdfX0gZm9yIHZpZXctc3dhcCBhdWdtZW50YXRpb24uDQogICAgVGllLWJyZWFrIGlzIGRldGVybWluaXN0aWMgKHNlcmllcyBVSUQpLCB1bmxpa2UgYSBzdGFibGUgc29ydCBvdmVyIGRpcmVjdG9yeSBvcmRlci4iIiINCiAgICBpZiBpbmRleF9kZi5lbXB0eSBvciAnU3R1ZHlJbnN0YW5jZVVJRCcgbm90IGluIGluZGV4X2RmLmNvbHVtbnM6DQogICAgICAgIHN0dWRpZXMgPSBwZC5JbmRleChbXSwgbmFtZT0nU3R1ZHlJbnN0YW5jZVVJRCcpDQogICAgICAgIHRhYiA9IHBkLkRhdGFGcmFtZShpbmRleD1zdHVkaWVzLCBjb2x1bW5zPVtuIGZvciBuLCAqXyBpbiBjb25maWcuU0xPVFNdLCBkdHlwZT1vYmplY3QpDQogICAgICAgIHRhYi5hdHRyc1snYWx0cyddID0ge30NCiAgICAgICAgcmV0dXJuIGluZGV4X2RmLmNvcHkoKSwgdGFiDQogICAgZGYgPSBpbmRleF9kZi5jb3B5KCkNCiAgICBkZlsncGxhbmUnXSA9IHBsYW5lX29mKGRmKQ0KICAgIGRmID0gYW5ub3RhdGUoZGYsIGZzX3ByaW9yaXR5LCBjc3ZfZmFsbGJhY2spDQogICAgaGFzX2NvdW50ID0gKCduX3NsaWNlcycgaW4gZGYuY29sdW1ucykgb3IgKCduX2ZpbGVzJyBpbiBkZi5jb2x1bW5zKQ0KICAgIGlmICduX3NsaWNlcycgaW4gZGYuY29sdW1uczoNCiAgICAgICAgbnMgPSBwZC50b19udW1lcmljKGRmWyduX3NsaWNlcyddLCBlcnJvcnM9J2NvZXJjZScpLmZpbGxuYSgwKQ0KICAgIGVsaWYgJ25fZmlsZXMnIGluIGRmLmNvbHVtbnM6DQogICAgICAgIG5zID0gcGQudG9fbnVtZXJpYyhkZlsnbl9maWxlcyddLCBlcnJvcnM9J2NvZXJjZScpLmZpbGxuYSgwKQ0KICAgIGVsc2U6DQogICAgICAgIG5zID0gcGQuU2VyaWVzKDEsIGluZGV4PWRmLmluZGV4KQ0KICAgIHVzYWJsZSA9IGRmWydwbGFuZSddLmlzaW4oWydTYWdpdHRhbCcsICdDb3JvbmFsJywgJ0F4aWFsJ10pICYgKChucyA+IDApIGlmIGhhc19jb3VudCBlbHNlIFRydWUpDQogICAgd29yayA9IGRmLmFzc2lnbihfbj1ucywgX2s9KGRmWydpczNkJ10uYXN0eXBlKGludCkgaWYgcHJlZmVyXzJkIGVsc2UgMCkpW3VzYWJsZV0NCiAgICB3b3JrID0gd29yay5zb3J0X3ZhbHVlcyhbJ1N0dWR5SW5zdGFuY2VVSUQnLCAnX2snLCAnX24nLCAnU2VyaWVzSW5zdGFuY2VVSUQnXSwgYXNjZW5kaW5nPVtUcnVlLCBUcnVlLCBGYWxzZSwgVHJ1ZV0pDQogICAgc3R1ZGllcyA9IHBkLkluZGV4KGRmWydTdHVkeUluc3RhbmNlVUlEJ10udW5pcXVlKCksIG5hbWU9J1N0dWR5SW5zdGFuY2VVSUQnKQ0KICAgIHRhYiA9IHBkLkRhdGFGcmFtZShpbmRleD1zdHVkaWVzLCBjb2x1bW5zPVtuIGZvciBuLCAqXyBpbiBjb25maWcuU0xPVFNdLCBkdHlwZT1vYmplY3QpDQogICAgYWx0cyA9IHt9DQogICAgZm9yIG5hbWUsIHBsYW5lLCBmbHVpZCwgZnMgaW4gY29uZmlnLlNMT1RTOg0KICAgICAgICBtID0gKHdvcmtbJ3BsYW5lJ10gPT0gcGxhbmUpICYgKHdvcmtbJ2ZhdHNhdCddID09IGZzKQ0KICAgICAgICBpZiBmbHVpZCBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIG0gJj0gKHdvcmtbJ2ZsdWlkJ10gPT0gZmx1aWQpDQogICAgICAgIGMgPSB3b3JrW21dDQogICAgICAgIGZpcnN0ID0gfmMuZHVwbGljYXRlZCgnU3R1ZHlJbnN0YW5jZVVJRCcpDQogICAgICAgIHRhYi5sb2NbYy5sb2NbZmlyc3QsICdTdHVkeUluc3RhbmNlVUlEJ10udmFsdWVzLCBuYW1lXSA9IGMubG9jW2ZpcnN0LCAnU2VyaWVzSW5zdGFuY2VVSUQnXS52YWx1ZXMNCiAgICAgICAgcmVzdCA9IGNbfmZpcnN0XQ0KICAgICAgICBhbHRzW25hbWVdID0gcmVzdC5ncm91cGJ5KCdTdHVkeUluc3RhbmNlVUlEJylbJ1Nlcmllc0luc3RhbmNlVUlEJ10uYXBwbHkobGlzdCkudG9fZGljdCgpIGlmIGxlbihyZXN0KSBlbHNlIHt9DQogICAgdGFiID0gdGFiLndoZXJlKHRhYi5ub3RuYSgpLCBOb25lKQ0KICAgIHRhYi5hdHRyc1snYWx0cyddID0gYWx0cw0KICAgIHJldHVybiBkZiwgdGFiDQo=',
    'src/data/preprocess/splits.py': 'IiIiTGVha2FnZS1zYWZlIGdyb3VwIGZvbGRzLg0KDQpHcm91cHMgPSAocmVwb3J0IGxhbmd1YWdlIHwgc2Nhbm5lciBtb2RlbCk6IGxhbmd1YWdlIGlzIGEgc3Ryb25nIHNpdGUgcHJveHkgaW4gdGhpcyBkYXRhc2V0IChlLmcuIGV2ZXJ5IEN5cmlsbGljDQpyZXBvcnQgaXMgYSBQaGlsaXBzIEluZ2VuaWEsIGV2ZXJ5IEdlcm1hbi9GcmVuY2gvR3JlZWsvRHV0Y2ggcmVwb3J0IGlzIFNpZW1lbnMpIGFuZCBzY2FubmVyIG1vZGVsIHNlcGFyYXRlcyB0aGUNCm1hbnkgc2l0ZXMgaGlkaW5nIGluc2lkZSB0aGUgMSw3MTcgRW5nbGlzaCByZXBvcnRzLiBBdWRpdCBudW1iZXJzOiA3MiBncm91cHMsIGxhcmdlc3QgNS45JSBvZiBzdHVkaWVzLCAyNCBncm91cHMNCndpdGggPDEwIHN0dWRpZXMuIFJhcmUgZ3JvdXBzIGFyZSBtZXJnZWQgaW50byAncmFyZXw8dmVuZG9yPicuIEtub3duIGR1cGxpY2F0ZSBleGFtcyBzaGFyZSBhIGdyb3VwLg0KSWRlbnRpY2FsIHJlcG9ydCBURVhUUyBhcmUgZGVsaWJlcmF0ZWx5IE5PVCB1bmlvbmVkIGludG8gZ3JvdXBzOiB0ZW1wbGF0ZWQgRW5nbGlzaCByZXBvcnRzIHJlY3VyIGFjcm9zcyBzaXRlcyBhbmQNCndvdWxkIGNoYWluIHVucmVsYXRlZCBzaXRlcyBpbnRvIG9uZSBnaWFudCBjb21wb25lbnQuDQoiIiINCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQojIHRoZSBvbmx5IHRydWUgZHVwbGljYXRlIGV4YW1zIGZvdW5kIGJ5IHRoZSBhdWRpdCAoNSBpZGVudGljYWwgc2VyaWVzIGVhY2gsIFNpZW1lbnMgQWVyYSwgbm9uLWdvbGQpDQpLTk9XTl9EVVBfUEFJUlMgPSBbKA0KICAgICcxLjIuODI2LjAuMS4zNjgwMDQzLjguNDk4Ljk5NzI1NTIzNTYwMjU0NzMyMDk0NDMyMjQ0NTU2NjIzMzc2NzExJywNCiAgICAnMS4yLjgyNi4wLjEuMzY4MDA0My44LjQ5OC45OTkyNjYyNDk2ODI0MDY4MTc3MjczNTEwMjMwMzUzMTYxMzIwOScsDQopXQ0KDQoNCmRlZiBtYWtlX2dyb3VwcyhzdHVkeV9tZXRhLCBtaW5fZ3JvdXA9MTAsIGR1cF9wYWlycz1Ob25lLCBuX3NwbGl0cz01KToNCiAgICBzbSA9IHN0dWR5X21ldGEuY29weSgpDQogICAgZyA9IHNtWydsYW5nJ10uZmlsbG5hKCc/JykuYXN0eXBlKHN0cikgKyAnfCcgKyBzbVsnbW9kZWwnXS5maWxsbmEoJz8nKS5hc3R5cGUoc3RyKQ0KICAgIGNudCA9IGcubWFwKGcudmFsdWVfY291bnRzKCkpDQogICAgZyA9IGcud2hlcmUoY250ID49IG1pbl9ncm91cCwgJ3JhcmV8JyArIHNtWyd2ZW5kb3InXS5maWxsbmEoJz8nKS5hc3R5cGUoc3RyKSkNCiAgICBjbnQgPSBnLm1hcChnLnZhbHVlX2NvdW50cygpKSAgICAgICAgICAgICAgICAgICAgICAgIyBhICdyYXJlfDx2ZW5kb3I+JyBidWNrZXQgY2FuIGl0c2VsZiBiZSB0b28gc21hbGwgdG8gc3RyYXRpZnkNCiAgICBnID0gZy53aGVyZShjbnQgPj0gbl9zcGxpdHMsICdyYXJlfEFMTCcpDQogICAgc21bJ2dyb3VwJ10gPSBnDQogICAgZm9yIGEsIGIgaW4gKGR1cF9wYWlycyBpZiBkdXBfcGFpcnMgaXMgbm90IE5vbmUgZWxzZSBLTk9XTl9EVVBfUEFJUlMpOg0KICAgICAgICBpZiBhIGluIHNldChzbVsnU3R1ZHlJbnN0YW5jZVVJRCddKSBhbmQgYiBpbiBzZXQoc21bJ1N0dWR5SW5zdGFuY2VVSUQnXSk6DQogICAgICAgICAgICBnYSA9IHNtLmxvY1tzbVsnU3R1ZHlJbnN0YW5jZVVJRCddID09IGEsICdncm91cCddLmlsb2NbMF0NCiAgICAgICAgICAgIHNtLmxvY1tzbVsnU3R1ZHlJbnN0YW5jZVVJRCddID09IGIsICdncm91cCddID0gZ2ENCiAgICByZXR1cm4gc20NCg0KDQpkZWYgZ3JvdXBfZm9sZHMoc3R1ZHlfbWV0YSwgbl9zcGxpdHM9NSwgc2VlZD0yMDI2LCBtaW5fZ3JvdXA9MTAsIGR1cF9wYWlycz1Ob25lLCBzY2hlbWU9J3NpdGUnKToNCiAgICAiIiItPiBEYXRhRnJhbWVbU3R1ZHlJbnN0YW5jZVVJRCwgZ3JvdXAsIGZvbGQsIC4uLl0uDQoNCiAgICBzY2hlbWU9J3NpdGUnICAoZGVmYXVsdCwgZm9yIG1vZGVsIHNlbGVjdGlvbik6IGV2ZXJ5IHNpdGUvc2Nhbm5lciBncm91cCBpcyBzcHJlYWQgZXZlbmx5IG92ZXIgYWxsIGZvbGRzDQogICAgICAgICAgICAgICAgICAgKHN0cmF0aWZpZWQgYnkgZ3JvdXApIC0gbWlycm9ycyB0aGUgaGlkZGVuIHRlc3QsIHdoaWNoIGlzIGRyYXduIGZyb20gdGhlIHNhbWUgc2l0ZXMgYXMgdHJhaW4uDQogICAgICAgICAgICAgICAgICAgRHVwbGljYXRlIGV4YW1zIGFyZSBrZXB0IHRvZ2V0aGVyLg0KICAgIHNjaGVtZT0nZ3JvdXAnIChyb2J1c3RuZXNzIGNoZWNrKTogd2hvbGUgZ3JvdXBzIGFyZSBoZWxkIG91dCB0b2dldGhlciA9IHVuc2Vlbi1zY2FubmVyIGdlbmVyYWxpc2F0aW9uLg0KICAgICAgICAgICAgICAgICAgIFBlc3NpbWlzdGljIGJ5IGNvbnN0cnVjdGlvbjogYXVkaXQgc2hvd3MgMjA1LzIzOCBtZXRhZGF0YS1wb29yIHN0dWRpZXMgbGFuZCBpbiBvbmUgZm9sZCBhbmQNCiAgICAgICAgICAgICAgICAgICBtb3N0IGxhbmd1YWdlcyBhcmUgYWJzZW50IGZyb20gc29tZSB0cmFpbmluZyBmb2xkcy4iIiINCiAgICBzbSA9IG1ha2VfZ3JvdXBzKHN0dWR5X21ldGEsIG1pbl9ncm91cCwgZHVwX3BhaXJzLCBuX3NwbGl0cykNCiAgICBmb2xkID0gbnAuZnVsbChsZW4oc20pLCAtMSkNCiAgICBpZiBsZW4oc20pIDwgMiAqIG5fc3BsaXRzOiAgICAgICAgICAgICAgICAgICAgICAgICAgICMgdGlueSBkYXRhc2V0cyAoc21va2UgdGVzdHMpOiByb3VuZC1yb2Jpbiwgc3RpbGwgZHVwLXNhZmUNCiAgICAgICAgb3JkZXIgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkucGVybXV0YXRpb24obGVuKHNtKSkNCiAgICAgICAgZm9sZFtvcmRlcl0gPSBucC5hcmFuZ2UobGVuKHNtKSkgJSBuX3NwbGl0cw0KICAgICAgICBzbVsnZm9sZCddID0gZm9sZA0KICAgICAgICByZXR1cm4gc21bWydTdHVkeUluc3RhbmNlVUlEJywgJ2dyb3VwJywgJ2ZvbGQnXSArIFtjIGZvciBjIGluICgnbGFuZycsICdtb2RlbCcsICd2ZW5kb3InLCAnZ29sZCcpIGlmIGMgaW4gc20uY29sdW1uc11dDQogICAgaWYgc2NoZW1lID09ICdncm91cCc6DQogICAgICAgIGlmIHNtWydncm91cCddLm51bmlxdWUoKSA8IG5fc3BsaXRzOg0KICAgICAgICAgICAgZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgS0ZvbGQNCiAgICAgICAgICAgIGtmID0gS0ZvbGQobl9zcGxpdHM9bl9zcGxpdHMsIHNodWZmbGU9VHJ1ZSwgcmFuZG9tX3N0YXRlPXNlZWQpDQogICAgICAgICAgICBmb3IgaywgKF8sIHZhKSBpbiBlbnVtZXJhdGUoa2Yuc3BsaXQoc20pKToNCiAgICAgICAgICAgICAgICBmb2xkW3ZhXSA9IGsNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIGZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IFN0cmF0aWZpZWRHcm91cEtGb2xkDQogICAgICAgICAgICB5ID0gc21bJ3ZlbmRvciddLmZpbGxuYSgnPycpLmFzdHlwZSgnY2F0ZWdvcnknKS5jYXQuY29kZXMudmFsdWVzDQogICAgICAgICAgICBmb3IgaywgKF8sIHZhKSBpbiBlbnVtZXJhdGUoU3RyYXRpZmllZEdyb3VwS0ZvbGQobl9zcGxpdHM9bl9zcGxpdHMsIHNodWZmbGU9VHJ1ZSwgcmFuZG9tX3N0YXRlPXNlZWQpLnNwbGl0KHNtLCB5LCBzbVsnZ3JvdXAnXSkpOg0KICAgICAgICAgICAgICAgIGZvbGRbdmFdID0gaw0KICAgIGVsaWYgc2NoZW1lID09ICdzaXRlJzoNCiAgICAgICAgZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgU3RyYXRpZmllZEtGb2xkDQogICAgICAgIHVpZCA9IHNtWydTdHVkeUluc3RhbmNlVUlEJ10udmFsdWVzDQogICAgICAgIHBhcnRuZXIgPSB7fQ0KICAgICAgICBmb3IgYSwgYiBpbiAoZHVwX3BhaXJzIGlmIGR1cF9wYWlycyBpcyBub3QgTm9uZSBlbHNlIEtOT1dOX0RVUF9QQUlSUyk6DQogICAgICAgICAgICBwYXJ0bmVyW2JdID0gYSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGIgZm9sbG93cyBhDQogICAgICAgIHVuaXQgPSBucC5hcnJheShbcGFydG5lci5nZXQodSwgdSkgZm9yIHUgaW4gdWlkXSkNCiAgICAgICAgZmlyc3QgPSB+cGQuU2VyaWVzKHVuaXQpLmR1cGxpY2F0ZWQoKS52YWx1ZXMgICAgICAgICAgICAgICAgICMgb25lIHJlcHJlc2VudGF0aXZlIHJvdyBwZXIgdW5pdA0KICAgICAgICByZXAgPSBzbVtmaXJzdF0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQ0KICAgICAgICBjb3VudHMgPSByZXBbJ2dyb3VwJ10udmFsdWVfY291bnRzKCkNCiAgICAgICAgdG9vX3NtYWxsID0gY291bnRzW2NvdW50cyA8IG5fc3BsaXRzXS5pbmRleA0KICAgICAgICBpZiBsZW4odG9vX3NtYWxsKSA+IDAgYW5kIGxlbihjb3VudHMpID4gMDoNCiAgICAgICAgICAgIG1ham9yaXR5X2dycCA9IGNvdW50cy5pbmRleFswXQ0KICAgICAgICAgICAgcmVwWydncm91cCddID0gcmVwWydncm91cCddLnJlcGxhY2Uoe2c6IG1ham9yaXR5X2dycCBmb3IgZyBpbiB0b29fc21hbGx9KQ0KICAgICAgICB5ID0gcmVwWydncm91cCddLmFzdHlwZSgnY2F0ZWdvcnknKS5jYXQuY29kZXMudmFsdWVzDQogICAgICAgIHNrZiA9IFN0cmF0aWZpZWRLRm9sZChuX3NwbGl0cz1uX3NwbGl0cywgc2h1ZmZsZT1UcnVlLCByYW5kb21fc3RhdGU9c2VlZCkNCiAgICAgICAgdWZvbGQgPSB7fQ0KICAgICAgICBmb3IgaywgKF8sIHZhKSBpbiBlbnVtZXJhdGUoc2tmLnNwbGl0KHJlcCwgeSkpOg0KICAgICAgICAgICAgZm9yIHUgaW4gcmVwLmxvY1t2YSwgJ1N0dWR5SW5zdGFuY2VVSUQnXToNCiAgICAgICAgICAgICAgICB1Zm9sZFt1XSA9IGsNCiAgICAgICAgZm9sZCA9IG5wLmFycmF5KFt1Zm9sZFt1XSBmb3IgdSBpbiB1bml0XSkNCiAgICBlbHNlOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKHNjaGVtZSkNCiAgICBzbVsnZm9sZCddID0gZm9sZA0KICAgIHJldHVybiBzbVtbJ1N0dWR5SW5zdGFuY2VVSUQnLCAnZ3JvdXAnLCAnZm9sZCddICsgW2MgZm9yIGMgaW4gKCdsYW5nJywgJ21vZGVsJywgJ3ZlbmRvcicsICdnb2xkJykgaWYgYyBpbiBzbS5jb2x1bW5zXV0NCg0KDQpkZWYgbWFrZV9zdHVkeV9tZXRhKGluZGV4X2RmLCB0cmFpbl9jc3Y9Tm9uZSwgbGFiZWxzX2RmPU5vbmUpOg0KICAgICIiIlN0dWR5LWxldmVsIHRhYmxlICh2ZW5kb3IsIHNjYW5uZXIgbW9kZWwsIHJlcG9ydCBsYW5ndWFnZSwgZ29sZCBmbGFnLCBtZXRhZGF0YS1wb29yIGZsYWcpIHN0cmFpZ2h0IGZyb20gdGhlDQogICAgaW5kZXggKyB0cmFpbi5jc3YsIHNvIHRoZSBub3RlYm9vayBuZXZlciBuZWVkcyB0aGUgYXVkaXQncyBzdHVkeV9tZXRhLmNzdi4iIiINCiAgICBmcm9tIC50ZXh0IGltcG9ydCBndWVzc19sYW5nLCB2ZW5kb3Jfb2YNCiAgICBpZiBpbmRleF9kZiBpcyBOb25lIG9yIGxlbihpbmRleF9kZikgPT0gMDoNCiAgICAgICAgZGYgPSBwZC5EYXRhRnJhbWUoKQ0KICAgIGVsc2U6DQogICAgICAgIGRmID0gaW5kZXhfZGZbaW5kZXhfZGZbJ3NwbGl0J10gPT0gJ3RyYWluJ10gaWYgJ3NwbGl0JyBpbiBpbmRleF9kZi5jb2x1bW5zIGVsc2UgaW5kZXhfZGYNCiAgICBpZiBkZi5lbXB0eSBvciAnU3R1ZHlJbnN0YW5jZVVJRCcgbm90IGluIGRmLmNvbHVtbnM6DQogICAgICAgIGlmIGxhYmVsc19kZiBpcyBub3QgTm9uZSBhbmQgJ1N0dWR5SW5zdGFuY2VVSUQnIGluIGxhYmVsc19kZi5jb2x1bW5zOg0KICAgICAgICAgICAgc3R1ZGllcyA9IHBkLkluZGV4KGxhYmVsc19kZlsnU3R1ZHlJbnN0YW5jZVVJRCddLnVuaXF1ZSgpLCBuYW1lPSdTdHVkeUluc3RhbmNlVUlEJykNCiAgICAgICAgZWxpZiB0cmFpbl9jc3YgaXMgbm90IE5vbmUgYW5kICdTdHVkeUluc3RhbmNlVUlEJyBpbiB0cmFpbl9jc3YuY29sdW1uczoNCiAgICAgICAgICAgIHN0dWRpZXMgPSBwZC5JbmRleCh0cmFpbl9jc3ZbJ1N0dWR5SW5zdGFuY2VVSUQnXS51bmlxdWUoKSwgbmFtZT0nU3R1ZHlJbnN0YW5jZVVJRCcpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBzdHVkaWVzID0gcGQuSW5kZXgoW10sIG5hbWU9J1N0dWR5SW5zdGFuY2VVSUQnKQ0KICAgICAgICBzbSA9IHBkLkRhdGFGcmFtZSh7J3ZlbmRvcic6ICc/JywgJ21vZGVsJzogJz8nLCAncG9vcic6IEZhbHNlfSwgaW5kZXg9c3R1ZGllcykucmVzZXRfaW5kZXgoKQ0KICAgICAgICBzbVsnbGFuZyddID0gJz8nDQogICAgICAgIGlmIHRyYWluX2NzdiBpcyBub3QgTm9uZSBhbmQgJ1JlcG9ydCcgaW4gdHJhaW5fY3N2LmNvbHVtbnM6DQogICAgICAgICAgICBsYW5nID0gdHJhaW5fY3N2LnNldF9pbmRleCgnU3R1ZHlJbnN0YW5jZVVJRCcpWydSZXBvcnQnXS5tYXAoZ3Vlc3NfbGFuZykNCiAgICAgICAgICAgIHNtWydsYW5nJ10gPSBzbVsnU3R1ZHlJbnN0YW5jZVVJRCddLm1hcChsYW5nKS5maWxsbmEoJz8nKQ0KICAgICAgICBzbVsnZ29sZCddID0gRmFsc2UNCiAgICAgICAgaWYgbGFiZWxzX2RmIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCiAgICAgICAgICAgIGlmICdzb3VyY2UnIGluIGxhYmVsc19kZi5jb2x1bW5zOg0KICAgICAgICAgICAgICAgIGhhcyA9IGxhYmVsc19kZlsnc291cmNlJ10gPT0gJ2dvbGQnDQogICAgICAgICAgICBlbHNlOg0KICAgICAgICAgICAgICAgIGhhcyA9IGxhYmVsc19kZltbdCBmb3IgdCBpbiBjb25maWcuVEFSR0VUUyBpZiB0IGluIGxhYmVsc19kZi5jb2x1bW5zXV0ubm90bmEoKS5hbnkoYXhpcz0xKQ0KICAgICAgICAgICAgZ29sZCA9IGxhYmVsc19kZi5hc3NpZ24oX2c9aGFzKS5zZXRfaW5kZXgoJ1N0dWR5SW5zdGFuY2VVSUQnKVsnX2cnXQ0KICAgICAgICAgICAgc21bJ2dvbGQnXSA9IHNtWydTdHVkeUluc3RhbmNlVUlEJ10ubWFwKGdvbGQpLmZpbGxuYShGYWxzZSkNCiAgICAgICAgcmV0dXJuIHNtDQogICAgbW9kZSA9IGxhbWJkYSBzOiBzLmRyb3BuYSgpLm1vZGUoKS5pbG9jWzBdIGlmIHMubm90bmEoKS5hbnkoKSBlbHNlIG5wLm5hbg0KICAgIGcgPSBkZi5ncm91cGJ5KCdTdHVkeUluc3RhbmNlVUlEJykNCiAgICBzdHVkaWVzID0gcGQuSW5kZXgoZGZbJ1N0dWR5SW5zdGFuY2VVSUQnXS51bmlxdWUoKSwgbmFtZT0nU3R1ZHlJbnN0YW5jZVVJRCcpDQogICAgdmVuZG9yID0gZ1snTWFudWZhY3R1cmVyJ10uYWdnKGxhbWJkYSBzOiBtb2RlKHMubWFwKHZlbmRvcl9vZikpKSBpZiAnTWFudWZhY3R1cmVyJyBpbiBkZiBlbHNlIHBkLlNlcmllcygnPycsIGluZGV4PXN0dWRpZXMpDQogICAgbW9kZWwgPSBnWydNYW51ZmFjdHVyZXJNb2RlbE5hbWUnXS5hZ2cobW9kZSkgaWYgJ01hbnVmYWN0dXJlck1vZGVsTmFtZScgaW4gZGYgZWxzZSBwZC5TZXJpZXMoJz8nLCBpbmRleD1zdHVkaWVzKQ0KICAgIHBvb3IgPSBnWydSZXBldGl0aW9uVGltZSddLmFnZyhsYW1iZGEgczogcy5pc25hKCkuYWxsKCkpIGlmICdSZXBldGl0aW9uVGltZScgaW4gZGYgZWxzZSBwZC5TZXJpZXMoRmFsc2UsIGluZGV4PXN0dWRpZXMpDQogICAgc20gPSBwZC5EYXRhRnJhbWUoeyd2ZW5kb3InOiB2ZW5kb3IsICdtb2RlbCc6IG1vZGVsLCAncG9vcic6IHBvb3J9LCBpbmRleD1zdHVkaWVzKS5yZXNldF9pbmRleCgpDQogICAgc21bJ2xhbmcnXSA9ICc/Jw0KICAgIGlmIHRyYWluX2NzdiBpcyBub3QgTm9uZSBhbmQgJ1JlcG9ydCcgaW4gdHJhaW5fY3N2LmNvbHVtbnM6DQogICAgICAgIGxhbmcgPSB0cmFpbl9jc3Yuc2V0X2luZGV4KCdTdHVkeUluc3RhbmNlVUlEJylbJ1JlcG9ydCddLm1hcChndWVzc19sYW5nKQ0KICAgICAgICBzbVsnbGFuZyddID0gc21bJ1N0dWR5SW5zdGFuY2VVSUQnXS5tYXAobGFuZykuZmlsbG5hKCc/JykNCiAgICBpZiBsYWJlbHNfZGYgaXMgbm90IE5vbmU6DQogICAgICAgIGltcG9ydCBzcmMuY29yZS5jb25maWcgYXMgY29uZmlnDQogICAgICAgIGlmICdzb3VyY2UnIGluIGxhYmVsc19kZi5jb2x1bW5zOiAgICAgICAgICAgICAgICAgICAgICAgIyBidWlsZF9sYWJlbHMoKSBtYXJrcyBnb2xkIC8gZXh0cmEgLyBub25lDQogICAgICAgICAgICBoYXMgPSBsYWJlbHNfZGZbJ3NvdXJjZSddID09ICdnb2xkJw0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgaGFzID0gbGFiZWxzX2RmW1t0IGZvciB0IGluIGNvbmZpZy5UQVJHRVRTIGlmIHQgaW4gbGFiZWxzX2RmLmNvbHVtbnNdXS5ub3RuYSgpLmFueShheGlzPTEpDQogICAgICAgIGdvbGQgPSBsYWJlbHNfZGYuYXNzaWduKF9nPWhhcykuc2V0X2luZGV4KCdTdHVkeUluc3RhbmNlVUlEJylbJ19nJ10NCiAgICAgICAgc21bJ2dvbGQnXSA9IHNtWydTdHVkeUluc3RhbmNlVUlEJ10ubWFwKGdvbGQpLmZpbGxuYShGYWxzZSkNCiAgICBlbHNlOg0KICAgICAgICBzbVsnZ29sZCddID0gRmFsc2UNCiAgICByZXR1cm4gc20NCg==',
    'src/data/preprocess/text.py': 'IiIiVGlueSB0ZXh0L3ZlbmRvciBoZWxwZXJzIHVzZWQgdG8gYnVpbGQgbGVha2FnZS1zYWZlIGZvbGRzIChubyBleHRlcm5hbCBOTFAgZGVwZW5kZW5jeSkuIiIiCmltcG9ydCByZQoKX1NUT1AgPSB7CiAgICAnZW4nOiAndGhlIGFuZCBvZiB3aXRoIGlzIGFyZSB0aGVyZSBubyB3aXRob3V0IGluIHRvIGFuIGF0IGZyb20gd2hpY2ggc2VlbiBub3RlZCcsCiAgICAnZXMnOiAnZWwgbGEgbG9zIGxhcyBkZSBkZWwgY29uIHNpbiBzZSBlbiB5IHVuYSB1biBxdWUgZXMgaGF5IG5vIG9ic2VydmEnLAogICAgJ3B0JzogJ28gb3MgYXMgZGUgZGEgZG8gY29tIHNlbSBlbSBlIHVtYSB1bSBxdWUgbsOjbyBuYW8gaMOhIGhhJywKICAgICdmcic6ICdsZSBsYSBsZXMgZGVzIGR1IGRlIGV0IGF2ZWMgc2FucyB1biB1bmUgZXN0IHBhcyBpbCB5IGEgZW4gZGFucycsCiAgICAnaXQnOiAnaWwgbG8gZ2xpIGxlIGRpIGRlbCBkZWxsYSBjb24gc2VuemEgbm9uIMOoIHVuYSB1biBlIG5lbCBuZWxsYSBzaScsCiAgICAnZGUnOiAnZGVyIGRpZSBkYXMgdW5kIG1pdCBvaG5lIGtlaW4ga2VpbmUgaXN0IG5pY2h0IGVpbmUgaW0gZGVuIGRlcyB6ZWlndCcsCiAgICAndHInOiAndmUgaWxlIGJpciBvbGFyYWsgeW9rIGl6bGVuZGkgZ8O2csO8bG1la3RlZGlyIG5vcm1hbCBkZSBkYSBidWx1bm1ha3RhZMSxcicsCiAgICAnaHIvYnMvc3InOiAnaSB1IGplIG5hIHNlIGJleiBzIG5lbWEgbmlqZSB2aWRsaml2IHZpZGxqaXZhIHVyZWRhbiB1cmVkbmEnLAogICAgJ25sJzogJ2RlIGhldCBlZW4gZW4gbWV0IHpvbmRlciBnZWVuIGlzIG5pZXQgdmFuIGluIG9wIHRlJywKfQpfU1RPUCA9IHtrOiBzZXQodi5zcGxpdCgpKSBmb3IgaywgdiBpbiBfU1RPUC5pdGVtcygpfQoKCmRlZiBndWVzc19sYW5nKHRleHQpOgogICAgIiIiQ29hcnNlIHJlcG9ydC1sYW5ndWFnZSBwcm94eSAoc2NyaXB0IHJhbmdlcyArIHN0b3Atd29yZCBzY29yZXMpLiBPbmx5IHVzZWQgYXMgYSBTSVRFIHByb3h5IGZvciBmb2xkIGdyb3VwaW5nLAogICAgbmV2ZXIgYXMgYSBmZWF0dXJlLiBLbm93biBsaW1pdGF0aW9uOiB0ZW1wbGF0ZWQgRW5nbGlzaCByZXBvcnRzIGNhbiBiZSB0YWdnZWQgJ3RyJzsgcGFpcmVkIHdpdGggc2Nhbm5lciBtb2RlbAogICAgaW4gdGhlIGdyb3VwIGtleSwgdGhpcyBkb2VzIG5vdCBtYXR0ZXIuIiIiCiAgICBpZiBub3QgaXNpbnN0YW5jZSh0ZXh0LCBzdHIpIG9yIGxlbih0ZXh0LnN0cmlwKCkpIDwgNToKICAgICAgICByZXR1cm4gJ2VtcHR5JwogICAgdCA9IHRleHQubG93ZXIoKQogICAgbGV0dGVycyA9IFtjaCBmb3IgY2ggaW4gdCBpZiBjaC5pc2FscGhhKCldCiAgICBpZiBub3QgbGV0dGVyczoKICAgICAgICByZXR1cm4gJ2VtcHR5JwoKICAgIGRlZiBmcmFjKGEsIGIpOgogICAgICAgIHJldHVybiBzdW0oYSA8PSBjaCA8PSBiIGZvciBjaCBpbiBsZXR0ZXJzKSAvIGxlbihsZXR0ZXJzKQogICAgaWYgZnJhYygnXHUwNDAwJywgJ1x1MDRmZicpID4gLjM6CiAgICAgICAgcmV0dXJuICdjeXJpbGxpYycKICAgIGlmIGZyYWMoJ1x1MDM3MCcsICdcdTAzZmYnKSA+IC4zOgogICAgICAgIHJldHVybiAnZ3JlZWsnCiAgICBpZiBmcmFjKCdcdTBlMDAnLCAnXHUwZTdmJykgPiAuMzoKICAgICAgICByZXR1cm4gJ3RoYWknCiAgICBpZiBmcmFjKCdcdTRlMDAnLCAnXHU5ZmZmJykgPiAuMjoKICAgICAgICByZXR1cm4gJ2NqaycKICAgIGlmIGZyYWMoJ1x1MDYwMCcsICdcdTA2ZmYnKSA+IC4zOgogICAgICAgIHJldHVybiAnYXJhYmljJwogICAgdG9rcyA9IHJlLmZpbmRhbGwocidbXlxXXGRfXSsnLCB0KQogICAgc2MgPSB7azogc3VtKHcgaW4gdiBmb3IgdyBpbiB0b2tzKSBmb3IgaywgdiBpbiBfU1RPUC5pdGVtcygpfQogICAgc2NbJ3RyJ10gKz0gMiAqIHN1bShjaCBpbiAnxJ/EscWfxLAnIGZvciBjaCBpbiB0ZXh0KSAvIG1heChsZW4odG9rcyksIDEpICogMTAKICAgIGJlc3QgPSBtYXgoc2MsIGtleT1zYy5nZXQpCiAgICByZXR1cm4gYmVzdCBpZiBzY1tiZXN0XSA+PSAzIGVsc2UgJ3Vua25vd24nCgoKZGVmIHZlbmRvcl9vZihtYW51ZmFjdHVyZXIpOgogICAgIiIiTm9ybWFsaXNlIHRoZSBtYW55IHNwZWxsaW5ncyBzZWVuIGluIHRoZSBkYXRhIChTaWVtZW5zL1NJRU1FTlMvU2llbWVucyBIZWFsdGhpbmVlcnMsIEdFL0dFSEMsIC4uLikuIiIiCiAgICBtID0gc3RyKG1hbnVmYWN0dXJlcikubG93ZXIoKQogICAgZm9yIGssIHYgaW4gKCgnc2llbWVucycsICdTSUVNRU5TJyksICgnZ2UgbWVkaWNhbCcsICdHRScpLCAoJ2dlaGMnLCAnR0UnKSwgKCdwaGlsaXBzJywgJ1BISUxJUFMnKSwKICAgICAgICAgICAgICAgICAoJ3Rvc2hpYmEnLCAnQ0FOT04vVE9TSElCQScpLCAoJ2Nhbm9uJywgJ0NBTk9OL1RPU0hJQkEnKSwgKCdmdWppJywgJ0ZVSkkvSElUQUNISScpLAogICAgICAgICAgICAgICAgICgnaGl0YWNoaScsICdGVUpJL0hJVEFDSEknKSk6CiAgICAgICAgaWYgayBpbiBtOgogICAgICAgICAgICByZXR1cm4gdgogICAgcmV0dXJuICdPVEhFUicK',
    'src/data/preprocess/__init__.py': 'IiIiRGF0YXNldC12ZXJpZmllZCBwcmVwcm9jZXNzaW5nIGxheWVyIChudW1weS9PcGVuQ1Ygb25seTsgdG9yY2gtZnJlZSkuIiIiCmltcG9ydCBvcwpvcy5lbnZpcm9uWydPTVBfTlVNX1RIUkVBRFMnXSA9ICcxJwpvcy5lbnZpcm9uWydPUEVOQkxBU19OVU1fVEhSRUFEUyddID0gJzEnCm9zLmVudmlyb25bJ01LTF9OVU1fVEhSRUFEUyddID0gJzEnCm9zLmVudmlyb25bJ1ZFQ0xJQl9NQVhJTVVNX1RIUkVBRFMnXSA9ICcxJwpvcy5lbnZpcm9uWydOVU1FWFBSX05VTV9USFJFQURTJ10gPSAnMScKCmltcG9ydCBjdjIKY3YyLnNldE51bVRocmVhZHMoMCkK',
    'src/inference/inference.py': '...',
    'src/inference/__init__.py': '',
    'src/modeling/ensemble.py': 'IiIiRW5zZW1ibGUgdXRpbGl0aWVzLg0KDQpJbXBsZW1lbnRzOg0KICBUZW1wZXJhdHVyZUNhbGlicmF0aW9uICBQb3N0LWhvYyBwcm9iYWJpbGl0eSBjYWxpYnJhdGlvbiBwZXIgbW9kZWwgYXJtLg0KICAgICAgICAgICAgICAgICAgICAgICAgICBHdW8gZXQgYWwuLCBJQ01MIDIwMTcg4oCUICJPbiBDYWxpYnJhdGlvbiBvZiBNb2Rlcm4gTmV1cmFsIE5ldHdvcmtzIi4NCiAgICAgICAgICAgICAgICAgICAgICAgICAgRmluZHMgc2NhbGFyIFQgcGVyIGFybSB0aGF0IG1pbmltaXNlcyBOTEwgb24gT09GIHByZWRpY3Rpb25zLg0KICAgICAgICAgICAgICAgICAgICAgICAgICBBVUMgaXMgcmFuay1pbnZhcmlhbnQgKHVuYWZmZWN0ZWQgYnkgbW9ub3RvbmUgVCBzY2FsaW5nKSBidXQgdGhlDQogICAgICAgICAgICAgICAgICAgICAgICAgIGNhbGlicmF0ZWQgcHJvYmFiaWxpdGllcyBtYWtlIHJhbmstcGVyY2VudGlsZSBibGVuZGluZyBtb3JlIHByaW5jaXBsZWQuDQoNCiAgZ3JlZWR5X2Vuc2VtYmxlX3NlbGVjdCAgQ2FydWFuYSBldCBhbC4sIElDTUwgMjAwNCDigJQgIkVuc2VtYmxlIFNlbGVjdGlvbiBmcm9tIExpYnJhcmllcyBvZiBNb2RlbHMiLg0KICAgICAgICAgICAgICAgICAgICAgICAgICBHcmVlZHkgZm9yd2FyZCBzZWxlY3Rpb24gZnJvbSBhIHBvb2wgb2YgbW9kZWwgcHJlZGljdGlvbnMgb24gYSBoZWxkLW91dA0KICAgICAgICAgICAgICAgICAgICAgICAgICBPT0Ygc2V0LiBBbHdheXMgPj0gbWVhbiBibGVuZGluZy4gU3VwcG9ydHMgd2l0aC1yZXBsYWNlbWVudCAobW9kZWxzIGNhbg0KICAgICAgICAgICAgICAgICAgICAgICAgICBiZSBzZWxlY3RlZCBtdWx0aXBsZSB0aW1lcywgZWZmZWN0aXZlbHkgdXAtd2VpZ2h0aW5nIHRoZW0pLg0KDQogIHJhbmtfZW5zZW1ibGVfbiAgICAgICAgIE4tYXJtIHJhbmstcGVyY2VudGlsZSBibGVuZGluZyAoZ2VuZXJhbGl6YXRpb24gb2YgYmFzZWxpbmUgMi1hcm0gYmxlbmQpLg0KICAgICAgICAgICAgICAgICAgICAgICAgICBQZXItdGFyZ2V0IENvQXROZXQgd2VpZ2h0aW5nIGtlcHQgZnJvbSBiYXNlbGluZSAoMC45NDMgc291cmNlKS4NCiAgICAgICAgICAgICAgICAgICAgICAgICAgVGhpcmQgYXJtIChDb252TmVYdCAvIFJhZEltYWdlTmV0IENOTikgYWRkZWQgd2l0aCBwZXItdGFyZ2V0IHdlaWdodHMuDQoiIiINCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQpmcm9tIHNjaXB5Lm9wdGltaXplIGltcG9ydCBtaW5pbWl6ZV9zY2FsYXINCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCg0KDQojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgCBUZW1wZXJhdHVyZSBDYWxpYnJhdGlvbiDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIANCmNsYXNzIFRlbXBlcmF0dXJlQ2FsaWJyYXRpb246DQogICAgIiIiUGVyLWFybSB0ZW1wZXJhdHVyZSBzY2FsaW5nIChHdW8gZXQgYWwuLCBJQ01MIDIwMTcpLg0KDQogICAgVXNhZ2UNCiAgICAtLS0tLQ0KICAgIGNhbCA9IFRlbXBlcmF0dXJlQ2FsaWJyYXRpb24oKQ0KICAgIGNhbC5maXQob29mX2xvZ2l0cywgb29mX2xhYmVscykgICAgICAgICAgIyBudW1weSBbTiwgQ10NCiAgICBjYWxpYnJhdGVkX3Byb2JzID0gY2FsLnRyYW5zZm9ybShsb2dpdHMpICMgbnVtcHkgW04sIENdDQogICAgIiIiDQoNCiAgICBkZWYgX19pbml0X18oc2VsZik6DQogICAgICAgIHNlbGYuVDogZmxvYXQgPSAxLjANCg0KICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQogICAgZGVmIF9ubGwoc2VsZiwgVDogZmxvYXQsIGxvZ2l0czogbnAubmRhcnJheSwgbGFiZWxzOiBucC5uZGFycmF5KSAtPiBmbG9hdDoNCiAgICAgICAgIiIiQmluYXJ5IGNyb3NzLWVudHJvcHkgTkxMIGZvciBhIGdpdmVuIHRlbXBlcmF0dXJlLiIiIg0KICAgICAgICBwID0gMS4wIC8gKDEuMCArIG5wLmV4cCgtbG9naXRzIC8gbWF4KFQsIDFlLTYpKSkNCiAgICAgICAgcCA9IG5wLmNsaXAocCwgMWUtNywgMS4wIC0gMWUtNykNCiAgICAgICAgcmV0dXJuIC1ucC5tZWFuKGxhYmVscyAqIG5wLmxvZyhwKSArICgxIC0gbGFiZWxzKSAqIG5wLmxvZygxIC0gcCkpDQoNCiAgICBkZWYgZml0KHNlbGYsIG9vZl9sb2dpdHM6IG5wLm5kYXJyYXksIG9vZl9sYWJlbHM6IG5wLm5kYXJyYXkpIC0+ICJUZW1wZXJhdHVyZUNhbGlicmF0aW9uIjoNCiAgICAgICAgIiIiRmluZCBUKiA9IGFyZ21pbiBOTEwoVCkgb24gT09GIGRhdGEgdmlhIGJvdW5kZWQgc2NhbGFyIHNlYXJjaC4NCg0KICAgICAgICBsb2dpdHM6IHJhdyBtb2RlbCBvdXRwdXQgKHByZS1zaWdtb2lkKSBbTiwgQ10uDQogICAgICAgIGxhYmVsczogYmluYXJ5IFtOLCBDXS4NCiAgICAgICAgT25seSB1c2VzIGxhYmVsbGVkIGVudHJpZXMgKHdlaWdodCA+IDApIGlmIG9vZl9sYWJlbHMgY29udGFpbnMgTmFOcy4NCiAgICAgICAgIiIiDQogICAgICAgIG1hc2sgPSBucC5pc2Zpbml0ZShvb2ZfbGFiZWxzKSAmIG5wLmlzZmluaXRlKG9vZl9sb2dpdHMpDQogICAgICAgIGxvID0gb29mX2xvZ2l0c1ttYXNrXS5yYXZlbCgpDQogICAgICAgIGxhID0gb29mX2xhYmVsc1ttYXNrXS5yYXZlbCgpDQogICAgICAgIGlmIGxlbihsbykgPT0gMCBvciBsYS5zdW0oKSA9PSAwIG9yIGxhLnN1bSgpID09IGxlbihsYSk6DQogICAgICAgICAgICBzZWxmLlQgPSAxLjANCiAgICAgICAgICAgIHJldHVybiBzZWxmDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHJlcyA9IG1pbmltaXplX3NjYWxhcigNCiAgICAgICAgICAgICAgICBsYW1iZGEgVDogc2VsZi5fbmxsKFQsIGxvLCBsYSksDQogICAgICAgICAgICAgICAgYm91bmRzPSgwLjEsIDEwLjApLA0KICAgICAgICAgICAgICAgIG1ldGhvZD0iYm91bmRlZCIsDQogICAgICAgICAgICApDQogICAgICAgICAgICBzZWxmLlQgPSBmbG9hdChyZXMueCkgaWYgKGhhc2F0dHIocmVzLCAieCIpIGFuZCBucC5pc2Zpbml0ZShyZXMueCkpIGVsc2UgMS4wDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICBzZWxmLlQgPSAxLjANCiAgICAgICAgcmV0dXJuIHNlbGYNCg0KICAgIGRlZiB0cmFuc2Zvcm0oc2VsZiwgbG9naXRzOiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5Og0KICAgICAgICAiIiJBcHBseSBjYWxpYnJhdGlvbjogc2lnbW9pZChsb2dpdHMgLyBUKS4iIiINCiAgICAgICAgcmV0dXJuIDEuMCAvICgxLjAgKyBucC5leHAoLWxvZ2l0cyAvIG1heChzZWxmLlQsIDFlLTYpKSkNCg0KICAgIGRlZiBmaXRfdHJhbnNmb3JtKHNlbGYsIG9vZl9sb2dpdHMsIG9vZl9sYWJlbHMsIHRlc3RfbG9naXRzPU5vbmUpOg0KICAgICAgICBzZWxmLmZpdChvb2ZfbG9naXRzLCBvb2ZfbGFiZWxzKQ0KICAgICAgICBpZiB0ZXN0X2xvZ2l0cyBpcyBOb25lOg0KICAgICAgICAgICAgcmV0dXJuIHNlbGYudHJhbnNmb3JtKG9vZl9sb2dpdHMpDQogICAgICAgIHJldHVybiBzZWxmLnRyYW5zZm9ybSh0ZXN0X2xvZ2l0cykNCg0KICAgIGRlZiBfX3JlcHJfXyhzZWxmKToNCiAgICAgICAgcmV0dXJuIGYiVGVtcGVyYXR1cmVDYWxpYnJhdGlvbihUPXtzZWxmLlQ6LjRmfSkiDQoNCg0KIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgR3JlZWR5IEVuc2VtYmxlIFNlbGVjdGlvbiDilIDilIDilIDilIDilIDilIDilIDilIANCmRlZiBncmVlZHlfZW5zZW1ibGVfc2VsZWN0KA0KICAgIG9vZl9wcmVkczogbGlzdFtucC5uZGFycmF5XSwgICAjIGxpc3Qgb2YgW04sIENdIHByb2JhYmlsaXR5IGFycmF5cyAocGVyIG1vZGVsKQ0KICAgIG9vZl9sYWJlbHM6IG5wLm5kYXJyYXksICAgICAgICAjIFtOLCBDXSBiaW5hcnkgZ3JvdW5kIHRydXRoDQogICAgbl9zZWxlY3Q6IGludCA9IDEwLCAgICAgICAgICAgICMgbWF4IGVuc2VtYmxlIG1lbWJlcnMgKHdpdGggcmVwbGFjZW1lbnQpDQogICAgbWV0cmljOiBzdHIgPSAiYXVjIiwgICAgICAgICAgICMgImF1YyIgb3IgImJjZSINCikgLT4gbGlzdFtpbnRdOg0KICAgICIiIkNhcnVhbmEgZXQgYWwuLCBJQ01MIDIwMDQg4oCUIGdyZWVkeSBmb3J3YXJkIHNlbGVjdGlvbiB3aXRoIHJlcGxhY2VtZW50Lg0KDQogICAgUmV0dXJucyBhIGxpc3Qgb2YgaW5kaWNlcyAod2l0aCByZXBldGl0aW9ucykgaW50byBvb2ZfcHJlZHMgcmVwcmVzZW50aW5nIHRoZQ0KICAgIHNlbGVjdGVkIGVuc2VtYmxlIG1lbWJlcnMuIFdlaWdodCBvZiBlYWNoIG1vZGVsID0gY291bnQoaW5kZXgpIC8gbl9zZWxlY3QuDQoNCiAgICBUaGlzIGd1YXJhbnRlZXMgdGhlIHNlbGVjdGVkIGVuc2VtYmxlIGlzIGFsd2F5cyA+PSB0aGUgYmVzdCBzaW5nbGUgbW9kZWwNCiAgICBvbiB0aGUgT09GIG1ldHJpYyAoaGlsbC1jbGltYmluZyBwcm9wZXJ0eSkuDQogICAgIiIiDQogICAgZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHJvY19hdWNfc2NvcmUNCg0KICAgIGRlZiBzY29yZShibGVuZDogbnAubmRhcnJheSkgLT4gZmxvYXQ6DQogICAgICAgIGlmIG1ldHJpYyA9PSAiYXVjIjoNCiAgICAgICAgICAgIGF1Y3MgPSBbXQ0KICAgICAgICAgICAgZm9yIGogaW4gcmFuZ2UoYmxlbmQuc2hhcGVbMV0pOg0KICAgICAgICAgICAgICAgIG0gPSBucC5pc2Zpbml0ZShvb2ZfbGFiZWxzWzosIGpdKQ0KICAgICAgICAgICAgICAgIGlmIG0uc3VtKCkgPiAxIGFuZCAwIDwgb29mX2xhYmVsc1ttLCBqXS5zdW0oKSA8IG0uc3VtKCk6DQogICAgICAgICAgICAgICAgICAgIGF1Y3MuYXBwZW5kKHJvY19hdWNfc2NvcmUob29mX2xhYmVsc1ttLCBqXSwgYmxlbmRbbSwgal0pKQ0KICAgICAgICAgICAgcmV0dXJuIGZsb2F0KG5wLm1lYW4oYXVjcykpIGlmIGF1Y3MgZWxzZSAwLjANCiAgICAgICAgZWxzZTogICMgQkNFDQogICAgICAgICAgICBwID0gbnAuY2xpcChibGVuZCwgMWUtNywgMSAtIDFlLTcpDQogICAgICAgICAgICBtID0gbnAuaXNmaW5pdGUob29mX2xhYmVscykNCiAgICAgICAgICAgIHJldHVybiAtZmxvYXQobnAubWVhbigNCiAgICAgICAgICAgICAgICAtKG9vZl9sYWJlbHNbbV0gKiBucC5sb2cocFttXSkgKyAoMSAtIG9vZl9sYWJlbHNbbV0pICogbnAubG9nKDEgLSBwW21dKSkNCiAgICAgICAgICAgICkpDQoNCiAgICBzZWxlY3RlZDogbGlzdFtpbnRdID0gW10NCiAgICBjdXJyZW50X2JsZW5kID0gbnAuemVyb3NfbGlrZShvb2ZfcHJlZHNbMF0pDQoNCiAgICBmb3Igc3RlcCBpbiByYW5nZShuX3NlbGVjdCk6DQogICAgICAgIGJlc3RfaWR4LCBiZXN0X3Njb3JlID0gLTEsIC1ucC5pbmYNCiAgICAgICAgZm9yIGksIHByZWQgaW4gZW51bWVyYXRlKG9vZl9wcmVkcyk6DQogICAgICAgICAgICBjYW5kaWRhdGUgPSAoY3VycmVudF9ibGVuZCAqIHN0ZXAgKyBwcmVkKSAvIChzdGVwICsgMSkNCiAgICAgICAgICAgIHMgPSBzY29yZShjYW5kaWRhdGUpDQogICAgICAgICAgICBpZiBzID4gYmVzdF9zY29yZToNCiAgICAgICAgICAgICAgICBiZXN0X3Njb3JlLCBiZXN0X2lkeCA9IHMsIGkNCiAgICAgICAgc2VsZWN0ZWQuYXBwZW5kKGJlc3RfaWR4KQ0KICAgICAgICBjdXJyZW50X2JsZW5kID0gKGN1cnJlbnRfYmxlbmQgKiBzdGVwICsgb29mX3ByZWRzW2Jlc3RfaWR4XSkgLyAoc3RlcCArIDEpDQoNCiAgICByZXR1cm4gc2VsZWN0ZWQNCg0KDQpkZWYgYmxlbmRfZnJvbV9zZWxlY3Rpb24ocHJlZHM6IGxpc3RbbnAubmRhcnJheV0sIHNlbGVjdGVkOiBsaXN0W2ludF0pIC0+IG5wLm5kYXJyYXk6DQogICAgIiIiQXZlcmFnZSBwcmVkaWN0aW9ucyBhY2NvcmRpbmcgdG8gYSBncmVlZHktc2VsZWN0ZWQgaW5kZXggbGlzdCAod2l0aCByZXBldGl0aW9uKS4iIiINCiAgICBjb3VudHMgPSBucC5iaW5jb3VudChzZWxlY3RlZCwgbWlubGVuZ3RoPWxlbihwcmVkcykpLmFzdHlwZShmbG9hdCkNCiAgICBibGVuZCA9IHN1bShwICogYyBmb3IgcCwgYyBpbiB6aXAocHJlZHMsIGNvdW50cykpDQogICAgcmV0dXJuIGJsZW5kIC8gY291bnRzLnN1bSgpDQoNCg0KIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgTi1Bcm0gUmFuayBFbnNlbWJsZSDilIDilIDilIDilIDilIDilIDilIDilIDilIANCiMgUGVyLXRhcmdldCBibGVuZGluZyB3ZWlnaHRzIGZvciB0aGUgZXhpc3RpbmcgdHdvLWFybSBzeXN0ZW0uDQojIEtleXM6IHRhcmdldCBuYW1lIChjb25maWcuVEFSR0VUUykuICBWYWx1ZXM6IGZyYWN0aW9uIGFzc2lnbmVkIHRvIENvQXROZXQgYXJtLg0KX0NPQVRfV0VJR0hUUzogZGljdFtzdHIsIGZsb2F0XSA9IHt0OiAwLjYwIGZvciB0IGluIGNvbmZpZy5UQVJHRVRTfQ0KX0NPQVRfV0VJR0hUUy51cGRhdGUoew0KICAgICJBQ0wiOiAwLjc1LA0KICAgICJNZWRpYWwgTWVuaXNjdXMiOiAwLjgwLA0KICAgICJMYXRlcmFsIE1lbmlzY3VzIjogMS4wMCwNCiAgICAiTGF0ZXJhbCBPQSI6IDAuNzUsDQogICAgIkZyYWN0dXJlIjogMC43NSwNCn0pDQoNCiMgVGhpcmQgYXJtIChDb252TmVYdCAvIFJhZEltYWdlTmV0LXByZXRyYWluZWQgQ05OKS4NCiMgVmVyaWZpZWQgYmFzaXM6IE1laSBldCBhbC4gUmFkSW1hZ2VOZXQgMjAyMiDigJQgQUNMICs0LjglLCBNZW5pc2N1cyArNC41JS4NCiMgQ29udk5lWHQgdGV4dHVyZSBmb2N1cyDihpIgc3Ryb25nIGZvciBsb2NhbGlzZWQgZmluZGluZ3MgKEZyYWN0dXJlLCBPQSkuDQojIFN0YXJ0IGNvbnNlcnZhdGl2ZWx5IGF0IDAuMTUgYWNyb3NzIGJvYXJkOyBpbmNyZWFzZSBmb3IgQUNML01lbmlzY3VzIGVtcGlyaWNhbGx5Lg0KX0NPTlZORVhUX1dFSUdIVFM6IGRpY3Rbc3RyLCBmbG9hdF0gPSB7dDogMC4xMCBmb3IgdCBpbiBjb25maWcuVEFSR0VUU30NCl9DT05WTkVYVF9XRUlHSFRTLnVwZGF0ZSh7DQogICAgIkFDTCI6IDAuMTUsDQogICAgIk1lZGlhbCBNZW5pc2N1cyI6IDAuMTUsDQogICAgIkxhdGVyYWwgTWVuaXNjdXMiOiAwLjIwLA0KICAgICJGcmFjdHVyZSI6IDAuMTUsDQogICAgIkNvbnR1c2lvbiI6IDAuMTAsDQp9KQ0KDQoNCmRlZiByYW5rX2Vuc2VtYmxlX24oDQogICAgZGZzOiBsaXN0W3R1cGxlW3N0ciwgcGQuRGF0YUZyYW1lXV0sDQogICAgY29hdF93ZWlnaHRzOiBkaWN0W3N0ciwgZmxvYXRdIHwgTm9uZSA9IE5vbmUsDQogICAgY29udm5leHRfd2VpZ2h0czogZGljdFtzdHIsIGZsb2F0XSB8IE5vbmUgPSBOb25lLA0KKSAtPiBwZC5EYXRhRnJhbWU6DQogICAgIiIiTi1hcm0gcmFuay1wZXJjZW50aWxlIGJsZW5kaW5nLiAgR2VuZXJhbGlzZXMgdGhlIGJhc2VsaW5lIDItYXJtIGJsZW5kLg0KDQogICAgUGFyYW1ldGVycw0KICAgIC0tLS0tLS0tLS0NCiAgICBkZnMgOiBsaXN0IG9mIChhcm1fbmFtZSwgRGF0YUZyYW1lKSBwYWlycy4NCiAgICAgICAgRWFjaCBEYXRhRnJhbWUgbXVzdCBoYXZlICdTdHVkeUluc3RhbmNlVUlEJyArIGNvbmZpZy5UQVJHRVRTIGNvbHVtbnMuDQogICAgICAgIFN1cHBvcnRlZCBhcm0gbmFtZXM6ICdkaW5vJywgJ2NvYXRuZXQnLCAnY29udm5leHQnLiBPdGhlcnMgdHJlYXRlZCBhcw0KICAgICAgICBhZGRpdGlvbmFsIERJTk92Mi1zdHlsZSBhcm1zIGFuZCBhZGRlZCB0byB0aGUgRElOT3YyIHdlaWdodC4NCiAgICBjb2F0X3dlaWdodHMgLyBjb252bmV4dF93ZWlnaHRzIDogb3B0aW9uYWwgb3ZlcnJpZGUgZGljdHMuDQoNCiAgICBSZXR1cm5zDQogICAgLS0tLS0tLQ0KICAgIERhdGFGcmFtZSB3aXRoICdTdHVkeUluc3RhbmNlVUlEJyArIGNvbmZpZy5UQVJHRVRTIGFzIHJhbmstcGVyY2VudGlsZSBibGVuZC4NCiAgICAiIiINCiAgICBjdyA9IGNvYXRfd2VpZ2h0cyBvciBfQ09BVF9XRUlHSFRTDQogICAgeHcgPSBjb252bmV4dF93ZWlnaHRzIG9yIF9DT05WTkVYVF9XRUlHSFRTDQoNCiAgICAjIEFsaWduIG9uIHRoZSBmaXJzdCBhcm0ncyBzdHVkeSBvcmRlcg0KICAgIHJlZl91aWQgPSBkZnNbMF1bMV1bIlN0dWR5SW5zdGFuY2VVSUQiXS50b2xpc3QoKQ0KICAgIHJhbmtzOiBkaWN0W3N0ciwgcGQuRGF0YUZyYW1lXSA9IHt9DQogICAgZm9yIGFybV9uYW1lLCBkZiBpbiBkZnM6DQogICAgICAgIGFsaWduZWQgPSBkZi5zZXRfaW5kZXgoIlN0dWR5SW5zdGFuY2VVSUQiKS5yZWluZGV4KHJlZl91aWQpLnJlc2V0X2luZGV4KCkNCiAgICAgICAgciA9IGFsaWduZWRbY29uZmlnLlRBUkdFVFNdLnJhbmsobWV0aG9kPSJhdmVyYWdlIiwgcGN0PVRydWUpDQogICAgICAgIHIuaW5kZXggPSByZWZfdWlkDQogICAgICAgIHJhbmtzW2FybV9uYW1lXSA9IHINCg0KICAgIGJsZW5kID0gcGQuRGF0YUZyYW1lKGluZGV4PXJlZl91aWQsIGNvbHVtbnM9Y29uZmlnLlRBUkdFVFMsIGR0eXBlPWZsb2F0KQ0KICAgIGZvciB0IGluIGNvbmZpZy5UQVJHRVRTOg0KICAgICAgICBjb2F0X3cgPSBjdy5nZXQodCwgMC42MCkNCiAgICAgICAgY29udl93ID0geHcuZ2V0KHQsIDAuMTApDQogICAgICAgIGRpbm9fdG90YWwgPSAxLjAgLSBjb2F0X3cgLSBjb252X3cgICMgcmVtYWluaW5nIHdlaWdodCB0byBESU5PdjIgYXJtcw0KDQogICAgICAgICMgU3VtIERJTk92Mi10eXBlIGFybSBjb250cmlidXRpb25zIChoYW5kbGVzIDUtZm9sZCBhdmVyYWdlcykNCiAgICAgICAgZGlub19jb250cmliID0gMC4wDQogICAgICAgIG5fZGlubyA9IDANCiAgICAgICAgZm9yIGFybV9uYW1lLCBfIGluIGRmczoNCiAgICAgICAgICAgIGlmIGFybV9uYW1lIG5vdCBpbiAoImNvYXRuZXQiLCAiY29udm5leHQiKToNCiAgICAgICAgICAgICAgICBkaW5vX2NvbnRyaWIgKz0gcmFua3NbYXJtX25hbWVdW3RdDQogICAgICAgICAgICAgICAgbl9kaW5vICs9IDENCiAgICAgICAgaWYgbl9kaW5vID4gMDoNCiAgICAgICAgICAgIGRpbm9fY29udHJpYiA9IChkaW5vX2NvbnRyaWIgLyBuX2Rpbm8pICogZGlub190b3RhbA0KDQogICAgICAgIGNvYXRfY29udHJpYiA9IHJhbmtzLmdldCgiY29hdG5ldCIsIHJhbmtzW2Rmc1swXVswXV0pW3RdICogY29hdF93IGlmICJjb2F0bmV0IiBpbiByYW5rcyBlbHNlIDAuMA0KICAgICAgICBjb252X2NvbnRyaWIgPSByYW5rcy5nZXQoImNvbnZuZXh0IiwgcmFua3NbZGZzWzBdWzBdXSlbdF0gKiBjb252X3cgaWYgImNvbnZuZXh0IiBpbiByYW5rcyBlbHNlIDAuMA0KDQogICAgICAgIGJsZW5kW3RdID0gZGlub19jb250cmliICsgY29hdF9jb250cmliICsgY29udl9jb250cmliDQoNCiAgICAjIEZpbmFsIHJlLXJhbmsgdG8gdW5pZm9ybSBbMCwgMV0gcGVyY2VudGlsZSBzcGFjZQ0KICAgIGJsZW5kID0gYmxlbmQucmFuayhtZXRob2Q9ImF2ZXJhZ2UiLCBwY3Q9VHJ1ZSkNCiAgICByZXN1bHQgPSBwZC5EYXRhRnJhbWUoeyJTdHVkeUluc3RhbmNlVUlEIjogcmVmX3VpZH0pDQogICAgZm9yIHQgaW4gY29uZmlnLlRBUkdFVFM6DQogICAgICAgIHJlc3VsdFt0XSA9IGJsZW5kW3RdLnZhbHVlcw0KICAgIHJldHVybiByZXN1bHQNCg==',
    'src/modeling/losses.py': 'IiIiTG9zcyBmdW5jdGlvbnMgZm9yIG11bHRpLWxhYmVsIHRyYWluaW5nLgoKSW1wbGVtZW50YXRpb25zCi0tLS0tLS0tLS0tLS0tLQpBc3ltbWV0cmljTG9zcyAgUmlkbmlrIGV0IGFsLiwgSUNDViAyMDIxIChhclhpdjoyMDA5LjE0MTE5KS4KICAgICAgICAgICAgICAgIERlLWZhY3RvIFNPVEEgZm9yIG11bHRpLWxhYmVsIGNsYXNzaWZpY2F0aW9uLgogICAgICAgICAgICAgICAgT3V0cGVyZm9ybXMgQkNFIG9uIE1TLUNPQ08gKCsyLjVwcCBtQVApLCBPcGVuIEltYWdlcyAoKzEuNHBwKSwKICAgICAgICAgICAgICAgIGFuZCBGb2NhbCBMb3NzIG9uIGV2ZXJ5IHRlc3RlZCBiZW5jaG1hcmsuCiAgICAgICAgICAgICAgICBNZWNoYW5pc206IGRlY291cGxlZCBnYW1tYSsvZ2FtbWEtIGZvY3VzaW5nICsgcHJvYmFiaWxpdHktc2hpZnQKICAgICAgICAgICAgICAgIHRoYXQgZGlzY2FyZHMgZWFzeS9taXNsYWJlbGxlZCBuZWdhdGl2ZXMgKGRpcmVjdGx5IHJlbGV2YW50IHRvIG91cgogICAgICAgICAgICAgICAgTkxQLWRlcml2ZWQgbm9pc3kgbGFiZWxzIHdoZXJlIHNvbWUgbmVnYXRpdmUgbGFiZWxzIGFyZSB1bnJlbGlhYmxlKS4KCiAgICAgICAgICAgICAgICBVUEdSQURFICh2Myk6IGdhbW1hX25lZyBub3cgYWNjZXB0cyBhIHBlci10YXJnZXQgbGlzdC90ZW5zb3IgW0NdLgogICAgICAgICAgICAgICAgVXNpbmcgcGVyLXRhcmdldCBnYW1tYXMgYWxpZ25zIHRoZSBuZWdhdGl2ZS1zdXBwcmVzc2lvbiBzdHJlbmd0aAogICAgICAgICAgICAgICAgd2l0aCBlbXBpcmljYWwgbGFiZWwgcHJldmFsZW5jZToKICAgICAgICAgICAgICAgICAgLSBSYXJlIHBvc2l0aXZlcyAoTUNMIDE1JSwgQmFrZXIncyAyMSUpIGdldCBoaWdoIGdhbW1hX25lZyAoNC4wKQogICAgICAgICAgICAgICAgICAgIHNvIHRoZSBsb3NzIGZvY3VzZXMgaGFyZCBvbiB0aGVpciByYXJlIHBvc2l0aXZlIHNhbXBsZXMuCiAgICAgICAgICAgICAgICAgIC0gQ29tbW9uIHBvc2l0aXZlcyAoRWZmdXNpb24gNjAlKSBnZXQgbG93IGdhbW1hX25lZyAoMS4wKQogICAgICAgICAgICAgICAgICAgIHNvIHdlIGRvIG5vdCBzdXBwcmVzcyB0aGUgaW5mb3JtYXRpdmUgbmVnYXRpdmUgZ3JhZGllbnQuCgpXZWlnaHRlZEJDRSAgICAgTGVnYWN5IGZvciBBL0IgYWJsYXRpb247IGtlcHQgYXMgYSBkcm9wLWluIGFsdGVybmF0aXZlLgoKUmVmZXJlbmNlOiBodHRwczovL2dpdGh1Yi5jb20vQWxpYmFiYS1NSUlML0FTTAoiIiIKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgoKY2xhc3MgQXN5bW1ldHJpY0xvc3Mobm4uTW9kdWxlKToKICAgICIiIlJpZG5payBldCBhbC4sIElDQ1YgMjAyMS4KCiAgICBQYXJhbWV0ZXJzCiAgICAtLS0tLS0tLS0tCiAgICBnYW1tYV9uZWcgOiBmbG9hdCBvciBsaXN0W2Zsb2F0XQogICAgICAgIEZvY3VzaW5nIGV4cG9uZW50IGZvciBuZWdhdGl2ZXMuICBTY2FsYXI6IGFwcGxpZWQgdW5pZm9ybWx5IGFjcm9zcyBhbGwKICAgICAgICBjbGFzc2VzLiAgTGlzdFtDXTogcGVyLXRhcmdldCBnYW1tYSwgZW5hYmxpbmcgcHJldmFsZW5jZS1tYXRjaGVkIGZvY3VzaW5nCiAgICAgICAgKGhpZ2hlciBnYW1tYSBmb3IgcmFyZSB0YXJnZXRzLCBsb3dlciBmb3IgY29tbW9uIG9uZXMpLgogICAgZ2FtbWFfcG9zIDogZmxvYXQKICAgICAgICBGb2N1c2luZyBleHBvbmVudCBmb3IgcG9zaXRpdmVzLiBEZWZhdWx0IDAg4oCUIG5ldmVyIGRvd24td2VpZ2h0IHRydWUKICAgICAgICBwb3NpdGl2ZXMgKGNyaXRpY2FsIGZvciByYXJlLCBoaWdoLXN0YWtlcyBmaW5kaW5ncykuCiAgICBjbGlwIDogZmxvYXQKICAgICAgICBQcm9iYWJpbGl0eSBzaGlmdCBmb3IgbmVnYXRpdmVzLiBFbnRyaWVzIHdoZXJlIHBfbmVnIDwgY2xpcCBhcmUgemVyb2VkLAogICAgICAgIGRpc2NhcmRpbmcgdHJpdmlhbGx5IGVhc3kgLyBsaWtlbHkgbWlzbGFiZWxsZWQgbmVnYXRpdmVzLgogICAgICAgIFNldCBjbGlwPTAuMCB0byBkaXNhYmxlIChkZWdlbmVyYXRlcyB0byBhc3ltbWV0cmljIEZvY2FsIExvc3MpLgogICAgZXBzIDogZmxvYXQKICAgICAgICBMb2ctc3RhYmlsaXR5IGVwc2lsb24uCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZ2FtbWFfbmVnPTIuMCwgZ2FtbWFfcG9zOiBmbG9hdCA9IDAuMCwKICAgICAgICAgICAgICAgICBjbGlwOiBmbG9hdCA9IDAuMDUsIGVwczogZmxvYXQgPSAxZS04KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiBpc2luc3RhbmNlKGdhbW1hX25lZywgKGxpc3QsIHR1cGxlKSk6CiAgICAgICAgICAgICMgUmVnaXN0ZXIgYXMgYnVmZmVyIHNvIGl0IG1vdmVzIHRvIEdQVSB3aXRoIC50byhkZXZpY2UpIGF1dG9tYXRpY2FsbHkKICAgICAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoCiAgICAgICAgICAgICAgICAiZ2FtbWFfbmVnX3ZlYyIsCiAgICAgICAgICAgICAgICB0b3JjaC50ZW5zb3IoZ2FtbWFfbmVnLCBkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgICAgICAgICApCiAgICAgICAgICAgIHNlbGYuX2dhbW1hX25lZ19zY2FsYXIgPSBOb25lCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5nYW1tYV9uZWdfdmVjID0gTm9uZQogICAgICAgICAgICBzZWxmLl9nYW1tYV9uZWdfc2NhbGFyID0gZmxvYXQoZ2FtbWFfbmVnKQogICAgICAgIHNlbGYuZ2FtbWFfcG9zID0gZmxvYXQoZ2FtbWFfcG9zKQogICAgICAgIHNlbGYuY2xpcCA9IGZsb2F0KGNsaXApCiAgICAgICAgc2VsZi5lcHMgPSBlcHMKCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICBsb2dpdHM6IHRvcmNoLlRlbnNvciwgICAgICAgICAgIyBbQiwgQ10gcmF3IChwcmUtc2lnbW9pZCkKICAgICAgICB0YXJnZXRzOiB0b3JjaC5UZW5zb3IsICAgICAgICAgIyBbQiwgQ10gaW4gWzAsIDFdOyBzb2Z0IGxhYmVscyBPSwogICAgICAgIHdlaWdodHM6IHRvcmNoLlRlbnNvciB8IE5vbmUgPSBOb25lLCAgIyBbQiwgQ10gcGVyLWVudHJ5IHdlaWdodAogICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgeHNfcG9zID0gdG9yY2guc2lnbW9pZChsb2dpdHMpCiAgICAgICAgeHNfbmVnID0gMS4wIC0geHNfcG9zCgogICAgICAgICMgUHJvYmFiaWxpdHkgc2hpZnQg4oCUIGtpbGxzIGVhc3kgbmVnYXRpdmVzICYgbWlzbGFiZWxsZWQgZW50cmllcwogICAgICAgIGlmIHNlbGYuY2xpcCA+IDA6CiAgICAgICAgICAgIHhzX25lZyA9ICh4c19uZWcgKyBzZWxmLmNsaXApLmNsYW1wKG1heD0xLjApCgogICAgICAgICMgTG9nIGxvc3MgdGVybXMKICAgICAgICBsb3NzX3BvcyA9IHRhcmdldHMgICAgICAgKiB0b3JjaC5sb2coeHNfcG9zLmNsYW1wKG1pbj1zZWxmLmVwcykpCiAgICAgICAgbG9zc19uZWcgPSAoMS4wIC0gdGFyZ2V0cykgKiB0b3JjaC5sb2coeHNfbmVnLmNsYW1wKG1pbj1zZWxmLmVwcykpCiAgICAgICAgbG9zcyA9IGxvc3NfcG9zICsgbG9zc19uZWcgICAjIFtCLCBDXQoKICAgICAgICAjIEFzeW1tZXRyaWMgZm9jdXNpbmcg4oCUIGRvd24td2VpZ2h0IGVhc3kgZXhhbXBsZXMgcGVyIGNsYXNzCiAgICAgICAgIyBnYW1tYV9uZWc6IHNjYWxhciBvciBbQ10gdGVuc29yIChicm9hZGNhc3RzIGNvcnJlY3RseSBlaXRoZXIgd2F5KQogICAgICAgICMgU09UQSBCdWdmaXg6IGVuc3VyZSBnX25lZyBhbHdheXMgbWF0Y2hlcyB0YXJnZXRzLmRldmljZSBhbmQgdGFyZ2V0cy5kdHlwZQogICAgICAgIGlmIHNlbGYuZ2FtbWFfbmVnX3ZlYyBpcyBub3QgTm9uZToKICAgICAgICAgICAgZ19uZWcgPSBzZWxmLmdhbW1hX25lZ192ZWMudG8oZGV2aWNlPXRhcmdldHMuZGV2aWNlLCBkdHlwZT10YXJnZXRzLmR0eXBlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGdfbmVnID0gbG9naXRzLm5ld19mdWxsKCgxLCksIHNlbGYuX2dhbW1hX25lZ19zY2FsYXIsIGRldmljZT10YXJnZXRzLmRldmljZSwgZHR5cGU9dGFyZ2V0cy5kdHlwZSkKICAgICAgICBwdCAgICA9IHhzX3BvcyAqIHRhcmdldHMgKyB4c19uZWcgKiAoMS4wIC0gdGFyZ2V0cykgICAjIFtCLCBDXQogICAgICAgIGdhbW1hID0gc2VsZi5nYW1tYV9wb3MgKiB0YXJnZXRzICsgZ19uZWcgKiAoMS4wIC0gdGFyZ2V0cykgICMgW0IsIENdCiAgICAgICAgbG9zcyAgPSBsb3NzICogKCgxLjAgLSBwdCkgKiogZ2FtbWEpCgogICAgICAgIGxvc3MgPSAtbG9zcyAgIyBtaW5pbWlzZQoKICAgICAgICBpZiB3ZWlnaHRzIGlzIG5vdCBOb25lOgogICAgICAgICAgICBsb3NzID0gbG9zcyAqIHdlaWdodHMKICAgICAgICAgICAgIyBDb21wdXRlIGxvc3MgaW5kZXBlbmRlbnRseSBwZXIgY2xhc3MgZmlyc3QsIHRoZW4gYXZlcmFnZSBhY3Jvc3MgYWxsIEMgdGFyZ2V0cyAoMS9DIHBlciB0YXJnZXQpLgogICAgICAgICAgICAjIFRoaXMgYm91bmRzIGVhY2ggdGFyZ2V0IGhlYWQncyBncmFkaWVudCBjb250cmlidXRpb24gYXQgMS9DLCBwcmV2ZW50aW5nIHNwYXJzZSBiYXRjaGVzCiAgICAgICAgICAgICMgZnJvbSBjYXVzaW5nIGRlc3RydWN0aXZlIGdyYWRpZW50IHNwaWtlcyAodXAgdG8gMTJ4KSBvbiB0aGUgc2hhcmVkIFZpVCBiYWNrYm9uZS4KICAgICAgICAgICAgY2xhc3NfbG9zc2VzID0gbG9zcy5zdW0oZGltPTApIC8gd2VpZ2h0cy5zdW0oZGltPTApLmNsYW1wX21pbigxLjApCiAgICAgICAgICAgIHJldHVybiBjbGFzc19sb3NzZXMubWVhbigpCiAgICAgICAgcmV0dXJuIGxvc3MubWVhbigpCgoKY2xhc3MgV2VpZ2h0ZWRCQ0Uobm4uTW9kdWxlKToKICAgICIiIkxlZ2FjeSB3ZWlnaHRlZCBCQ0Ug4oCUIGtlcHQgZm9yIGFibGF0aW9uIGNvbXBhcmlzb25zLiIiIgoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGxvZ2l0cywgdGFyZ2V0cywgd2VpZ2h0cz1Ob25lKToKICAgICAgICBsb3NzID0gRi5iaW5hcnlfY3Jvc3NfZW50cm9weV93aXRoX2xvZ2l0cyhsb2dpdHMuZmxvYXQoKSwgdGFyZ2V0cywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByZWR1Y3Rpb249Im5vbmUiKQogICAgICAgIGlmIHdlaWdodHMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGxvc3MgPSBsb3NzICogd2VpZ2h0cwogICAgICAgICAgICAjIENvbXB1dGUgbG9zcyBpbmRlcGVuZGVudGx5IHBlciBjbGFzcyBmaXJzdCwgdGhlbiBhdmVyYWdlIGFjcm9zcyBhbGwgQyB0YXJnZXRzICgxL0MgcGVyIHRhcmdldCkuCiAgICAgICAgICAgICMgVGhpcyBib3VuZHMgZWFjaCB0YXJnZXQgaGVhZCdzIGdyYWRpZW50IGNvbnRyaWJ1dGlvbiBhdCAxL0MsIHByZXZlbnRpbmcgc3BhcnNlIGJhdGNoZXMKICAgICAgICAgICAgIyBmcm9tIGNhdXNpbmcgZGVzdHJ1Y3RpdmUgZ3JhZGllbnQgc3Bpa2VzICh1cCB0byAxMngpIG9uIHRoZSBzaGFyZWQgVmlUIGJhY2tib25lLgogICAgICAgICAgICBjbGFzc19sb3NzZXMgPSBsb3NzLnN1bShkaW09MCkgLyB3ZWlnaHRzLnN1bShkaW09MCkuY2xhbXBfbWluKDEuMCkKICAgICAgICAgICAgcmV0dXJuIGNsYXNzX2xvc3Nlcy5tZWFuKCkKICAgICAgICByZXR1cm4gbG9zcy5tZWFuKCkKCgpMT1NTX1JFR0lTVFJZOiBkaWN0W3N0ciwgdHlwZV0gPSB7ImFzbCI6IEFzeW1tZXRyaWNMb3NzLCAiYmNlIjogV2VpZ2h0ZWRCQ0V9CgoKZGVmIGJ1aWxkX2xvc3MobmFtZTogc3RyID0gImFzbCIsICoqa3dhcmdzKSAtPiBubi5Nb2R1bGU6CiAgICAiIiJGYWN0b3J5LiAgbmFtZSBpbiB7J2FzbCcsICdiY2UnfS4gIEV4dHJhIGt3YXJncyBmb3J3YXJkZWQgdG8gX19pbml0X18uIiIiCiAgICBpZiBuYW1lIG5vdCBpbiBMT1NTX1JFR0lTVFJZOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmtub3duIGxvc3MgJ3tuYW1lfScuIEF2YWlsYWJsZToge2xpc3QoTE9TU19SRUdJU1RSWSl9IikKICAgIHJldHVybiBMT1NTX1JFR0lTVFJZW25hbWVdKCoqa3dhcmdzKQo=',
    'src/modeling/model.py': '...',
    'src/modeling/__init__.py': '',
    'src/training/train.py': '...',
    'src/training/__init__.py': '',
    'src/modeling/mil.py': 'IiIiVGltbSBpbWFnZSBlbmNvZGVyIHdpdGggdGFyZ2V0LXNwZWNpZmljIGF0dGVudGlvbiBwb29saW5nIG92ZXIgc3R1ZHkgd2luZG93cy4iIiINCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4gYXMgbm4NCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYNCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCg0KDQpjbGFzcyBUaW1tQXR0ZW50aW9uTUlMKG5uLk1vZHVsZSk6DQogICAgIiIiRW5jb2RlIHZhbGlkIHNsb3Qgd2luZG93cyBhbmQgcG9vbCB0aGVtIHdpdGggb25lIGF0dGVudGlvbiBkaXN0cmlidXRpb24gcGVyIGZpbmRpbmcuIiIiDQoNCiAgICBkZWYgX19pbml0X18oDQogICAgICAgIHNlbGYsDQogICAgICAgIGJhY2tib25lOiBubi5Nb2R1bGUsDQogICAgICAgIGZlYXR1cmVfZGltOiBpbnQsDQogICAgICAgIGlucHV0X3NpemU6IGludCA9IGNvbmZpZy5DT0FUTkVUX0lOUFVUX1NJWkUsDQogICAgICAgIGVuY29kZV9jaHVua19zaXplOiBpbnQgPSBjb25maWcuQ09BVE5FVF9FTkNPREVfQ0hVTkssDQogICAgKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIGlmIGlucHV0X3NpemUgPD0gMCBvciBlbmNvZGVfY2h1bmtfc2l6ZSA8PSAwOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW5wdXRfc2l6ZSBhbmQgZW5jb2RlX2NodW5rX3NpemUgbXVzdCBiZSBwb3NpdGl2ZSIpDQogICAgICAgIHNlbGYuYmFja2JvbmUgPSBiYWNrYm9uZQ0KICAgICAgICBzZWxmLmZlYXR1cmVfZGltID0gaW50KGZlYXR1cmVfZGltKQ0KICAgICAgICBzZWxmLmlucHV0X3NpemUgPSBpbnQoaW5wdXRfc2l6ZSkNCiAgICAgICAgc2VsZi5lbmNvZGVfY2h1bmtfc2l6ZSA9IGludChlbmNvZGVfY2h1bmtfc2l6ZSkNCiAgICAgICAgc2VsZi5tb2RlbF90eXBlID0gImNvYXRuZXRfbWlsIg0KICAgICAgICBzZWxmLm5vcm0gPSBubi5MYXllck5vcm0oc2VsZi5mZWF0dXJlX2RpbSkNCiAgICAgICAgc2VsZi5hdHRlbnRpb24gPSBubi5TZXF1ZW50aWFsKA0KICAgICAgICAgICAgbm4uTGluZWFyKHNlbGYuZmVhdHVyZV9kaW0sIDI1NiksDQogICAgICAgICAgICBubi5UYW5oKCksDQogICAgICAgICAgICBubi5Ecm9wb3V0KDAuMiksDQogICAgICAgICAgICBubi5MaW5lYXIoMjU2LCBsZW4oY29uZmlnLlRBUkdFVFMpKSwNCiAgICAgICAgKQ0KICAgICAgICBzZWxmLmNsYXNzaWZpZXIgPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkobGVuKGNvbmZpZy5UQVJHRVRTKSwgc2VsZi5mZWF0dXJlX2RpbSkpDQogICAgICAgIHNlbGYuYmlhcyA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcyhsZW4oY29uZmlnLlRBUkdFVFMpKSkNCiAgICAgICAgbm4uaW5pdC50cnVuY19ub3JtYWxfKHNlbGYuY2xhc3NpZmllciwgc3RkPTAuMDIpDQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKA0KICAgICAgICAgICAgIm1lYW4iLA0KICAgICAgICAgICAgdG9yY2gudGVuc29yKFswLjQ4NSwgMC40NTYsIDAuNDA2XSkudmlldygxLCAzLCAxLCAxKSwNCiAgICAgICAgKQ0KICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigNCiAgICAgICAgICAgICJzdGQiLA0KICAgICAgICAgICAgdG9yY2gudGVuc29yKFswLjIyOSwgMC4yMjQsIDAuMjI1XSkudmlldygxLCAzLCAxLCAxKSwNCiAgICAgICAgKQ0KDQogICAgZGVmIF9lbmNvZGUoc2VsZiwgaW1hZ2VzOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoNCiAgICAgICAgZmVhdHVyZXMgPSBbXQ0KICAgICAgICBwYXJhbWV0ZXIgPSBuZXh0KHNlbGYuYmFja2JvbmUucGFyYW1ldGVycygpLCBOb25lKQ0KICAgICAgICBkdHlwZSA9IHBhcmFtZXRlci5kdHlwZSBpZiBwYXJhbWV0ZXIgaXMgbm90IE5vbmUgZWxzZSBpbWFnZXMuZHR5cGUNCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbihpbWFnZXMpLCBzZWxmLmVuY29kZV9jaHVua19zaXplKToNCiAgICAgICAgICAgIGJhdGNoID0gaW1hZ2VzW3N0YXJ0OnN0YXJ0ICsgc2VsZi5lbmNvZGVfY2h1bmtfc2l6ZV0NCiAgICAgICAgICAgIGlmIGJhdGNoLnNoYXBlWy0yOl0gIT0gKHNlbGYuaW5wdXRfc2l6ZSwgc2VsZi5pbnB1dF9zaXplKToNCiAgICAgICAgICAgICAgICBiYXRjaCA9IEYuaW50ZXJwb2xhdGUoDQogICAgICAgICAgICAgICAgICAgIGJhdGNoLA0KICAgICAgICAgICAgICAgICAgICBzaXplPShzZWxmLmlucHV0X3NpemUsIHNlbGYuaW5wdXRfc2l6ZSksDQogICAgICAgICAgICAgICAgICAgIG1vZGU9ImJpbGluZWFyIiwNCiAgICAgICAgICAgICAgICAgICAgYWxpZ25fY29ybmVycz1GYWxzZSwNCiAgICAgICAgICAgICAgICApDQogICAgICAgICAgICBiYXRjaCA9IGJhdGNoLnRvKGR0eXBlPWR0eXBlKQ0KICAgICAgICAgICAgYmF0Y2ggPSAoYmF0Y2ggLSBzZWxmLm1lYW4udG8oYmF0Y2guZHR5cGUpKSAvIHNlbGYuc3RkLnRvKGJhdGNoLmR0eXBlKQ0KICAgICAgICAgICAgaWYgc2VsZi50cmFpbmluZzoNCiAgICAgICAgICAgICAgICBiYXRjaC5yZXF1aXJlc19ncmFkXyhUcnVlKQ0KICAgICAgICAgICAgZW5jb2RlZCA9IHNlbGYuYmFja2JvbmUoYmF0Y2gpDQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKGVuY29kZWQsICh0dXBsZSwgbGlzdCkpOg0KICAgICAgICAgICAgICAgIGVuY29kZWQgPSBlbmNvZGVkWzBdDQogICAgICAgICAgICBpZiBlbmNvZGVkLm5kaW0gPiAyOg0KICAgICAgICAgICAgICAgIGVuY29kZWQgPSBlbmNvZGVkLmZsYXR0ZW4oMikubWVhbigtMSkNCiAgICAgICAgICAgIGlmIGVuY29kZWQubmRpbSAhPSAyIG9yIGVuY29kZWQuc2hhcGVbMV0gIT0gc2VsZi5mZWF0dXJlX2RpbToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoDQogICAgICAgICAgICAgICAgICAgIGYidGltbSBiYWNrYm9uZSByZXR1cm5lZCB7dHVwbGUoZW5jb2RlZC5zaGFwZSl9OyBleHBlY3RlZCAiDQogICAgICAgICAgICAgICAgICAgIGYiW04sIHtzZWxmLmZlYXR1cmVfZGltfV0gcG9vbGVkIGZlYXR1cmVzIg0KICAgICAgICAgICAgICAgICkNCiAgICAgICAgICAgIGZlYXR1cmVzLmFwcGVuZChlbmNvZGVkKQ0KICAgICAgICByZXR1cm4gdG9yY2guY2F0KGZlYXR1cmVzLCBkaW09MCkNCg0KICAgIGRlZiBmb3J3YXJkKA0KICAgICAgICBzZWxmLA0KICAgICAgICBpbWdzOiB0b3JjaC5UZW5zb3IsDQogICAgICAgIHNsb3RfbWFzazogdG9yY2guVGVuc29yLA0KICAgICAgICB3aW5kb3dfbWFzazogdG9yY2guVGVuc29yLA0KICAgICkgLT4gdG9yY2guVGVuc29yOg0KICAgICAgICBpZiBpbWdzLm5kaW0gIT0gNjoNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHBlY3RlZCBpbWFnZXMgW0IsUyxXLDMsSCxXXSwgcmVjZWl2ZWQge3R1cGxlKGltZ3Muc2hhcGUpfSIpDQogICAgICAgIGJhdGNoX3NpemUsIHNsb3RzLCB3aW5kb3dzID0gaW1ncy5zaGFwZVs6M10NCiAgICAgICAgdmFsaWQgPSAoc2xvdF9tYXNrLnVuc3F1ZWV6ZSgtMSkgPiAwLjUpICYgKHdpbmRvd19tYXNrID4gMC41KQ0KICAgICAgICBmbGF0X2ltYWdlcyA9IGltZ3MucmVzaGFwZShiYXRjaF9zaXplICogc2xvdHMgKiB3aW5kb3dzLCAqaW1ncy5zaGFwZVszOl0pDQogICAgICAgIHZhbGlkX2luZGljZXMgPSB2YWxpZC5yZXNoYXBlKC0xKS5ub256ZXJvKGFzX3R1cGxlPUZhbHNlKS5mbGF0dGVuKCkNCg0KICAgICAgICBmZWF0dXJlX2R0eXBlID0gc2VsZi5ub3JtLndlaWdodC5kdHlwZQ0KICAgICAgICBmbGF0X2ZlYXR1cmVzID0gaW1ncy5uZXdfemVyb3MoDQogICAgICAgICAgICAoYmF0Y2hfc2l6ZSAqIHNsb3RzICogd2luZG93cywgc2VsZi5mZWF0dXJlX2RpbSksIGR0eXBlPWZlYXR1cmVfZHR5cGUNCiAgICAgICAgKQ0KICAgICAgICBpZiB2YWxpZF9pbmRpY2VzLm51bWVsKCk6DQogICAgICAgICAgICBzZWxlY3RlZCA9IGZsYXRfaW1hZ2VzLmluZGV4X3NlbGVjdCgwLCB2YWxpZF9pbmRpY2VzKS5mbG9hdCgpLmRpdigyNTUuMCkNCiAgICAgICAgICAgIGVuY29kZWQgPSBzZWxmLl9lbmNvZGUoc2VsZWN0ZWQpDQogICAgICAgICAgICBmbGF0X2ZlYXR1cmVzID0gZmxhdF9mZWF0dXJlcy5pbmRleF9jb3B5KA0KICAgICAgICAgICAgICAgIDAsIHZhbGlkX2luZGljZXMsIGVuY29kZWQudG8oZmVhdHVyZV9kdHlwZSkNCiAgICAgICAgICAgICkNCg0KICAgICAgICBmZWF0dXJlcyA9IHNlbGYubm9ybSgNCiAgICAgICAgICAgIGZsYXRfZmVhdHVyZXMudmlldyhiYXRjaF9zaXplLCBzbG90cyAqIHdpbmRvd3MsIHNlbGYuZmVhdHVyZV9kaW0pDQogICAgICAgICkNCiAgICAgICAgdmFsaWRfd2luZG93cyA9IHZhbGlkLnJlc2hhcGUoYmF0Y2hfc2l6ZSwgc2xvdHMgKiB3aW5kb3dzKQ0KICAgICAgICBzY29yZXMgPSBzZWxmLmF0dGVudGlvbihmZWF0dXJlcykNCiAgICAgICAgc2NvcmVzID0gc2NvcmVzLm1hc2tlZF9maWxsKH52YWxpZF93aW5kb3dzLnVuc3F1ZWV6ZSgtMSksIC0xMDAwMC4wKQ0KICAgICAgICBhdHRlbnRpb24gPSBzY29yZXMuc29mdG1heChkaW09MSkNCiAgICAgICAgcG9vbGVkID0gdG9yY2guZWluc3VtKCJibmMsYm5mLT5iY2YiLCBhdHRlbnRpb24sIGZlYXR1cmVzKQ0KICAgICAgICByZXR1cm4gKHBvb2xlZCAqIHNlbGYuY2xhc3NpZmllci51bnNxdWVlemUoMCkpLnN1bSgtMSkgKyBzZWxmLmJpYXMNCg0KDQpkZWYgYnVpbGRfdGltbV9hdHRlbnRpb25fbWlsKA0KICAgIGFyY2g6IHN0ciwNCiAgICBwcmV0cmFpbmVkOiBib29sID0gRmFsc2UsDQogICAgaW5wdXRfc2l6ZTogaW50ID0gY29uZmlnLkNPQVRORVRfSU5QVVRfU0laRSwNCiAgICBlbmNvZGVfY2h1bmtfc2l6ZTogaW50ID0gY29uZmlnLkNPQVRORVRfRU5DT0RFX0NIVU5LLA0KKSAtPiBUaW1tQXR0ZW50aW9uTUlMOg0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IHRpbW0NCiAgICBleGNlcHQgSW1wb3J0RXJyb3IgYXMgZXhjOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoZSB0aW1tIHBhY2thZ2UgaXMgcmVxdWlyZWQgZm9yIHRoZSBDb0F0TmV0IE1JTCBtb2RlbCIpIGZyb20gZXhjDQoNCiAgICBiYWNrYm9uZSA9IHRpbW0uY3JlYXRlX21vZGVsKA0KICAgICAgICBhcmNoLA0KICAgICAgICBwcmV0cmFpbmVkPXByZXRyYWluZWQsDQogICAgICAgIG51bV9jbGFzc2VzPTAsDQogICAgICAgIGdsb2JhbF9wb29sPSJhdmciLA0KICAgICAgICBpbl9jaGFucz0zLA0KICAgICkNCiAgICBpZiBoYXNhdHRyKGJhY2tib25lLCAic2V0X2dyYWRfY2hlY2twb2ludGluZyIpOg0KICAgICAgICBiYWNrYm9uZS5zZXRfZ3JhZF9jaGVja3BvaW50aW5nKFRydWUpDQogICAgZmVhdHVyZV9kaW0gPSBnZXRhdHRyKGJhY2tib25lLCAibnVtX2ZlYXR1cmVzIiwgTm9uZSkNCiAgICBpZiBub3QgZmVhdHVyZV9kaW06DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ0aW1tIG1vZGVsIHthcmNoIXJ9IGRvZXMgbm90IGV4cG9zZSBudW1fZmVhdHVyZXMiKQ0KICAgIHJldHVybiBUaW1tQXR0ZW50aW9uTUlMKA0KICAgICAgICBiYWNrYm9uZSwNCiAgICAgICAgZmVhdHVyZV9kaW0sDQogICAgICAgIGlucHV0X3NpemU9aW5wdXRfc2l6ZSwNCiAgICAgICAgZW5jb2RlX2NodW5rX3NpemU9ZW5jb2RlX2NodW5rX3NpemUsDQogICAgKQ0K',
}

for file_path, b64_data in PIPELINE_MODULES.items():
    full_path = os.path.join(REPO_ROOT, file_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, 'wb') as module_file:
        module_file.write(base64.b64decode(b64_data))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print(f'SUCCESS: Extracted {len(PIPELINE_MODULES)} current pipeline modules locally.')

In [ ]:
# ==============================================================================
# STEP 4: VERIFY ENVIRONMENT, ARCHITECTURE & GPU SETUP
# ==============================================================================
import torch
import src.core.config as config
from src.modeling.model import build_model, load_checkpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'[HARDWARE] Device: {device} | CUDA Available: {torch.cuda.is_available()} | GPU Count: {n_gpus}')
if torch.cuda.is_available():
    for g in range(n_gpus):
        print(f'   GPU {g}: {torch.cuda.get_device_name(g)} (Memory: {torch.cuda.get_device_properties(g).total_memory / 1e9:.2f} GB)')

# Test model instantiation using embedded offline configuration
print('[VERIFY] Testing DINOv2-Base + CrossSlotTransformer offline instantiation...')
test_model = build_model('dinov2-base', use_cross_slot=True)
n_params = sum(p.numel() for p in test_model.parameters())
print(f'[SUCCESS] Model architecture verified! Total parameters: {n_params:,}')
del test_model


In [ ]:
# ==============================================================================
# STEP 5: LOAD AVAILABLE FOLD CHECKPOINTS BY MODEL FAMILY
# ==============================================================================
import re
import torch

# Prefer one best checkpoint per fold and family; use EMA/SWA only when best is absent.
checkpoint_priority = {'best': 0, 'ema': 1, 'swa': 2}
checkpoint_candidates = sorted(
    {os.path.abspath(path) for path in found_checkpoints if path.lower().endswith('.pt')},
    key=lambda path: (0 if os.path.commonpath([path, os.path.abspath(WEIGHTS_DIR)]) == os.path.abspath(WEIGHTS_DIR) else 1, path),
)
selected_checkpoints = {}
for checkpoint_path in checkpoint_candidates:
    match = re.search(r'fold(\d+)_(best|ema|swa)\.pt$', os.path.basename(checkpoint_path))
    if match is None:
        continue
    fold, checkpoint_kind = int(match.group(1)), match.group(2)
    metadata = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model_config = metadata.get('model_config', {})
    model_type = model_config.get('model_type', metadata.get('model_type'))
    variant = model_config.get('variant', metadata.get('variant', 'dinov2-base'))
    if model_type is None:
        model_type = 'coatnet_mil' if str(variant).startswith('coatnet') else 'dinov2'
    if model_type not in {'dinov2', 'coatnet_mil'}:
        raise ValueError(f'Unsupported checkpoint family {model_type!r}: {checkpoint_path}')
    key = (model_type, fold)
    candidate = (checkpoint_priority[checkpoint_kind], checkpoint_path)
    if key not in selected_checkpoints or candidate[0] < selected_checkpoints[key][0]:
        selected_checkpoints[key] = candidate

if not selected_checkpoints:
    raise RuntimeError('No supported fold checkpoints found in uploaded weights.')

selected_paths = [
    selected_checkpoints[key][1]
    for key in sorted(selected_checkpoints, key=lambda item: (item[0], item[1]))
]
models = []
model_types = []
for checkpoint_path in selected_paths:
    metadata = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model_config = metadata.get('model_config', {})
    model_type = model_config.get('model_type', metadata.get('model_type'))
    variant = model_config.get('variant', metadata.get('variant', 'dinov2-base'))
    if model_type is None:
        model_type = 'coatnet_mil' if str(variant).startswith('coatnet') else 'dinov2'
    print(f'[LOAD] {model_type} checkpoint: {checkpoint_path}')
    models.append(load_checkpoint(checkpoint_path, device='cpu'))
    model_types.append(model_type)

family_counts = {family: model_types.count(family) for family in sorted(set(model_types))}
print(f'[SUCCESS] Loaded checkpoints by family: {family_counts}')
if set(model_types) == {'dinov2', 'coatnet_mil'}:
    print('[BLEND] DINOv2 + CoAtNet detected; applying the d4-derived target-weight schedule (Raptor and auxiliary d4 arms are not included).')
else:
    print('[BLEND] Only one model family found; standard within-family fold rank ensemble will run.')

In [ ]:
# ==============================================================================
# STEP 6: EXECUTE INFERENCE PIPELINE (4-PASS TTA, DUAL-GPU)
# ==============================================================================
import pandas as pd
from src.inference.inference import run_inference

# Auto-locate competition test data directory
candidate_roots = [
    '/kaggle/input/rsna-knee-abnormality-detection',
    '/kaggle/input/rsna-knee-challenge',
    '/kaggle/input/rsna-2026'
]
DATA_ROOT = None
for cr in candidate_roots:
    if os.path.exists(os.path.join(cr, 'test.csv')) or os.path.exists(os.path.join(cr, 'test_series.csv')):
        DATA_ROOT = cr
        break

if DATA_ROOT is None:
    DATA_ROOT = '/kaggle/input/rsna-knee-abnormality-detection'

TEST_CSV = os.path.join(DATA_ROOT, 'test.csv')
OUT_CSV = '/kaggle/working/submission.csv'
CACHE_DIR = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/kaggle/working/cache'

print(f'[DATA] Data Root: {DATA_ROOT}')
print(f'[DATA] Test CSV : {TEST_CSV}')
print('[EXEC] Starting End-to-End Inference (Inverted-Cache 4-Pass TTA)...')

sub, stats = run_inference(
    root=DATA_ROOT,
    models=models,  # Use every selected fold checkpoint and available model family
    test_csv=TEST_CSV,
    cfg=cfg,
    out_csv=OUT_CSV,
    cache_dir=CACHE_DIR,
    workers=4,
    chunk=64,
    device=device,
    use_tta=False,  # Keep the time-bounded inference path; no horizontal flips
    n_tta=1,
    batch=8         # Safe Dual T4 saturation (OOM Protection) (Efficiency Track) (4 studies per GPU)
)

# Clean up temporary cache to avoid Kaggle disk quota errors
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    print('[CLEANUP] Temporary slice cache removed to conserve disk.')


In [ ]:
# ==============================================================================
# STEP 7: VALIDATE SUBMISSION INTEGRITY & DISPLAY PREVIEW
# ==============================================================================
assert os.path.exists(OUT_CSV), f'FATAL: {OUT_CSV} does not exist!'
sub_df = pd.read_csv(OUT_CSV)

print('=' * 80)
print('SUBMISSION VERIFICATION SUMMARY')
print('=' * 80)
print(f'1. Output File     : {OUT_CSV}')
print(f'2. Shape           : {sub_df.shape} (Expected: N x 13)')
print(f'3. Null Values     : {sub_df.isnull().sum().sum()} (Must be 0)')
print(f'4. Target Columns  : {list(sub_df.columns[1:])}')

# Verify matching row count with test.csv
if os.path.exists(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    print(f'5. Row Match Test  : len(sub) == len(test) -> {len(sub_df)} == {len(test_df)}')
    assert len(sub_df) == len(test_df), 'Row count mismatch with test.csv!'
    assert (sub_df['StudyInstanceUID'].values == test_df['StudyInstanceUID'].values).all(), 'UID ordering mismatch!'
    print('   -> PERFECT: Exact 1-to-1 StudyInstanceUID match and ordering verified!')

# Verify probabilities are in [0, 1]
vals = sub_df.iloc[:, 1:].values
assert (vals >= 0.0).all() and (vals <= 1.0).all(), 'Probabilities out of [0, 1] range!'
print('6. Probability Range: All values within [0.0, 1.0] valid range')

print('\n[SUCCESS] submission.csv is verified and ready for Kaggle scoring!')
print('=' * 80)
display(sub_df.head(10))
